# IRIS-X：G4 小模型訓練與 ARC-AGI-2 架構試驗

這是新的 private QUICK_SAVE，沒有執行。訓練目標為 Google Cloud g4-standard-48：一張 RTX PRO 6000 Blackwell（96 GB 等級）、48 vCPU、180 GiB RAM。正式提交仍是離線 L4 ×4；此 notebook 不會提交、建立 VM、開啟付費硬體或處理憑證。

先跑 smoke2 確认原生 tokenizer／BF16／LoRA／梯度／安全匯出與重載；再手動選 pilot16 或 pilot128 作真實 demonstration-only SFT。A/B/C 分別是 direct-grid、程式提案加示例驗證、再加失敗回饋修復。訓練與架構測量分開開啟，沒有自动全模型 sweep 或 RL。

前次 uncapped 完成 512 updates／六個 tests，41 個不同候選，但 top-1／top-2／pool oracle 仍各 1/6。因此本線測試提案品質；沒有宣稱新架構已提升準確率。AVO、Duck、Prime 的 ARC-AGI-3 成績不能代替本案的 ARC-AGI-2 證據。

## 手動順序
1. 選定你要使用的付費 G4 環境。Kaggle QUICK_SAVE 僅保存來源，不保證 G4 權限。可匯出此 notebook；不要借用 ARC-AGI-3 限定硬體。
2. 在新的 Linux x86_64 Python 3.11/3.12 環境，單獨開啟 PREPARE_ENVIRONMENT，執行後重啟 kernel 並關閉開關。
3. 單獨開啟 PREPARE_PUBLIC_MATERIALS。會下載一個指定的官方模型（約 18–33 GB）及固定的 36 個 public-training task demonstrations；預留模型、wheels、cache、adapter 與輸出磁碟空間。完成後關閉開關。也可在非 GPU 準備機先完成，再複製完整 materials/environment 資料夾。
4. 填好費用欄位，單獨開啟 RUN_CALIBRATION，以 smoke2 開始。之後要跑 pilot16/128，改新 RUN_TAG 並另開 ENABLE_TRAINING_PILOT。
5. 要測試架構，另改新 RUN_TAG、關閉 RUN_CALIBRATION、開啟 RUN_ARCHITECTURE_PILOT。先 ADAPTER_PATH=None 跑 base，再另外指定訓練 adapter 跑同一組 A/B/C。
6. 匯出並下載證據。程序 timeout 不會停止 VM 帳單：最後由你停止 VM／Editor session。

開啟材料準備需要 Internet；訓練、推論與正式 L4 提交只讀本機固定材料並強制 offline。這份 notebook 保持預設 Internet OFF，避免誤把準備需求帶入提交。


In [ ]:
from pathlib import Path
# 所有動作預設關閉。每次只選一個 phase；請在你自己已選定的訓練環境手動執行。
PREPARE_ENVIRONMENT = False       # 下載固定版本與完整相依 wheel，離線 hash 安裝；完成後重啟 kernel
PREPARE_PUBLIC_MATERIALS = False  # 只下載指定模型及固定 public-training demos；需要網路
RUN_CALIBRATION = False          # 單一模型、真實 LoRA；smoke2 先驗證，再另選 pilot16 / pilot128
ENABLE_TRAINING_PILOT = False    # pilot16 / pilot128 的第二道手動開關
RUN_ARCHITECTURE_PILOT = False   # 同權重 A/B/C；不會自動接在訓練之後
SCORE_DEVELOPMENT = False       # 求解程序停止後，獨立讀取 public-training dev answers；只產生彙總
EXPORT_REPORT = False           # 只包裝已完成或失敗的本機證據；不自動上傳
ALLOW_QWEN_HYBRID = False        # 第二候選的明確架構選擇，不會自動換模型
MODEL_KEY = 'granite-4.2-8b'      # 第二 qwen3.5-9b；第三 seed-coder-8b-reasoning
TRAINING_PROFILE = 'smoke2'      # smoke2 / pilot16 / pilot128
RUN_TAG = 'granite-smoke2-001'   # 每次新實驗使用新的名稱，禁止覆寫或隱式續跑
MAX_WALL_SECONDS = 1800
APPROVED_MAX_COST_USD = None     # 填入你接受的本次程序預算；不代表雲端帳單上限
ESTIMATED_HOURLY_COST_USD = None # 填入你的地區、VM、磁碟等目前全包時薪估計
ADAPTER_PATH = None             # None=原始 base；訓練後另填已驗證的 adapter 目錄作配對比較
BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()) / 'iris-next-manual'
MATERIALS = BASE / 'materials'
ENVIRONMENT = BASE / 'environment'
MODEL_PATH = MATERIALS / MODEL_KEY
DATA_PATH = MATERIALS / 'arc-training'
OUTPUT_PATH = BASE / RUN_TAG


In [ ]:
FLAGS=('PREPARE_ENVIRONMENT', 'PREPARE_PUBLIC_MATERIALS', 'RUN_CALIBRATION', 'ENABLE_TRAINING_PILOT', 'RUN_ARCHITECTURE_PILOT', 'SCORE_DEVELOPMENT', 'EXPORT_REPORT', 'ALLOW_QWEN_HYBRID')
PACKED={'research/arc-next-line/licenses/granite-4.2-8b-LICENSE.txt': 'eNrdWluT27YVfvevQDXT6e4MLTtp0jbOk+JdN2od7c5qXTeTyQNEghJqkmAAcrXqr++54EZJu3anb/V4EksiDg7O5TvfOaAQn/mz6GW5U+K9LlXn1ItnnvyHsk6bTnw9f12Iv8lulPYgvn79+psnF+2GoX/z6tV+v59L2mZu7PZVw1u5Vy9w4f313U9rsVhdibc3q6vl/fJmtRbvbu7Eh/V1Ie6ub+9urj68xa8Leupqub6/W/7wAb8hAV/NxZWqdacHUM7NX3htZv5EM+F2smlEq2QnBjjpoGzrhOwqUZqu4lWiNlaMThXCqt6aaizx68KLwmcr7QarNyN+L6QTFW6pKrE5iLUqWchXIN+acbsT3wlTwwcNz5lybFU3HOtl7IlipekPVm93gzD7TlkBKsFCPRyEHIedsfrftJ+Xc27FsJODgE23VsLCbksPeTtkCqitbMQ1iT5RYuzwgKS9ErIkKUELMAM868UYeMArqJXjrcGggzVNIaRV4UNDShd4Gvx27CpYVpq2NZ2X5B8Uez3sWA5vOBfvjCU9+tH2BiImWTU6PPho5qXM6ChOXOhLXmr2yhbgPgteQiV0x/8uxGBEKcHp+JyXwj+RBaxoZSe3Cp2H+7qx3HnFCrHfKTo+eJ/2lSQ7t8xeYzSBlAsNmpB73E73KKnWNVizV7ZE0Rffvv79JW1nwDxs+CBoHNwAVkcfgJusckEiiNyoDoxQanDlRHqmZ3L5z2aciQtYi/+ys8vc6/AXbfKgqxFlWZHHhxegHkFb7VAR0LvVzlHAU5xxEpBbTkJtDbuVkIKQXu1xpPVW1cpaWE6/1mTxT7hFayoNR5OUVcHBuiubkUwBSSg6M4hGtxp3Bz86Uw97DC9HG4JTKrB+yD0S5MXwA0XI/1pvR0u/g1salcHHzeZfEAqnqsvuwN+BO8aG8qO2poUfy53sQOuQIBAVncMnZQgo+qbxH2shBZuHxBXTA3oZR8eEtOk1JpQh5fwxtxAJcAb4enLgHL3gpA+M3g7lcO62qtJSDIc+P/ZHYz+dgMIeviSNCYcw0lIK6C4cIyYAm84fq5UVAMmD1I3cNCH/M1wqEE0xAEvpQ0lGXAjoBmaAhyO8saXgYU1mlcOAtYUsFLT1Ii7gAOpRtj3sDAsB2iHMeSE+ueh7BTs/QjI1Zn+ZrHClrH4AKz4ogQZxs+MIwD3O28Cf3ktiGwTFN9Kh8zpKxQr3wOiH6GGswq3IXZgL+50udxkYgLMGqAGQmVY9aHIlRjGYxueJUGBhY8MnEOHdnGeTF4ZVTjmIFLK+hM1MQ0kBy/RWd7DLqc9P8TjgVD1J/0Icm89bD6PZ+47E+6phVSt1zE/VS0uRgnahY7TKquYAedB9IsNtIFowTjrZqsvgdA1AZGtZUpEoshoZjXqiFFpHmTp5/S1Cua/xZz1+nAMxZbP9ogF9woVaGvVAYROfUAxXnokESYZtQ6vg96eUL7KkGBD1DWzdBNh24waww4NH4B0UXaQ5qedTgTYiHD+hFcHLVO6erRY5UUFUpu0x3jcKjFmDKZ4mL19W7cUsnmnmZXG9j7AMi1QDCWgNgHGBXtjIhuJob3FdR+Rj7Lz1BWZBbnSVDIV2GlxKFrK/K54tRRG78j3gb9IJEFE3uLgBSgnSspIVqZA7uEG1LodwqLmjwhJSUo30T7D7sfIxW4lcKzd6kcHIJAoya6PdgOOWo6MqTzu2hJeeRn4kxEulST0GI0zPGuIRjuJ6XY5mdJC8rbSfEPpsYkeBcimntx1hP4Qi+ogMezYSEaxmK7C3FHmuzmenKXzEr+OxQwZ+lvLkBkR8bI82FTtQZqMgnoAyKkJyUDrfJyWhU7+NED8NblsasDeXayS8WfoxEH09F39FWoXbvo3HD8xKrEcurj5WzzYzWZrlqKygSorMQAIhBHQmFke8AMghnBIYXq8GsEwIP4C+ptpr5Bqd6V6S5x2cGD++BNZjt9g4mYNshsPL2ir4pIHYPZgSgfykmvv+DzcM3RasgBzrMY5PkC7BeT9uYC1YEQK1byQEevwGdOZS6+gbTyzyvi2n+RGLiSyf7HimnBO2sIP+mDnoViLo/h945wKWqX7ABIOWYwgUCRR03BBdip7PmnkP6DoI28kHRSwvKER9tKlr5HlQBFQD8Mv/BUQxdmDHRBzwRNmzQoKZcDI0Afso7Cr7vsF203TgdLIyYpdXrWykBnvzs9nhwIokJLduxM0Ostc5aTVlZ20BfUJHo3SofXniX7hLaINNp3xFBPgDRhJZPS07XhAOxB2ur7agPpO8qXJ+iz26ItS6uVjW6P/YCzlAKozp6JRBb1kFuZX4M4Gcb9wvUsGK3Noa516SwfAYpRmRP/Fn8LwUjdy7UQ941EZtuQiAxYLyiRMcoeJzAEc1gRV3vtVOcsrknEM4VvBHS0wVxDAVm0ZioEyhGfWZEhqNlGO+5AVWxdUBUxS9F2JFukDYKvgyBF+0LkjDPrFiKPhmLu5UPhma09atPCRkO0YhwEEduM0Ej55heeQSpI2w2QggR3GEjAb+b2JFnrbNXMKfQLIitUJkkBRarVLs5do00BNxfQ/Y9eZF7Ksu+aQjRNoW9UX1uN8At2o4IoJWTn1jd4h/Tg4qqT4cdxLfUxkNe26yPXlwk6g09lHYv/NQx2IIQfugO4wT7h5dtj1CXAxplImt+5aMoVjOdOcy29mqARKsCLw5a+GpOwCNjg+XbRw3TAFRYIal6lj46C4QFiuFvKnIyASF6JDSzZ+NRxBn9DmG1ClzY/QMMki5yhChhSqDx0RzcsbZIRWuwOCPDzo1WnWJoBX97xs/dPVsdXO/fHs9g+R7HMjemHZ+D6Tc2T55dmUQcCZTTixL/spEhdZTgg9lRT1mCjp11qwIShLnvJkYD2qEDHwQOkLxJXbNxJy38Fm7UrCBjEZJh+1UPqX3S1K2AjGCTd8ENWXQMdk6WWgSVe5ZHb7PwXwSZHleTwdQQtcJZ7BkblMFPJVvbHFqZRm4Xjbl8r3BGSvVR5lCBAI6QHYWCLTVSzzkIfqmw/kcNMxILJSEJvR+x10Y4tepmTN/E3ngVjoO+aCHSM0rMpSpOj63CLEOk9l8LBuyqvDfFvudPCIzKUF1b6EvyYSCre/AEfmZqJ/C8UZVqa4a20BbJxETgIX7v+DOY0wjA4chBpjhbDLRtAp6JuYBdjyOPzbMU/cWZ02UugqirTSsZwJwNPjKXIFC/DlylXEkp5G1TljuGQafRntnroxYTHZXZOoz2hQpbWpqFg9PtCL5dC6mEsnDrbNpXlLg5LZqUoUj68ZZMlFpjKPJWCZ2KkedwMQh31Kz428CuFdNLNDNxYcOqqgjp6lH2KjU2P6SxOyCJM43DscsMhtmZWOsJ0dXienjjseDHKZ6m3z6/N+0Zp5mkZpZwLAIpq5VuH3k9Ssz4KJ4e0P1ZWO4KcO03VJ7h2WEVHMjlAOnKsUXQZgGmUv8RswueEAKVowt0RZ6Ogr8g88Q6sjUoyoziCfgjQaxaist3ysd9x7+LuBPAIWBgDiExYxHV4aQc2DKnd0IoeH9hRrTl3CNIVucm0VGg1MvZR9wpu8/gk4+hvnhELRB4yJNnXybatVvo/a3R1jQHfgESzq5FAq/afF6GrUBKwPvKOGA3hWx6cBJ7cl8NmRT8JuvBmdKAFvqz3NxpR21TnhpW4uPwD/BLoeYBFHVzYEbWOq8scVKMEBepOYlTcGK5DCf+y6peoG64tDguEXNn8bx5cS5lzjXAsifLdZiuZ6JHxbr5ToY9+Py/sebD/fi4+LubrG6X16vxc1dfi1/804sVj+Lvy9XV0B3NN8AP+J01KWTaMKVKhuTpgyiOakMOHWAJpdMRQ2RPYVYMOb98v79dQFWX71crt7dLVd/vf7penVfiJ+u797+CFoufli+X97/TCH0bnm/ul7z6wMLL+N2cQcO+/B+cSduP9zd3qyvudrybWGDNwugfw+barp1oJsZ7gqn4QKes6a3Guk5HbiG6MJHKP4S4mbzUp42OgecCI8b4Fo7QnZnSh3bZAZ1f89K09j8ovW0meXY+8scPgeT4qL3Wm50Q5fnS6y8AuhPN5AeLAO+amjYCTpCp52NWsJNFgTQkI8MOrVtNLCvUl0W8ba7mIxy4+Tns/F+wUQBZ/qN3hChI+W2OI+I9xZhywHfQHB0O34+Pxg9J+UDhzLBZY2mjf1EgFwrW7mdzvBxdXglIL0c4HqFd+vZ7TMkFBBbvkpAAsMzXbyQ80IDQuPMDfTGcbXlO3Os4rFW463xcaNL1hwjxoz8je68MzNczScGF8/eiQet8NiN4YDdGlPtdZPPDj9BUTZ9L3FKiJxgRMVrqZvRcjWSTT12idxQETzzJgjeAmDw5vbgjZWDwME4RIJ+PIjzMuIwXVYPmi5Ja//6BmSAN0J4ucGL5wz4bi4WJdYEtEJAXtx5kQp1lhQfd0jdp+l6fFn47HVbYKHlzhiegtKkc3LZTjNX4G21IjwBqCMNZVcqPkTPY1CPfgeKO9V2+GpJGoixWZuguzCbxk+hiLe8QthB5stXLXAezBffX2k3ue6BBuNHs8dOiFvJaDCyZyY4nY/eaOma7DYkcm5/LUJDXP81AmmCUdKXmE66RUmIniZFWRj4mTD2TLpmfMaE53wn29TRNpWqoV3hFcCMqzOjc2lbQqJArqMVUzqP1qbbMj85BkyGrhybVR6iFqdz483Bk410oANaINk0kvl9Fo0ZbYy6cABfr66wrp57DY5+X9zewiPLf75BF9K0ABD14F9fyF/dw99IlX28S8LX675wQeFfo5hOEwKtNpA1FtrwIUw1itTJ11o1lRNQICDZGfQ3eEupIDJnv/w6S00KTiZ8tTuEYCJU9V1f1knPxcWV6f4Q3xfIcjQI/92loG6d2lQH9AIiASh+1MN3B1nZzu5mMVfcAfD8MV6EUlPPCgBOwMLG4QUVP+3npAHF6VmOG4gyZKzcdhHN7EMxDlerG5VeWaEb0qCJw4UzUI4G14jBM6wV05tP//ILqgmBp+N9vLdcuHeN45k05JC23OGNNQdDukz85QB/fhW/kN6g59Et66/0uA+SKuuZpuFT5C+Eigt8IL5zefk9igj9CAIBly8/Pg80Xne+DSVojBEVKU7W9ZsNTcvkZGQXAlkOIdw/98rpe+Duq/X1S1CZlnwJQ3+Ke/h3zl7kU8qJvYJ6+ApD9sBTDPx/pN+BeJPZ1kpNVAhBTrQGYgaO1m1HCDigBFAWuuM3+/y0JPF1d3qu+Yv/AH5c7GY=', 'research/arc-next-line/licenses/seed-coder-8b-reasoning-LICENSE.txt': 'eNpdUl9v2jAQf8+nOPWplaJuqrSXvZnEFGtJHDmmjEeTGOIpxCg2Q3z73QXabpOQIt/d798dpdBQuNaOwSZJ5k/XyR36CI/tE7x8ffkGi2u0uRlb7NZ2OroQnB/BBejtZHdXOExmjLZLYT9ZC34PbW+mg00hejDjFU52Cgjwu2jc6MYDGGhRJcHJ2CNN8Pt4MZPF4Q5MCL51Bvmg8+35aMdoIunt3WADPMbewkNzRzw8zSKdNUPiRqDeewsuLvb+HGGyIU6uJY4U3NgO5448vLcHd3R3BYLP0UOCpOeACchnCkffuT197RzrdN4NLvQpdI6od+eIxUDFeYcp5fjiJwh2GBJkcOh7zvrpbp4h6ydaaLyvKFDl0vvjv0lcSPbnaURJO2M6jyubFX/ZNlKFxvd+GPyForV+7BwlCt+TRGPL7PxvO2e5XXb0Ea3eLNABTp9XvbdCb4YBdva+MNTF9Zq/4kwkHyIe3pkBTn6a9f6P+Yz6Kw6NXOoNUxxEA7WSbyLnOTywBt8PKWyEXsm1BpxQrNJbkEtg1RZ+iCpPgf+sFW8akCoRZV0IjjVRZcU6F9UrLBBXSfz7ilJoJNUSSPBOJXhDZCVX2QqfbCEKobdpshS6Is6lVMCgZkqLbF0wBfVa1bLhKJ8jbSWqpUIVXvJKP6Mq1oC/4QOaFSsKkkrYGt0r8geZrLdKvK40rGSRcywuODpji4LfpDBUVjBRppCzkr3yGSWRRSU0dnMHmxWnEukx/GVayIpiZLLSCp8pplT6A7oRDU+BKdHQQpZKlmlC60SEnEkQV/EbC60a/rkIjtB73fAPQsg5K5CrITBFfB9+Tv4A2c9TVA==', 'research/arc-next-line/model-source-lock.json': 'eNrtfflzI8eV5u/+Kyo4sRF7CEDeR0vyhiTLWkfIs1q7Y2N3ZryIPEm4cRlHd3Mc/t/3e1UgCIAEAbYId/fIloNNVmVlvsr83pX13su//qpprpbppkzC1avmarQYLd/3J7NcxsPlbL1IZTiepTf9t/zqC2o5i8uyeFvyMKyG61WiRwQTpsdZj7muyXy9mM+WhW79tI7jUWrW8+VqUcKkmZRVyGEVmlEu09VodftlM52tmnlZTEbL5Wg2bVazJs/eTcezkJt3ZXR9s1p+0ZT3Ja1X5Ytmvpi9HbXtZotmuY6T0arfDbpchdV6SWNuqL4bavi2LEZ1BIqv5+vhenr3Z/dYLjWsx6thCtM8QvOW6lGc9K4XYTpalcHm357qi56Lm7HKuKQVqBgu1uP2ie/fhvEaTzejKWgstSzKNBUQmcviyybdzDAfzXo5ml5jCsJyvSi5SeMS8L5h+aaXR8s/z0bTVYNuiAp6QRDU5DIfz24nmKpmhsmeTQomgyYsLug2frzZvP6WfpqCf8WVpvlr+5PWY0vQkJ5AA/7F3b1Fmc+Go3zGW2+ad9NP7avLhZuUU47JJGt9Cppbw1RhmmUnUuLcF6Xun76Dwf3arBdj6upmtZovXw0GN+vra8xRDan002wQ5qNBi8Tl4DhxgzuaBucS9N/jGCj+erVYl6doW94EoU0LcFWDsaHoLGWITnNpY5RcRM4SZ1mW4Gs2xdSYmE+6Gp2jdIYr7UMtOtyPAmYo0441wjyA6Xqiz+5vzxejtx0Gaxgvy/b6NS7mB1cT6F21nLjlQuZ6zL7m5pXwr5jvM8b+ZWfwsFwNMZ8d/LeP+B5TrwV/xcQroQ8eCYt0g6lOKyD2HlntrR+6VfjtbPFdWC/D+MffX21u/mn7eCdGVrfz9o0363bf+6q8bwl6qk2aTevoGrf+ej/0caqepGuHMupktSIRBCaOo7A8nNy9Bnkxm4P/0Ib12aMtJpAho/l4VBZdI2YdF3qnaZwth6vZmzLtmI0zJpzcuZ/vJiBWiL4VN1c7N8sklpzBGPvj8D1iysMRtN25fzPKkLnDkFq0LEfj9dXDu8vRvxMVinmzc3OECR1BMv17WZAEuS7tS/K9FiuI8JJHAORdJ1w4tkvgeHY9Wi2Hy4SeptcP6J+E90OojVE7n9s3poXhkjMrdpuO58cW7Wk0ocF0PRner9tNCZm6keKgyWY+xuG2LJbtjBw0eFNuhyT0y7YPt9NiHvKTiwHcjvI6jJ9a0MVkOZzOFpNhmbezUHpsF1HAZBnOwyJAYHU0/nWXDdrbqxsIs2789n9fPGyxmamNHrxnlb8dDnW/btP1eHx49/hIV6tRGb6DJtxf1MOFW2GplhWvi3chlX2nZVRf2z7fXcH1skBdQ3biNsnwnVtvZynELf4Yk3p3XVczyI3hQ0771cEb31kQ28kdptl6SmzjrOfaC2/YYdu218Mn9hfl6tvfcrPXybGBl9AaQOhythjW0bgM422n17nVTgqw1T1c7/XWaJrL+2EuaRxgXYCUW7Kh7p811nDOhJTmXv4SQomr9+lED8vWkvrhp9fi9bbNUS779qfvd2/GgAXqVgJcthh1wx8uxno6+guYp8CCW4WIl7xjl0dbk3B42PSuJX9UVnYkLGbvHu0x3cCAXZXJfEwiCyZua3XR6+zd6f95NP1zuNpn3dmYDODtw/eGQmUxKFmtjxJjw/yA8seaKRgjoSQvqvK1hKJ0ytWSgeJs9jVplbKHhHB7Ejksh+3gw+syLYvQ6RlYi8NIBvmWhx6AiCCzrxX3BMM8rG6I1j98/81vfv99f5Kv9qTC0xbZk2ZYOy9nW2GDIwRsuBf2k1B7N6A96NUjzTcnGp1nQXKVRIkxgTOYkbF44XjmyoqkTTZVOeeuDsReZ7J3GE5hcTD+ovxlPVq0BtK+dCHativNuPQm8uC0N9oZ7wwWVohUQ3AmYy115p5zmS2vhfnCovEpiViBD2ZzuHqk32FYr25mC3hFrUTeeEFlw8P3fky4DqMpTLkt6+/PzF7PEwB00enuPIR4HWKZCtlPm9d7TN4/CpenueLvCpyTpGwg5LFITyOo6OSVMVlLAEbbqFRw3vMos1CC+1h9YiHz+DiCVo+IxrMB9IKi4jMBUGvI9/+8hJT9WMA5RsIWMPYEXphSypkCv06WXAyvVcAPlEE4ESznVoDXU62P46Ubfog5bHX1h6DGaSjxqopURsqsCleumACPU8UkMuiQJAtFZBwXnK0xB2tUKakKw4Gczw41O7rvEwDQOdRssCROYanyDAUStdRJliA9Ux5qq5TspSzalWygYCpLF8MSjz6GyoL12mfJM9CiXa0mamaSsJn5YGWWXqeQU0q2mJx5VtEWyK3s82eHJbgW12XZX71ffSz8HKPgTv5wY5R5GjZSK03owFjZqBRsiFkxG10sMmssGlYsW8cvoLIibBwlIHyk8gyqq0anIHM4M9lbyUBXVVapCMUlk+ClcEl3AmMF2pWJzw8wZCL2yJ3lvVltf1H9e+ds+dFg9Cy6NuDSUmopmGDuaYAF2Kwq5BoYGN1Zxk2UTigHS1plZgJUiQWF4XGAbbbsz4fXdkHHdbnjScVQsqrQrS7zKoq0XoTsPIbXRnAuFBzoGFWF8W2FqFwKaGPFmbOaF2/S1TEMX6DffQzvvdBslM/H5q479xxwik8UnOIZ4BROQwFacwKczGJhfGASqxJZlUmKqp2uOqickxLZmFQS45cFp/HFBu2j46yqxAQ0to82gLWiyiGxBNlXTHQuGFPglxYL5hEpkkvomDwKzkv0+7HBKT9RcMrnSE7DtCaB8DQ4IViAxagj9zbDPi+QtrCnHP5UFiMGlZSVml0WnEkkLWoWMQRTksa40NHJOyVihWEg4NHq4FhNPvCoINAZsynAi6iMObi/x8B5iX4/NjjVJwpOdT44uWFCMsHdic0y4z1sdlMU1s2rHJnjAh6hltHoIoXliWvvDo20lwYnyzUlZhX3MvEEQScso0+nHpOSCpwOFZkyvDgrvfE8iSCVBYU2MclqPSo5L9HvRwTn7qr3Wwfvo7q959N0t3nLpT7h/mIlbJYqGxGz4EFAdftqsrOWSdhasD+TyTaKi7m/WdoC1emVcRlWoGYgJikGIw+GMvOlWmWqVlkLUIIJKZDl8NQlsQuHsfLZeTPLeUnop/t6shxOwvyjguoscu40sDmxkQvnk1cY81yUHG11QcXKVLa2FNjw1XBwOdNcX8ArLl5JXQukTXbSVdqkjTGRG2yDK7JWKUQsFtiutHUCDQrEaQfZo2LUIn52ONrO1keFz1NUbFBjuZaKn9CLnHm4uNCGzElSCzEGD3HEOQ8pGiyTUcLUXC6AHOekt1pyyBhnlbTCQCmlEKqAWxMS/JyadcgajqhItQJUwLCoIkQBwZjt54ucT2Ev9wxi7kwsa+wJbaZtgQ7JQRava1RROQ3PlL4/Wmg5XWJW1ntlL4AiATu7CkgXDqcwcxagSWtwDgZdcvCQc+aaF8hHBQUL51LAafS2cqWUd5Bcnx2K2u/6HxU6xyjYmuRcGH/q47Uqlfx47T2TxoDvddXWWSNUVSW7yDRMkHSJfVz4Z4X2Ehi8MnirIeUAL81UF43MlXljHEzozCuEnxC+FKhVSCmWvKa4NvbpIeZB4CGF+m4DfQ/jJlcL0EARIiAg3z643QUGvCm3OxFkDwJhl+v5fFwoMhcmTFisRsDZ6omoC4qzfSrIdhym1+twXXpdvOvhst5H3UJhaF0MwJO4zVYEG6UFgLQKsCZwU1RdWDz4PkwxIdMwabHz4++++/6f//j9k8yzCO/6WJmbdVwvywICkgLmwEaTwZkvMDiXzsGj5DxgFwHnrURfPFAqIGBZBi+iu5wDfDwoaBKzyaijuxeWZyF9rp5Dz9akOZ4q0YosuSvJaCeYEVZIEUW13ghwurM5VakzC1E9zugP3Jw7GfXoS90x710Q8DNZ9xIsNp2t4EENb+qQALocrWaL26Pc9asd6fxEcLl4LLj8f70r0wH9kH3d898+HlGeBBNCGoFZNaJAfVkeMkuww1hypUBUSWmjDy8SUX5A0X0Y+blUPD+MPGvlYnQySpmKqbDzrNWOgpXgXDK4o1YWEYNxEQMaV7hM1XkpGMR0iPbvGEYuesK+5uKVdq+EOS+MXOKp14y90vyVkueHkbcrMNQUrj2b5jb6N4x/2H5nPxVT/pfu8SdjyjdthnTrZwSWn0npfpT5aAKZuBuwKJRj2hyNpTx8n7s3eoTaM2LYz4li79pMh7g1X6+G1x1wHsigpwLUH0af01uqfQuorsfjncjrNmL8bSBm3W9H4dTDPJpQL3r/Y/iTIewngtiPhrEzcdDosUh24fY/TLVx4e2iHcKk5c4pzJT7d90j8gXu78/kS/f+j9H/Mfo/Rn9y9J2bf9oXC10/ENZv4UAsphDsnSRTjzWjFJIdaceFe6zVXbLJXZoJN8da7aejSPFYu/s2R0Y9noYjYBQfyHRKxJlNx7f3eTL/uj8hZyjirt1qPnws7YY/aEXZHzB6R6m1WZ5KKTma7HMwg4+NezB5jyf87M/FI+k6++Mcz1s5zFyB9oTRd7Askwhbcjm5z19p9bB88IHkqawg6qht0Wq5cQlvH3U4ts2WXZrrAxXXwPX54vQVtnfhTwdDPJp/9GibJ7KYWp9rQWp9uJitwuJ2SI74rEvDE3qn5d8eTW16geQk1s/l7e6myNXbUS6zB0af3WuxvA8iPbTqcgGEViG9GaIZJYm137jKI+yVy7z1N4V9wk66LuP1cH7bZT+twvTmKaOJc31oDw3TTZhOaVMCfPEyBpWSTJ2WE1cP2fQ4Bx+RWYcDwcodnjAT4cNjorZ7efueOG4S1tpAzi277jWhxITZAm32+hGPom8DgzLND9CiHjYDJharBw3l+elr3mjJmZL3UVYfkL7WdcL0rt64+i3kEOXsqOfntHnK2+HMOfb8nDZ6VnCj+Pk5beTFvVRS26GX83RSG7W27JykNmpp/OmktkM9cZmktqBCKS7UVEVhlnOvRKoUDup9LMYwbqrLXAVcqNWXjEYxBeZ9Eq7avW+uF09qO2dv82BL6OE+UPc14NxtoMc3MO/FqTrxIaB66bg0RYYcUs1exMKcraZwLqPOFEFdjXX1RfcSW0cemEqyZrwHo+3P6K0VRlfGozPcZ2e1ZcFY6TOI02iSpDIKPF550fbzS2H7oIzHnw2Pp9McrTXqVGxDld6KFEXyXgdnYi4+JC+UYjloyWoRuGcukuaYKLTWJZFzDpTBWCN9grYerx2CrjlHLq2XPjBoBZ9UskbaEDwDVdVk9ktLc/zZaDk3t9FafSJRJGinoTmdV8XIHJS22QMyJZkQKMqAmyyYM+wSHxhfTmP8h85t/PloORG5ILk4gRJhZSpK2gKpH6o3gdmUfYky66pcDMllm4y2lwvDg+UnGSVOcmidklkUWdlYZOImSRNccTwJAclSawgwFEGNdjVK6E0Rmf6FZKH9bKicSD1rk4O0PyFSFIQJ4z4FrjEs9zUzLpzRDraDLzAddChYsEuIFJ+lpoDQDOOjgg5fJKU/FK+jK5yF7LMqojjpYCkn6QILNproVeY5uvx55p7txty+UB7azwfSBxO2zfUhm4f8thPpFDVlUz1WUfEKP8vmqI23BeirWGphk6MYpnTZiPUcTcUwUUmKFKDEfGlY4BL3teHBBKOpDBj5sABihikdlDCZsiSAQlaOBSRcot9PJGL9hZLSLoRU8byUSW645icSf4zSUekMRQYfrpBvpHKM1WptbQ1GKlhayUR9WaRKnmy2Bba6xngWflrEpaQ9Rba6SslH3HntOWQnk9olJo2CNR8jJdRJdgypl+j3U0Kq/FSR+rwUNXRulWInzD1WBa85KR9zCq4yAbMPKq/wkiCTsxOQtbxcOAsIMlwavHxQ3JqgVJKJOfyWHUvaBFEt5oci+JkDRVDxjOdqlKjcZlB7NAvoEv1+SkhVnypSn5GvJsnM9Nqekqk6MLiCMPpcrJ5zrbMyxXAG205kSFvvfOBB1wunoRv4GEklyD+lwCawNCVc26oqDBBdhTJZF2scebAVLpPQ0cjg4F3XjPkzR9PQL9Dv552v9vLIPJWkZr05laRWFGUTwvAMmnkDWy162qBN1vGAa0UGrEnK7mLeMSw+OGY+wgyJQL+s0msjDbcJlEgJL9jJrOCEeao3Cq/ZiwRhV2p01dby+XnH80WZL2apLOlD0cfaVjmPiDuB5k/p3BKcCkrarJSrEGoR6qcaB06GG5t1SqrEeBi6/JIgspApSrOQfDIaULECAscLVVMErEWtwJRyUlXHLVOM/BLOHefwUIrm5heUofazoXNGWhrVxbXendB/wULeK0Nl6bxy3FPWPU++eJtdTt5YDpqYyS+213JEA+rqi8rKMx4g+Tz8b2VkggdONRGYc9GkFHQK5LNqITKPHiZVUjE6iE4ljmnAS/T7Uhrw75vU9nKYO53JBtickFYlB+WyLFYHR3XunLK+FFWSi9JWa4rlPPuL5CVJjpdl2QTHvGWAQo4asrO6mhzspSSDoWRan7y0Cc5OMVbVKlNgwaiQPr+SiF2c0Ceh8p5DyrZCqz5RqizS1+OgQ4bqyzBetIIdX1KWLorgVTAZq1bLxRSfhQUdqFwVr6EGnxKkDU88t5LTFgbryXONf2Fca6pfxgrArZmheq4g+BeSGvnzwfN0PiSUmLCnvi2wEGC3MqNdNrBUVND418MACTmy4oqSmpNbfwG5k4r3UaUIy0hmGD6OmUTJkdFaG6SGjUSlfBiDEyBLzEVoyu51KnmqksbkLykf8i93MDkrF/LclDH5WMrYt5ip3wS06v2xlDygH73vQMui577t/aEEwI3q0T+aSea9sFg+oXKJHGat8znwlAwtoYD1HZ0jfOUXySQ7j9D7BLNziXt2glkoTEGslsp0AtNZpWxkleoZVO2MrYnBLUzaO6plpWHr66CYUxUsHnnaAfJugtlktHqxzDLdY4oyy5h9pSm57IzMMjxi8P/XTLzi/pVQ52eW/TgOk/CM40nG1P7JRLKDFo9lkA0py5aU1p1IHszmq8FqdF0Wg+VoAj5ZjHsJvfbAjuPBT+Y3B+FbT+egHXunlzja5PhpJU8cM7JbUv9YVsHLHz3CmHdOQFSfOoJE7Rx3sE3b2onSPzjK4njyg9F6r6ezjyDZx8yHH0DyoMHzDiDZvbUoW4m/mh/cbA8mGc4JFgcL/3R+w3POCNFtFH//A84I2T/A42ECxBOB+upDTxHRmkvzjCNCBCxJI3bKrD4nxrqLo77v47nR1FIyBgNGqPt8xbOjqbtn8eTO6SKnoql/WpTXhKWStyHVv4UQ/5lnhWDKBT/7rJDD1sfPCqGW7IyzQtolPyus+lHX+5/2o/jOjbNWiRkPy9PDLbJYpVJ9Zor7JEKtJbPAq0+OFx2YyppCpOCtJKmzNSyyJD/5w0POtpM69+NcM+lEhK32np+IwGYOdhC8C5ngppokeIhMyywyDyorJWEgGS3ZJQJsoyh0kEAFMPF6mnNothgl4yQAYKkpm7xhKuZqvTZVOFYVU3R4hMSvMvxSQiUvhZ1TmxyOnwCPCkJomZKqIUsuLTOqeu9ztIoblehUES+CvlwdQy8htHmwKkqfIA7ow3cpvFruPQCSRJR0ugDwIwvL0ZuKuwZIk0rIGuov8EiIS4HpGcdD8FO4isVkm2pbL0T7WrUtClqBFZFMVEAVY9ZE6y931AgEkE0cJFglbJA5lGAcxXkqsoU9pKX0kJjF1UDpCDYGE5isqWr4nLr8R6j2bz80zuJSCHsejXculReKOy7FieLqriZrLItSylyT01ZwrI1r975MEEwGAQEn84WLq9tIJ2hlEk+KCk5TRLGtThhllYY2LiBCUcljWLleV+EZtDR9XI0mqyKPFle/QL+fTOX/Txio4jlA9R44ZacKrQdtsqvKUImxkC2MdW+4YyybWiEqsWjRhVwvXGjdQUQnJmGAK6rdR1/Rg4E5V2PNpGhdpA/qBkabFsUEn4sD4Zw0Mcvc1WNAvUS/n8wpAJ8wUOWzgcrZidxOUb2rUgTruckyx2qT9tCVBvIUJr3ilEwXXb30cRXZZAN1bXjI2jmIcpGkTFSQGA6nrMyrUpWAwxnhccK99PAnSxaYNLAWO35cxcv3+8mcCPAJA1U9A6iUIM6s8idyLYwNKdFJN1iYmCqdYWGNSzLnLOBCZJaweKVeGKhKcjgqDO6wpGxXJ51x2eUsrYPdQYlgVkqnQEvxsSaWYRFXSoilCgAxu2NAvUS/Hxuo+jMAqn6+RBUnwlO8zVxo5yMnKzX56BQFgEgsmcbCcpgAJfN64QOAJBgkwSMrnHmhc6gZClpyK2RwTppqgrUehIgsPaClUmVcpRQlixF8dbRK6SX6/dhANZ8BUM0HAPWEM1Ul49ZyS7t5XspIRwJLiFQhTA5wqU3xJYh04UwLU2JwsDhyMpnS9EONkXvBEo86AUkl5ygEx8yAsWSg/LqQNK8icyo/cdyZukC/Hxuo9jMAqj0fqFIooxSc2RMS1XE64i4xihpgwUFTOu0yVUA2IuicKavIF3Zhrz9a5n2hVPOc4dTYGkM08PGiqNraKKyV8IKkrjlkOn9Q4e1DFTEFWNW12KOJFhfo9/NOtLgoSJ+TfyGk1+xUUFgSrhQrBO2vwygLuUgB59hHQec6MWBc2iAvtgkaYFhIk2zBe5cIxVtTdEVnKs3jYTHDl9OwRAwdpeAlbdkK2kTSEuykpS2/xEOCLgWw5xwYZE6d/J4qnKIQlBVWWlYwcHHwd4QK2RqKrpfRh+LEBcINc2KwAoo3IeL9FFyr4JjLOkZFp4RkAaFkamYaXnOOKSmhKP9LeRk47OP8C0rHuBSUzsnSwNPcnPKbucy0t1MycwXsnoPIVDtFBDpSz7JkhHSu2nDpLI0cjRbwBag2gEvK60CfkUug5B6TWYsxLzUkGIQVwAeDNEpZkuUBtlvJT1QpePF+P88sjYtD8fSJ8k66E0HUVcPGNi5wlyFDZIrWacWspsOtPNQWy7p6IcIlDgevybHonZV03JCpNZN2tiFEK+HwyJLBCErR6eSZXNvipFLOJBWkYsqFX1IQ9ZLAkzrwxN7iYSjzh54ucxSkx8+TYcEWx+CUWpt9hs6LgY5Tie3hLREqxzAF9meXOk/mKMmDcyk78wQZJmmXT/h260glxkVQJUcXea6aCurFoLL2Rz0JOgkcjm0UwDYnG8Eo7kxNMBSqVIVniRtFuhTorBkXfI3JqEzpJHA7wpETZJgxv7QDZH614amrOcUeLykY7o53yvTtaDGbEvDvAyNhx08ms3aMWLkZjmeLsBOOeLVchdW6jUbcBEduX4QYdj3d/km0BohZisJelWEbKzpajd62IX1vdoLcbzFN0yEx4WxB8euT8GcIislo2taV3gkxl32+F1SKv8XDMPO7/sq4pIfd7UbKXlFMIIWvdqs8Xb8fvndmuJMbfHU9HsVED48m60krwLb9duGx725KaXlP9IXbeamQ3oTrchBvuhsve1iGeid8Vvdd/2Dh54vRhMptd3O+y+vz2/moP1tct78MdkcYtP0MHmq4DXaXwzxarh4eqLGjinsQIV+JPvvi11/zvu6zw4Lh0/Vkfkv3uL16UCecZgD9/PprwR4+Ob+9DZPxr7/WB2/akndd3tNTAjdZX4jDBlvNufzqa9yXLX3U8OEwFAi7OLy44zXTc6ovHzz2lzz59deqL+zx0wfalX94HslfD8qy7wnv3eXptcvTm9/K3nQ2Lb0wve2/uxlfHdZ1PxTubcxov0P5zWwJKHbLv0HcoKZB5AMXi8V/lUVYlEx52kOFaem5i4YLXSptJ1ZYB6zCcGCJ4l9DckIVEwcfROZOsjFsjBq97na1Uq6wP3RKnmdbHczU7EXJVuNmooq9ScMh1J5RLJMBQeWRvrfyG+bcfi73xk5vY6tXo0kZrlfp/pgk3ePyNZOvBH+lRd9BkUj5L1d7j/9td4UfrRs+L3X1BLsCefwAeWfzK3U96Dp4PqP+DO5brlej8eM8+ZDbIOfaUeQj/LUr0R5hosNr9N1zTFF7peVY/rDHYxut+4JpCMHU9kBC6UVZlFak15H28rzJ5aCygfBJWsboSL6qU4Q/p5WjZFlYRAn+K2wehsvMiuBioFrtVjuwj4g1D55H324NHFWi1RXGXUnFKsaTZcVYHR1dghDwXkYL54CiDaPi8KbBL6DEOuajPcqUjlhSns+Svsf1ay5fSflK+7612kvzbJa8x9ETjMlJfXygIr0fYNB184Hs+RWYZqM//x5cKvqP9PMY3zzNeXdq8UU5635Ke92Uvjx/6ToIbJB9qdxXSyH7lNDBS/GspFrgEUueYikpeHBdonN5GX011InbaJjwcfAhVO64LVB5qVYD3zwICz7zzmkLhmO6CLguGi4MHBYDMmpb169aXpnUkAicV86Pcpm0yjN/JpfpHnM90XKZtq+Y6mtNuwPP5rJ7Q+tp9Qfb9wPN1e0Ag66bl7JWsW6mr17YeNsS2+uI7aW59L0QR7I3ATY6D0IMud24Ef3tVdrUu7v4EjgXZWDNgEoyFwVFUaoBasmck8kY6SPTJnhjgzMCmsPA4GLMKFkZHZUsOE87ZUwu8Da7/EB1IGpyKXHoNKG4ks7KwLLX+C1GVwolbUQHPcO9ozrDRYBbXbYimeNaRwowxHMsQQ5j8DVnrxSjTGgpKUD72Qyxa5w8xRHuQ+3BnQEGbTfnMMTPh/ZehUEalsDA2cfBthtoNqiCyVplkDnXQifPuAIUyaSDDMKXzGAMWWesFCVJXUxbt9CIKqXRg4u+zj246cR5KoyePJW/TklYBzgbWaM1VOE7q0hx72BHqzmVsYOHY3MOjMUSrFVHwa1ps42dj22yqjZZ/hB7YB/BxbOxfSBIn7SrpPhQgB+MMuj6er7YT+NRevOV3xhXDtbKg20CAhptNf36a0mK/IG9U5f0jbPdaJCPbW7c1N77svpKbA043RdfNnd7RsNJSDejaWm+/rr5t6sOMP921cwWR1qESX66wTe//82JHhaTEw1oY52aPHgTLMX7r3i3S/KIF7dvh/pn7NWsZpPxqDVu+wxz0229baDSfAWaaM/uIUF3+ytKPNIlYDK97pX3xMHoZkkN+Uu7eQc2ca/D4QU8vjhIFRapy1kbniklxkabbTAqcw1Z4UKgE0KEpK80BW4XTz6mJHMwQglW8uCDSd1x/qBdE52EzL3x2accEujRNcEBjLCFnUopwGbNqlA5MG0hZWETK6Uxqj6o9bXn/ClIG/Ms58+SFhb2lTLgSmE+QAu3qdbiCQEF0PXNh8mnru9B28PzpdLvw+LNev5HqJ/WFXtZyHak9VrSXh6oRgwCH8hsYIm1RXBVdJXrEBPP2nJnWQJeZNFexagdFYijLQU6R4aOJSraDJ5J4E4OjoaHBLDxqjxFv3Po+Zwt/gWbGMdhx4YaoFmDrlJZz1wOlTM61j2L+oSVyKVy/hlekyQjUbBXTL9irA/j1Qr+AV5T933zGDoFWST/La25cOdi9O7rUX9zsGS7dJjUQdvLoL02uHoBDwf99Hboa+0mvvm5azgJ9xyT74783kI89QbdyP9JfPuzxr6HFQWu+QxJZyKd5MBZcJBq+KtQhFuKskjtfJCKF109w5UIpKcgXNUUZHGw8F/83IkUm58fYyI/YOzdjA6tamI6RuujAPcZOhsNExZ4KnA+oUGUjTwpKRXa6jYmL0vHExUXteksDnpQHGOyXq6GmwiPYZjm4U1Y3gzDeLz7LbGD+f6XuU7IHXyZvZqMlsu2CM/+h8hNiY7hrFY6tHlY3pe03hQc2Xt+Nqer7Umjncgchrgs09Uw3g7vztPd+1zZLkWY7RVgGa2WeJOuBszO9TpuX221mu4dbDseT/aK6EyX49nqZq+8UZnQ0bez8ZvR6uGn0HaubhalDKehfeFN9QyqJwP6Dz7gHr7wAvMZ6IPxpl2hqpXTVjCdqFey+1VAOz5sy9jpbnTg9miYu+I5UJyKqiZ6a3wRznCdSCUlVSRkdKixanBz5VS1RZsMfzAb6zx3aedb3cn4pAMCr/H3aFVOUigdc9qwKIRj0EBwN70T3BrOFWc2KMk450wHL+GB1eoD10IoqvzMLRWT+XAK28JWJ+lThqpilMQq6DAwIzlYtgppvTI2RKE8jDqmaE4VvFTtisja48/sIiSh/nD6wno1O0ke7AdeCoWFCE2xlqH46lPxytfE6bABlVMwmD9TqqSas1DxXngqXezharsPJ2+vUhUdGN+pYwprJW/jCZoxY0mzWmAJMc9zpGOvgoCZJEiPUP1cFRLlW8T2IAEfScnkAv0hVFYyn08zfdmB586GIzrSeDWkzf/lE5QxC4msggKLmKIcQJYF1V51VRbGuCkSbofylklNh44WEaKxAh6HyVjuGq+eCL4ijm0Pjh8uZuuDjyxXbRzL4u54XznUuyXydnfLOrmxLdz3yEnTV5uYmAdtDk+tv9pIMKpmOXrfHpJ9fV8QqD/Gn2sI5Q0E/2vT+3Wz+XW3k201rtH1dEb/boNu3pTbB+X//t9kNd97vr3W9todFY6bj1YEbDVXK1ZvO31FE0kjtGos5DDvKpKhSbh+pCTam7KYQk7PZ+NRaqOHvn8/x6+j1fi2WZQ0W+Tmp9vXpF6abXHPhiC9XjbLQjXPVgVNgexpyc1vf/xmkNrl6RHyef6ymc6aJZkrq2ZeFsQWFBLWQCusJ62W61890MaEzuFGpe1QtrnSdLrwy6YFcft5CaT9Gfp42VB0E2YBlGwaf9XuzrN+8830trlTq3c3mxZwzbSUjEfxOEye7l7zzbe/ayZhtRi9799H2rQGwHC7IbIzZ5iQ2wZDX2PktlWD3xtam6atXrgEdc1266OPm8v+EmtEv/zn/9Kgu2Z1g1nJVN0MffxIZlPTWUzNYj0l96HffP8+oJufWpOjaWON+m2sUTdQLJsFw/OjabMTcdWQ2fFlA7NjRHH2DRk3TRjDY2ryrCyxRCBuAYi0r01hVGCAOBqPVreb1SGubdfnqtPgXZ00irvaqSl29c+zpssaaiA5AJx3YdncB6PRZlJn8cAf3Ezq1fcBk73TBm5c78ff/rEhC7f54//4BvKnGS2biKnD3XjbhPYlJ3PqpbljqeaHEd5/PIv0CN/vfI+i+x5pureP04izUW7qYjZp73zz0++asGp/3SD7LrJy2/nviTmbNv8EmKFwTnIEqetclmkxmpMMuR/irqjrF+1sb+P1GmLMDXFU7RFQoXmawNYCosoSCw8GW25H/aksegmcPUJfpemk3rZvYOK2oRDMgPWfh0wTuq2E11ANvC+b7beYBjx5vbppyl8gXYiSUQeEu2jD7ZA/dAbLzqs8kQ+0MxVfc0PWChNStjM+moK05WhJAaLNuxGGdhb2AJSuYQ0dQd9sKyguG7xiw2E0SDJ/pOi67eap12KjtWshAyjId4GJ7hYCkxkBzUn7/Fu8VztN4Jo1PQ2Km47k7VzTY/OCH1MIvO0rA8hbZrmr53c3sUuSXXiLgLe4LwvVoxJ8TWfff3nfojebQjiOQ4TjcDezoG1/WAxBeyS4jwuLshhQX1taKG72nhqqP3hP0p3YCw2ECa42FEf6ZbNelg2ftMKAAnM7QsDkELpNK4VXLWt38Zo0re+grEg3wnxJ9yFI2+jn3bhMkhWjjVr+YTa7xnJ8N56tc/Ndx5jN91OIuZ3Sxput422tyGvVo+nJtJpqJ5SRwjrvym3yvasYMq+7KrP//L9/95vffdP84fX/aX76w/9sDGOAzxgC4l0Zj5s/EhoWzfe5LfW6W9K3JW2Ywjx0om1PBe8eWsUexnmu43i0vNkEnnbMObym7xb3aSQ7jWjPa9tqRM24235iuXqb5m1Qq7ov5JnoDXcCRL/5/W+a73/6v981r9cLsLIGr/ywhdrutL4fTsvq3WzxBtS0xVvv8+7au1tHcjXCfI/Wk+Fyme9o0oztOGAUi0sAzMPOZ88LyK/FMIL7U1cZUjt2dfqBTQBrG5znWN/r/s5RvlfXLVyGu09vXMrdB7nc2au/2sq7TRxsXE/zmAy5dQ5ta9HfDYfdLgMtNQyZ2+Fo2UYIb/TosJKX2i3PQz/0rn75enGwo7WzMZKW/UR473dv00a7b/DV3r0P7JiB8bAEE8pJ6G24YPlPgP/byZ5T/oy+gcDlYGPn9Lo5X/Zw8dHuytsynsHu6sMYyKPQ9YZpa3u52o/e3sib5Tq22xftrsRfH+NMdYIzf1Rns91Oqr0/l+uGJbSAFPd0hOvrRaFK5XetNisehrQNA7xhIkbp0PptiSur0aaEaiei7iTgzs7od+vFgnTWikQ8zfwC74tnvjwQ450V3uz0CtNtXDplttFBG8X/7yTuN2bdjoX1q7/96v8DjwfS+w==', 'research/arc-next-line/training-source-inventory.json': '...', 'tools/__init__.py': 'eNpTUlLyDPIM1o1QyE3MK03MUcjLL0lNys/PVigqzSvJzE1VKEhMzk5MT9VTUlLiAgBdtA7w', 'tools/arc_kaggle_submission.py': 'eNqtGtty47b1XV+BMi9SInHXm7bTKqsHT+N03KbJTtbJi6rhwCRkIaZIFiBtq1v/e885uBC8Wc62+5BYIHDudyCKohvV1IfVXgnBLn/6y+ryr9erd6x+LFe8rsWxqplubo9Sa1kWTDxVpaq/YUXJKlU+yEwoViqm01LJ4o5dfriOZ7Obg9SsLsucZSKXt0LxWuQnVogH3F2JQjNd5k0NAHXMrmt2L0SlGX4+sfTA81wUd4LVQtdMFrP6IFjZ1FVTL+FnmjcZotpzmYvsTVPsZSH1QWS0Xy8ZLzKmBJKpAX8t1BE31DJlewmQ1cyyBUSIihvaYnbJHnguM9wjGJBflDUQBPwVKWDfA1T4716mkufErYhnURTN9qo8siTZN3WjRJIweUTEsBvOc2JwNnNr6g7QaeF+H7g+gHTcz191Wbi/S20AV7zGLQ7qB/g5m83SnGvNPnqlXClVqvkvPG8E/blYzxj8q2AX7P7u8vr7n3+6Sj7eXN78/PHqI9uwT5EsiNtoyaJaHgVIF/98LNW9UEmqgDL8XZbHaEmw+v8AAAhWNVUtsgQUihrBE8B1Ip5E2sA6/taCKExyeZSEwmgtep7dXH28CWkakPkfIDMtj1UOKsSTxGxxt0pBvzIDrQGQq18uv//58ubHn5Lvrq++/9bwBiYqUqTL2Aye7S1pWgMJNLwuVVKj/RO1ziinuAa9KwAUQO6uGMBPHBaOvE6NEK3VJGQ1QPRslok9WOi/GqnEPC2BHUS6ZEcQFr8TVn1yTzbov689SYpLLQb6d6ct/KS8/RUIm1dcKm1BKqGbvEYhPdPvPXjuvTgt0fQbsPqC0e4AkyUSNhExsMPAAL6ypsplCnpgf/v44w8IJ1oEB3HXFhZ3gI7AWwrATxwQSyk5RYLmP78ts5OlFb9YK01ABrrmRT1PCNLinCjADIsVhgVHXNEcIQpZ+iwNiDDOS55pQrtkRmAJiSA5lOX9xspwaUl0ZGz6dDmZIzTDCDquFzqhC7hENzY7YiXgxO0JItd84aDcKZnNQ0adFiTYP+JLhfm8ZDnEtQUFvAv2fsMgbNqD+Ovrt10rjhAwOzYaQ8+DYBdx/PVbpspH7QUzjmf7djeNCj7+BmwpeNixcAgfZVYfWA9YhxRIBSE5QGxICZ6DpQXbbCwsWB14LsKoTxW4msjzBcZ2CF50/i0Sjqv4/z+TO9AvNHMAO4CEG+ADfjdiBjdA7+fFXZNzhXDFHaQ44BJ2kgAsI1nX9rYAZbveDQDurAn4HAgm1hS1nvsFPW0T7Z4ly2RqZRSuol9g5guT7JFXFabTHqEGbzdU1FzfJzJb0h9Icws7Blc7gg0PQ0dAoD+va2WI8ystZQT7+ts+PS+ADLnVoqY1MolPUQ3xoaAkB6VB9DyM6lEgCDRV8OgajrCygHqFThs68fhZQrYWX8dbOh9G7XMIBbGNAbHrgbAQ7Buqk3CzHpMaag5jGiqsQ8y6Q8kIT3iqL1xcs8KVhc2BNhs+TyTNTBwxTioqh6B2OogjB6j7vVA66joZhT5EsbXQd4tvwkWLafcidySl/w9zUyyRxKlswOJXPAFzplrdS5FnGGKArluZQfV4nkP/3fjc1jrFzgbGkKtOEDHbbcSghATJOGlr9U5AaJetj3oHfynWTIWaFlpfhAEeFCQuGbCLXnoIegpyeRBVCYU6FmQ94+iEHgJGsYegDuOOLcSAsRaDF+lLHmwP9vOLXSZuCOdIEAl5AT2tzJkXWHJsIfXIjEVy1mhxf1/iuGaN1jY2yQV6pfvxbtKKqVDFOPdYsoIfoYPyndFIHPHWiwi3AS7vo90P73rmCtEYtUAyjNYkW2cZxqFsDe136OZod8SUHLFEctXzbSPzbMrWISWXKnOZErq0K2pakRAIfMCmAqDQ9biNUDvUB0CO0R/rUkyHvNCPoC90b8WLe1jS0NgivCueHuxJdPRPbVbE9nO9/US8yCITT0sv0CWL4/h5Z/4Xs3+YXobg4XlNMZywmcDuFcGVaOnmGvtvqLplbXtZpvMS5MOuMAQRuOtvMf372hzokCkKxdaspClCdIRuhNUH6GtR/x4hNmgszUststjJ738PGFbS1rlA4RUYF5gxuozTAiXgW8E4bbKWZ1olKszbho2irJ73mwHomIYlMjnLuGNhR9PvIGNqbLUg0ua+yvO4TYQ3evExnjqnAUM933FYCWZ8B54b2WpDJ8Ne9Duea0HFKv01Wjjgvw40akUL6ANNr5kN4UxFAoMfXcMqI6UJBktVqTXDAUwQ425B7RlXpx6DVD4fJOY/W856v10P0E7plAC0cEU+qlSyovUZ7J+FdGqLiym06faUUM7qVMcuJHj9v1gLmz1hLB98NFq14SVatGXzCF+4f+v37toE2XU1jPIaR1lNcV+Ujyb1Bmq0AICxAcgBM26vnQtYmXQGAwHmHqJJiVimMRoiyyPhwkTK0WDhWQCQdaM9GwaoWcTU2BS6EqkE1b6ivbDAXhD/vINz04XPXPViv4OoOqOvxUg9EWqHWbIXIxWOtb5+NeF1R1Lc/ZaqYijqVtKFnZD03J4yHRCDUFrtmQQ4EQKpG6cNw3bcgHs/KIbJReibNThXr000Hd4IUTA299mB8pAwn/1CNtziBBOBANss360d3ToNSN5NUBrIm0Stm6rKsQKADI1dBiRn0KZQQXnWo8hbe0v7OVufqut7ps7mL1j16w26Y9QBvwPTDsmy4K22B+NhIAA/edGPIzXjZqN4l9JyLo8g20zkzM+SzxDxuw1rZ9GIuqvdzbR23SlDggWrSauGhDNWFhAwHIAPKXk/ScjgtNMfFiGKo0r0FDGQg935xG9BraDBjaXZ0M2pKRic3oWeP0oxIZ44TBJnqy7zy0kwnnl3ocVTCi9Y6I8PIjpJ7rdZIvweJaR1l7P3GmeMGbNCa7msELDo+4JKqBXZWlEWeKBRYtzPbPzcUjzcsaB/1NB8bYMpNCUXf0XgNuEgygJeWwzPYSneT4qe5/F8NNprd+P/qFRwuotnO80n2RfZyEiX3XaTZ30WJR0G5dsSZ8y+VcJ2je4se/PBEe6HWn6FGIZZcMK2eHGaEML7z5TBMFwYswuChRfTSNSw1WA4y/pk65G1z9yB/QTiMkYUtBu+zuFNJusELyxgbbv7jOHQyEioBeXLKKoysI7i0NOaEcSgg6Raf+P4pKRrIfZCJ8p9w34oC4Fm0c4NaEVga9cr1wiWKY6mapTtjnpdUmkIqa/+bpWAJgy5244VfdjsbPqCfesLJ46X9KKfKb+haQne/7em0WCDCp3xk8h60HJxx3OaOcTsB7r+lwWEW2bHpWbKwuiu2zibqDmA5HG3t5NK47Wlo3n7lgTgOSLmt7Da41jg5Wl47qJ7jiIEHSYMPXENTCXmFV51z8NB29qMvuj8ojN1s18MEYvnnjK9KbdAXau1Dq5FglJ6bQxzOsu100jDXhKEc7d2/jRNpZPAH+E0Xes7ENE5O7bHdq/GhTy+gGvKzl+Ph0zYiSOhkVliAxxgxELi9fLxUMpGpYJSpYFPtCJsiLldy4zMrC46B3vsHwYkAurj0bvFKzhGjElVgiefjEwLEY2b/llQvpNKyMgHYC4sk+4yPPm3UGVy8XQRPburRPB7fifMQ4DX3lLMes188sKNZBvyg1DvmqJ+KmoNx+dpA8FNdQw7HTMcmy5YgrbmF+aR4UeK5a6pertgX7GLNpNRTgQujhyn3ZFUUj+94Spd3fO7u1yszFOrFe1dPeBM/8svnSx7FhANUXeTql3sHzOOMR4wjni1vh2JJp1r6zaQLcaBT3mdR/DStteiotSZ2DF+YiAGvd0ZdsCI374SkfHkRGqLpG1mAYmZsLKxQex6dPzafxu0phiHjwrwUcY6IOQ5vEkJb99oi3s1wMHPwVXz3usROjSn9y5Zc6y0m6JqMK/kXpz05kbRb/McrlR6M4+W2Juso+muShQan7xxnUq5sayDJ5ePScGLjZ07f8WifxbRIhZFCvKaR029X/0pWnRuccCiaagA52r5MBkbqpwDv+5JHlKPATjJJMq6VKclq8t7UUgIP+3Nz8dagXqYgQz6bbCAUfxxZFxvnyeCJfHbXOqDO4TPHEWB45HYXYWYeqUscx0XD+CxQ8rdiz3D4eBzqJiJLZ/Nu5Xto4IsmRTikV4aLVnwtArYuMSXnPgokl5+0Ksw6SSAIQZP16II31sCUxxpVicvB6oEg9dM+LRzHj3dRgu8qtK1EvwYRmT8HRNh5qVX/9M+b/RhHkw6dbzXpyKdu+/Q/xZl+1LqyGUx968dlQbiN/6ZZXyp7pqjKOoP9GWeCZ0qWaHKN0mSlWmStPfLOBhH15+3D0JoeuauctpXDljjUUS2v4POwJAQ8wyii8U9j1arCLwA4YOSTpXYfCB1uAtWcj1DB5zBfGWhmPdiuDbvPYmCpdggj4lsbC9bu4nRz5HKv1MWsc2Wn+Kae97+9oUVYX2g9qKKwSbK/EHMTe9fkWgIcWiVtOBv9gK67A8TvszSYjfrt7nmxQWgXFCv/vulLQjfWJINOY7wjKzVT5/DO7xQImTVIDKaXvqPISV+Cz6kEY+sg63/CspfddIbPXe1bH4ZWwtF0nnO15VP51Ovw136emBwrR28GuzRsliGTwpDuhb2kgo8DG9rfVLoF1eENG4qzJFBtEn0gb/7wx839nlybH4O0McH8ZRJ2F/Px3KEZXsCWIfcM5BasieAUSgJgRj2vmA3B2ElC7EtBQXgBScVsTG7NhESA1vvdhOHZqRpeghWH3htwYH1+cEQlOKap+bZrnlrztywDQ3PNMRoXT5Ew6+nmuGzA9PbtvF5YKHLQGm0bjnqHXGb21BaKQkhJ0j0nyC72zLCvMW1L35NqAveY4w+wQiVca7EWoYxyCoqGq+FoBfGAA7Fd5JgOEoSCmFJguE8SexgzMT22X8Bx1t5XQ==', 'tools/arc_next_agent.py': 'eNrNPWtz4zaS3/0rEG5dleRQsj2bzc0q0dRNsklqrvKqzOQ+nEqloinI4loiFZLyY33+79cPAARAkKIn2b2bqsQUATQajUaju9FoRlH09c+/ilJuZCnzVIq3v3wtDmVxKKpkd5FuZXp7UcpDkpVim5S5rKpYZPlaHiT8L69FsRH7Yi134jpJb+Hd9OzsO5nLMqnlWvz8WG+LXFSPeZ08iKyClrUsD6XEwqQS66ROYpHLO1mKtNgfsh28L0ohH2R6xDrXj2cMYyo+bLF9JZKcoOR1VuTJbvcoqj38EbskvzkmNxLAFbVITNcAqMxqqJXk6+viYXr21qosiuu/y7QGoKUU18URhrRG6DDG66LYwZ8fixxA7rIKK+VrUR8PO1l9IeotkAvRyYuzpK7L7BrwFUmaMoH2h6KsY7Etqlr1EYsNjA4HW98X5S2Ocv2YJ/ssnaRAP5Emh+Q62wGq07MPW/y92wFV9keAcDjW4npXpLdZfqOojcWVuJZbmAuRAXLFfS5kvinKFIaAPUDjtUzWuyyX07MfCyHvkt0xqaHfJK/uZVkhCgysLpMsR9hIBsadaA+jhBFCneMOYERRdLYpi71YrTbH+ljK1UpVBohA9ARnpDo70++qWj+mxeFRP2+TarvLrvXPv1dFrp/rbC+5B+SLdJdUlaxMF9U6QyqaIq5ZPx4QcVXpa6BKcr2TZ2dnVEf8XBY3ZbL/piyLcvRfQABJj+PZmYB/BwSjq351XN/Immvazdy6/2H6H0H//5D5/EN5lGMF4/tsD3PBDfbJw4pmaYYcJebiM/P6Ri+QVV3AmjE1Xl/99ZWpBGtwf6i9Gn9+9e+fv27AF7AQHuo+KLm8V8WrgywJIV3v1eVnDah7KFjBYinyNQDa7IqEuru8nF7aGCFVVlUtD1UDBf+ZOjXwwS5c87VbE/hjlQP7VUF8quIInLy6fqxlz+CBbWFQWS33ptLV539+3VAaeV6uQFrV26aCMz1eYUO5zTFPiaMN/qYIlmLSzOvVq6YEJdONLE3RJY+ZytdyIwCfDDhIjiq52yjOwn+wbsWtfIwFIQytFb9zvSmNcGTVx3/ZBpuI+VxE/gxGbk1VG9aKHFEHYxKS0MsI8Ix5uscoD/D1pfhSofElDPrzzy4v29DwHwiNSopmUY2iLKfxiXsSyLgWorHTVO48NEiA1kQt6N30Guix3RvsUFmd3UmhiM49wk722zEr5XomIvEpUmgMyxZpv1pnN7KqVdfcA+xExzLXQmlabZNXf/l8hEJpuj7uDxVXjkUF0mUFsCpa7vAbdsQShWk1H0VxFItoFo3jIJla/4A4xf0qT/L5t8BFcjyFbRfWwWg8nm7lg0JSI60ZZnVTZmsHdaDkKEBK3Kr0TF4hLXcy11W+xBXtIEm7wSPDKYt7FwoyJbxEPmEAftN2H4vLZU83WI16+cRvcrIrg2UqdzufbxTXzgUW4t+/hmbC74NeUAt4g2hZ64u5zdk9IvlwgH0cNsYS/qACsUtKcTWd/vlS4OSI+6zeGl5Mix0wh7icTv+q1oBiNeran9w6qW5H+L9mbmmw9EoPFuUBjraStSoAMj5FtHkjB4KkrKNnfwz2isFWrFEo4QmKVpLWoEMREFZwEMpYY0FErrK8ggGnjM9C9bhktWjcYgSnEnHD1WUfVoauQMurSxCTexC60JyE7ylUENtTmFCdlyOC7WA+QfnSWCDDkCYMtHJG2UDVM4fVgjPHBTRzBBtnrjjW+PTsSr1u+eqQyBKxrrDAnhaqk2V/JYXAsnOYRMLfM8qTYyNq1+UR1hCy4V7WCapapJICQtfZGsyNl4xVrTfUPadrKQ/4wOvGqHyrX6jO6JuHFJQAoKVafrRbwF6e1asV7cBqX7YEBL6dshiZmzWtoH5VyuS2BdRRNldfwwLM8qM8Ue2dsZdKLgYl/D3ZO9paMWr9F+L6mO0AKFsz+yQ/knm0zqpDUoMhRxo9WBxyipp810BZ8Yp5M638EfNbGDI/uIWk7UHZpfuaFCzntWYf7sswEHB0iEtcKcxtWIpdS4H6r8UVdfnogrD1SMCBf+odNzrWm8lrWIESYVfzCDAAJrbgSZod8WueYQNCAe1WeD0YUzU4Xrm/fvgWOoT9Dm0XgGKTBCWWje5YvFFknrY04qGdV9k/ZEsTaxGpLiWyMWjkU1BrKjMxPhlG78mQpy5i8Qua1hVwrvr9A0il8pFtpsFE0iJN40sdhAhEpgIxHmhHiCkombBnAubjsU9FquuRrzE3TuL09v2HMOFIc4N1NBeEwQ9kGMdEt2+VtfA3ueEXSXlz3EtyJaif/MAiJ6gqYvHbqspucq769nhj//zm4VDy0zvVx7eFekESpxOmljVc9+ekUjj9CLIg1jVwW6351/dAYn76gN6OTrjvj9dVWmYHVfn9LksVvK+y/KcDP/6aJ+Wj/vFVUezguRvRPfCf1IOEASv0gPANaliLf6HLqJRbMG6B8p1Qvy+StcIQxKSC/nat3x2v+eGH404N5NtdUZR/y+46IcLEq8GZ1r8agD8WCszbXL35Sc3SN791QoRGUMro6lmov+GH72r9V714l5ue3uUO86OW7OlJyPOx5lxWtPEV7u28TE6vh2NeHQ/KKaQXqN+r3WPDXWPay/NptobNISnrCnXkUbSKFCIvweJQZnew34sM3Y7ZJkMPVkgxoD3HGO+wVp+eHTNbjx5Fx/S6WD+2rOowBd1FPu6yUF2k6+IAEulO7ly1W6OHGzDuvoRpke8ePXs5AfSxexQflY8mIQa7a1qQJbpqLD94zdoP+fiSKVjKCByB8Is7MF7Lm6C5SuW391CsH92mgHECK6XSpSv9ohMacAc0nzY+QpoG7ABNDhrauKsxjYV0Fp9/7EIE5E57FzzcHVSX7vbALp2TY4D/Qow8bDDY+ZMC0272bMxiBWYgi9mrM8BXHke5ZFoYEi4Vp51568DYUm5DsqYs8jXrDYaFVlFewQD3kfYzua1PL3bCXwoDaIRa/jhmG1urvlu5O4CpbcBGLUXAG62F2JJorDgB6H51GifTmNfyNrmTsGal0Ds9dG+U6jpLb5VCnc+vxo7lZOnKn4IIfGNr1tO2n7U9fVrP7m0XGI7t4Y5UZYGVW4qOjSLwRTMuRfoOoyiF0trR9LENKWrowYwFmQIgS0GsVug881g8B1GD65ABtQdOVHX5mbsEJK988cglbep6fmM6iCELpV3V9h53rEWHpmwPkupooIbcoNr+QQTG6lgs6LFNritVqY2cdjX3+GYd5LRnap/cgGQ4rmW3hxa7RLTw8AtH4iKLJ2MBctCG0+HtVW1zMdqRbkmnaOPgkMGYN7UNn/QN0l2k6WMKaqgwkxzWEWx9IN1mOzz0o1GHOyIOpnqKhWHhXllsLP5HPGm8n316VvLlcpwZCUlnIU1YKH/0ZUxuFsv0cd2Leu1V8reOtdpxHtA9P70uUa17UMPoFFaKFYdh1rh6X4CP6uEkJtdZroUZbOuwWBQ6sZD5ne/88MQPvdMC0fNaIyPbvkoGbdTilsrJNUBP1hMB3ZPujLsYoiLetOW9OoYaIplYrILal+HxaHjl53cLg8bSeLVcwdAe06ixE42F5qsuvJ0AKwZ8+2qnVj1LUCTd44HBWtAhSW+x2Q0sz31W7dHn5Q0RlzvOMBL4H9nB7rTNhu40I59gU2aMk1RxzOMWNX7TmzfwaykrtF41Lhb3BbcMaNw66mn1P60aU5ys8oFERE2foar1M8FIjweAg14I1HG0IelvaFRNDcpe3YERKtxwhL5q+tuCALWZr7XckAxn3TK2z9NkDYYxsvU2irLQnlCwVqRy4bjSwDYjm1qzQQqLyy+Nbcl+oc6J0q5q2pJJNwfFXR0Wq40aydBiq6ZeiORdyLDbKYCM9nif6ia40zK5tc0/5Z8de7u16tpyOaCxBEeh/WenBgL9JOUjD6I4xOHRaTwQgWEnvS+bC3fUTpcvHzr6Crtk2dBp64b+btMpJ2n9EFh0rRhzp0VMWSlSMtsy6Ut8fhEm3xZl1+SihUTqyMhsP0FUQGsr1Xy0J+S6LG6RTeikPsjUJmCDO+3h5oAw+Mh573Tkn5qPHljKxW8OproBp6pGLxjySHfD0HTFaIqeSgAjZCPg5kcQ+mRHM/KP4CtCv1sY8wnfUGCapN3wzGngUJDoxA+Ao8PD329+mD3N2RpZSPLeWBwkOR1j0Lg2wKlldrOtT+jL7lgaCModz75iUnIQJmk5HJdCsFulvcaK0iq56qf0rMCcdAgM0qdBmhwpWhb1oZS9miGlWlkdju6CSAFOjM8g6uDRBBNg1EMBY6mrkfrlbTU0FjmeZdlTaBTNjj5IWnMPPPcB96JWAPOgpk8a7DlGy43Ais3/P8wIauTnIvdMOz0Md5RO0QvmEDl8Br1XMJd4jGkxQac14UIAe6IFYfIiCMRHPojzF4HQZ2MtMBcXL0OlaJPj31oQQHidFFYsl4xEapv8zlRzj5ZYs9QB5a8lIXva/O+R9nSW6rtPAsg06tdpsGG/AWswttdgqLS/zTFiXHsEOFhSQQsuGHQPqPJlL7KjwBGyL3uMgrZoqWRKK3SOB8lCX54kp4LaRUHyTLAAY72Q649P077LkjfD6NcwX2B+DbHcD3j+hLTTtuAhYAJ2GeGHLtPDUWoPJPgbhKZo9Je8NtQbWH/eGzwvGC87HMOhmVpQu9E5DWi8DLKdrtk3JptwNLjlyRmlAIW+FTrEIuR9k+zBcAW1TboE70JJBUqEWKzL4CaZh1EGIXeRz1nFwUQNkOSlwarIpJCrRVM74LMLTelEGULdHWOwhFp/A2Ueh4t4FHF89s36qoY7K5QnP2gK9eD/Nlc6qiHazPUqvwDWTyWDOgUmWNC981D4jEcv2lrnpzi4RWLE1JAZnaUK/0qt+ZQ6SwKneIQ3cv4LSd9Skk9U8ZSvLiVMx94A0xe3WtWYzz1do1triZsAHQfEJy8F8X3ttP/yxc2/cdu/tP/v3P7fvLi52/+bl/b/Lnfa66gA3DY7prJnMjxgdphBL8AhGiWzdlYVoSND5X4obs2qDfuG1LJrk0i1cjwf3Y62bx76til/Pf9BPrdOAQO21XhIuBTFXNhhYDpa51Y+3hflunrBeQRGp6zVfUcK/+B7P3j504ReVB3nEjatEEKDHKixcVv7TCztMzH7CwaJLE9TR8cmtuZL2TgLVzfiI7k5hc2PWgKRrodAN/WIjkS6/K0dPkVEkY9k5iYgdqpuHYYFtmWgKxMpdOJ40jDvMdCdYM1TVrpLvGlywOvGYcVPYtgmEXPcBaMVmYD/gBw6uq6hjDqAChEUKk2zapVUj3k6PB6Bqp+MQTBO5BMOa8KB/NVqwONTpyfYwnMnc9MuDsBV0tq1C1QyuR3rX0VOegHhs+niJ9LBGh7GoAm3Wy68PK1caqt9qAOzwd32YKZ6bDEJlVhQ9Fe/le/MDtafdR9KtrXacHBiS4Ja1yRCi665pTrEEYbVe2KPNrlWy6woQAwADPnqTHgc/tjkVszcMAGu5bMKxuo4lreG3wrkCh5xWA5+QIoPNp4oanIm7lh4W6f8Nt5qzv0gHXVowWeseHcBmHA2yJIt9Zm9R+WMchN0oM5jnfhjVfDRkm8xEd7yhUmITFAIT828HbnIQEz0I4oRrIvXLMdBsGBC3kgGbKI9GTpe5GwbX+5yOGkscrgDdTI6b8fH2l7hP8YTTF19lGMer5YgImE6yRx4Ga/tD56ENnQDIzA5Hd0CC3dNzuuBveIqOG96VArOnZ7DYNdQ9hTt+WYpzAT+qY57+kMyFp9KTNxR8fMxz347yuj5JG3aTjNNgbNTZ6sDeE1LcBM/VOnDhqtYWC9hd6xXHC00Grfdcob6SIGZ8STALw2gfVDXtAFyWW2g79NtkLZNG/g1pA1PhNWMXgxoaSauaUuMot+3XaON9eR1hmGIwdqdfr8wmyfX1eB1hWGxDiv0CzjWHgPQXzWHihoCulKRFLOu7ZAqfYxqHJBUjNgpUaV61Uow/bxajrv1kOB6OrW54PVYuksbDVnB1tXbvt4ctBbMYnj/37qKr5ovw1OH6Us6kPJ2pm0s7mO+do9nmq6k8AVeyIQl8brl5AXOTfJ79x0nGqB+MNPAMEVIj4MD6KsBm9FiwX0g4ivKUEA76P146b/ajjuIRxcWwFqW/+Ippe1Gw+jALS2LQwCtv7hoITaxKC9hYuG/EkR4eoUXVf3bUC76dFekvZkYbruaLf2tJMgRNNMlpkMpzd5LsF1WwAqpUwHHPYwvkAwv4Akg8CK9nKVXS5ve1GV5OSuvupYR2t6w8PO6Or2QmOaYzeumxH5xUeW5BFPhDvNofQT1HVI3gHsq2T12T00DilaiYMO0QVXHtH8Wi9dD58MQSueRqexueEkm9GR19Bl2/dqbwD+J90D+CTdp4H6hZ1anHis2AueVZNdxnysZthSH7AE23KlHJAkVjEeJrgDEvmOJOKMRD4ZrAyQgW75dlbbSj4tkw3uJJYxhzGavzJE0zKDLRbpEfmvo+ZExZrgNZvlN3FAVvWvc7bJFDpt202S9HnHFMPr322wndQc97i0EkUKvqub0UBxG457RKDz1Hr5gALGiDJImXS67AeA8raHJmiZrNJqAJLyEmccImsmVeqC/XDAsEZEQBIgB4JOGMNEPV+Me184QdnDuY9lSZA6LBtfRmpfTOp2dxLmXJ8wVMjyTxpkpS/Ep0QzmCR7SU+iNSKTksKlbch5R4/ep9b61NXZmesKILcZo3BxRSE60Q5BymPqcl0WzSManadEwswJ/GiHNqooF+xt6fldW9Q0bj/8ojx0HoeAVhjwtSsyrtHvkzIdrcShRF87udGwKniOovEkqPaS+8DjSOUt462KlfM65+Eaag6MoUu4dTnajpTiYWuq+kMqzQjciQUxzVsr9EWMhdTZGSjBD6VPIg8/av8npxqPt2g33SbrFO61zJ6vLyMu3ctZyehlprwDwQYZ1lRUknpvkhvpctuKgnnSan5mXFMfFWM3cGC19pARU1z3Tb75NrzOC2DMai3fo51XP/y3L4m8ZunhNhpAPYGmpxyb1jyus/KQiHGobNyGy7dQi7nkMFMUikL/R8vWZC7TOuNoXSJpANMr7Mm8lJOAbtOUxx8yVXIuZlUxK6Gk8Xa2QcVersQvsBAq8Wrhbkwnl7OznX3767pe3P6ze/fj+wy+/fv3h3U8/vkdjhxlcszfw2H++/+lHpWDwmdltlq/nGmHkfJVvhe+J5zf+TeupYBaPfq0kJSbQN4Iu8PbnhZUV9YIzonJS2AcAdUGROORphZ73EnYHoMhEnGOI4L/FGnBzyIptNzHucQxSOCdFmIoVWeAiNVlMmJ9jjsWg00F1E1yBNo7tKZi7KjUo4qPTtFZ4t6cEVOEBT9/gD+37seJqfY+dsre+/fldZajxs5ZJ1Yx2g4cKlgnrTlVdHMgw0D+Tso7xZUxxUXiwqF1/1AqNpIcqfsRnBX2f5Rf75OECRNIFi9wL7ZC5YPfaSM0C5XqAhugIyfEvy2mA9zCOhTHlRzcNcG2BjraSopLus3W9jUnXhErGVKQmaJWMbmKwu8DsAqsrvRozGwF9dscKcy5ipl97Klm5hUaW4eBo8eIGySaqRic2uXcrlV9Xa8JQHCslWPXAurC6/h9Qvafi620B2Nu6Oq+bO0wXAowOhN8Ch9TbBP37QGFk+kuHzekUEJh1VxSHSm8Pa3XUXIm6oARbwHLyvlkuiWI0fXTgJAdUwDlHoL5/3eQKtHIETsW3WS1wsh/d7G5cE3AnyplcB9PozKS13Ei5xnGbrQSTnOkNBVfLnrP7cl4AnfiyuIcy3hBoPbA4WpD+HKsb/FruGafRhrZqWOexSRDXsLWb96+RusldklFWXuVA7cpYADKC/rZQ1gVu0hkDtZ0vVONtRDRlXJYP2+RYocf0rPuWiT6VV9rD+fnT+blKxap2aPaHu6jDHmnwee5JtAVbBcDuUl+cJHbBc1qTkgEALdT27Fo7IDgRWeQcVUvn9nOP1fgq3ly4+f/cznDjqrbJQfIlgXzUAB+zM715QSYjqrGj5javqsQ/XPcsKUTqNBChL5basrd6laG4Gv1vsUDdPrYGzMqzvsjNv5b9ca6emazQHofs4q5R+HaEjw+enjoYebozrEOtZj9FuPRXtMCimV5oEeXohN/21GqYPZZeRERc8YHrzKJrjOuCQMk1v4HixYD5Xfb1xvi48DrYoBeO5ooVHRlHvM/ql3Q9xeMQTtHeA3GTlVW90iAkKbTqx2J29Wr53MpzZ+t5wUx2Zh3CiFCtYK1TLUi0wIPiqOKK4w7xQ5NPky3YlYi8MdYhPF5W1YTPDbDOQnGI4xOktjp1agPZvcPcn+TznyS7B8jvF8vw8G3BP1SWdwYj/F6ZPkyuUwop3q21uHCkezCg4SQfv5yXT/KzOxXG9PSy+s6UAhJxbItdttLyruFad3VH1D9UkGwiRv70MSe6bRTtoFQ96TM99dNfZypnMC4zHq5zxQKfnpX+VRW7OyuDM9o9+5g/rDAzHySIOQLGT+pvin0Sej4MvF8g1/NLtFA4w8mKOlhl6zniYvk4jmwBbkBjXk/uSdcXby++uvga8dJa/AFMNNA39zKpjiDiGdsJRQ4lKWGKniKOnKKyEXrDJd6fmbwRT5h0NXaGEvufNIg1fjEamNV2BSwBtt4z+7MDxBjtZVUlN3heT+m1jnhNB9ReFohredgVjxgHuk1qUcv9YQedMbC3zbcxcASoCjfO+4T8PaB2m29j6I9ifMFftzAp09DKZa2Xa0LbNewTHK2ILkpZk7mi24sCtHVc/MkG4Sd255R7b6rnpMdZRBm556FE4Fp246wp1+FTREehDcdzDAaKmP6839rRBrDUIQWZp3hBPtvLKay+oi7yLFVIUawtcCyMmrCCZ21j2DaC8rPTFx8oz2LkTCgsNeDXyGcM9dpfs5fsVkI3TnnU+uvotE+DrCmcHr2s1Tk8cgO2ikKKQWQbaY5dhl6EolxLXBRN/u+peJdvaI7LI2xYZFlylJ0tuKbBvtQU4ikcH2STAAl5ctSUY9o+2rYWT1FZ7FCZiqpHoBNNN33yg/Qii1TPLfnRNIV1VLoNra8bsMAKfsxg/MzTq5elmHteQ4Wo4mNZ7jHkLYZ1C7VXzG948ogCnqMa4ibmPUHzXRoZRvunJ9dce9PSxd3AyMpLHOXxM6kfyOpvnHSFztdO3C1eDQRxv+fNiZd7n9Go5vgTM8cUVkLrYuFx+tLDJPS9lE6EdDVUhVbXFEnShxYvR5q4trh1J9MI33E7xzs18z+5oGC3dbdQWn2U4OZrHaoloaHiQvUnO9wEjv83c8lyZEX3CdCh3FZ8A5/UOXngZjX3BSIMS7NKS1YuXwLY/RoQgFUzZ9O0NTqcQEoc0HCrJcKX4lM9X062VFeN6SQ6lQ9gVKVc0B6iObF1SmFYVCntzTQYpa4ZV5BuESpRqB/CHxiYtq9JRGVrY3EHm2rWWRmGJEUT2fHTLm5s8g7LOz6kfmp1JjTF+bsv8F5/pqYZ71loAwOxfgeM4m+5/PsjSRRh4o8jVo7AEgAlS8V5+p2w+z+0tXfY4pGW6k1jRy1UrxuCXUsQ/bJLQVHMrCr9S8QC7TXa8qLp7HPzyeoACgHJDkd79h0N5uMPQeuMupkeD/zBKJqaeaS1zBVbQWAR7JJDhZ4cHus8IDaZUD2yhGCtUNzP+46u8J+1v8c2KekWHMUBU9mK6xlsN2BRg6nRR2Ia7iLyxhMtA1xgxuSIENxCaJHRRyFafBvi2NhizRZTPtvMpr6RpGa29dETfNY4TMFqOF5jcI6p3rblda4WVWPBSC/tj2EMaOSJa3+X7qwovhSXA8C3N6TuHtp1B3ZiJsAZfEPUdr0BQN2ZdCCPA6tsoUXfMu7WZVUGQWbtqMXrulgjATXaV0Ndlkf9dpNJvmRtWHiiWfh5prCjOrgQzAD5TSOCurbveTcPtBoHZtBp3y73166uq3czRFnvZy1O72zM5fZoVRNH4fZMCOWfmQVsi14WooC1zmUCKr3Sf07x+pvWBhvi0hAq2ImH9FAWJQsL1L4S2q/usmLHX4QyUi1Q1m/G/G6duxdV9NnYy2P49uuBFSqmdScdV7DxWDQOC3V6GbuB760du7n66jBc82m+7m/xiHPxalhwqVELgh+X4dQuj7siQZ4lIx2fq56FE/IvdH/5UQHv+F6XKqNPdqFnpfkuWRU9k8XHVRZcumys3eGXYXUKVnK1NNRIt3KfBG7GapfOvOlbY7TsHKXx53pJch1f7ycBX+/wYVBWdA4iQ38R+YkCJqwt5GnEmY49q2YnY6YddhZmUkAzX2fovZpoV2DLsX3CZOQzRh0XgQCBSitQbHdZiq5ovuGF7N1SyrsTCn48i6muOjlMezpfzmQmPOskn1myogmmMMgoDJdeXMWwENt+T49l+nteojaSXe4k2KE1zq3C0Dw1lc1a0rNF+2dwXGFOtD7mPbE80g1HBvoaQLc2h8IuMXF9rRNyKk026FYcANEwdGtWn1vfOLMiE53vvvkfPAt/48zipSdzUKUPyvrPu9iC6hxN22Xu0FqZ1a1ztsXSMWMMfp1zblz9TVhCy2lxfq5rPY//GeqD4TfQFv5OfmHe8+9kmW0yvivXoTd8rAO3YfFOypgq2iZXdJkrqrCiO9cKr6lusf+p0PxdsS5W2tRHwVwe/ZwRFlg8oKpRC03qU+YyTUMXHLZfAIg58SIwdFLOcqcDBcMj5s34LKCYqDMqcouHGNAp8ARTexr+JH7CoEjUodTn60Et0uDS4pCBCXXM9UeCQEgVgr9XL5SH1LtUcygzOs729azFrEPjc+WqdUCiz24+tc9uMN88JYR0T2Go1+ch20jfWU70C5EWLFU+8MJP+eKpphtDqElLwclDNi7riEi3PXFMFNYNhh0ecWxr86U2ir5qlmNu8VsDnj59bjOxNjltBsRK6vCUO5mpvwsoQRZ/Oj+3jIborqildRrZIEf1F6p82aSpKJP8lpxO6ooE11YZ6fDcHtrNd8n+ep2IdCZGk9QAiUW6MJKVf+khqFOYSu74ayYmdxBl0YcqK/9QzFdmLW12DdyX5amXgahFYx5J+yqgS2SzNBcNGq3MIdRMkV33PjNPWmS4Oq8ea7PpNB3AhDQ/YuObxonSQBczE8dlQk44PJcUlTKrHi6SMp3ksLInwJF5Pbm7QhbGI/EZh2zQ57Hbfng6jfeiSVgwYMNWOFHj6PZs+tg5f/AAknsZjzz46J0lMo6bpK9+wa4oeG07YV1AevtrHOuNO6vlWp117hQt9PQCJBybuICI2WblVOB36E5VssNSBf24HD3vRBZ+JAFHl4FyEB+rCn5QufYwW3ukNl9wlwwCRj3hUIBp84hV77cgSCc4oZM7KCzKCa7FVb2V+YSiBCkEhn/z7H9hOGwCpt6EVpfXkXMoC+ufOE7tPhPrXtHkJsFbRQV+Oa5c3+M3mu1oBw539GBjTcVA5gBKB7Joijyf/S/nIJ59', 'tools/arc_next_environment.py': 'eNqVWN1z2zYSf9dfgbgPIGOSjt02cycfZ87XJDeZubSeXpKH6jQciAQtxCTIAqAl2eP/vbsASJGSkpvoQSS+9nt/u+DZ2dnbbVuJXBhSM9mxihS85bLgMt+RVvGWKWZEIwmTBcmbuq244aQpy0pITjZrzitSNfl9Mpu9EZqtKl6Q1Q6IlKyrTEI+aa5J262ABZ4CRsDidnf7/uJ297FR+ZoUQhslVp3loptO5VxHM8kfuCK54iCJgTOaNDCsmq4gQpaKwZkuN53iCfm9kzBHGCkV12sgbtZA6cfk8vIC/q5m/xGy25Lt315nr38isjF81TT3hMsHoRpZA/lrmCV1U/Dq4t+3n0jTcq8z34Jsmqw5sJmdnZ3NStXUJMvKDjlnGRF12ygDtgGy9oie+SleM1H1gzXT60qs+qF7wERSc8MKZli/8kU30jFpmcEjPYdbGPab2oqZslF1P1a8f9PdqlUN2G8QQ++G10fRlqLis9kMnEP0mgXII5zPCPzWqZcxgYWrn18HoZ3eCLO2vN3eBEwjA6pWNCRMk9KdxR/IQ1boBWG4CipWrwo2LxPFWRFcvrr66SX+hdGK0nC+TroWlObByjFRHKwpyTpZ820h7rg2wN1JuVFAz/KOHljVcS8tTqR7seycKO104nwWhHPFhObkMx57q1SjAvrOxsfI8YQ/CIx0DvGqjZDO64r/2QkIPBoOzBJIA9if1PeFUIEb6PSj6nhk+WXNvR2NTljRM8O3JkCvJkVXtzqwWkQa/JHd850nITDdTHoVntP/STrojrmVCRv/Zjd2lvWKd2fyh2jfwdOto1ce916RrOY6XUjrHpskjwnOVQJtjCaTCeS6RoIBTTATY8iu5uLD2483b24+3tBwORCD3RV439IMX6SXJwz8G0CCNoAUTBUeG/oIJ70ih+bFX63vUpswkBBaszueYQ5kq53hOnh0UWT5Ll4tw3CiXgqpCWEfKLqIs2R5TiMa0wgILuivsEyXYVI1G4jJ8BoARYN/U7v42Q3oRD/IYhAvKbuqqpnJwSYLFj++iv++9M94+ZJGyDZEODq9/Sb+A3Ym5y+yGMXxXMO9V2zMH9nuk9Ss7BG1N9bIRj5LkHlP0weKQ2mejQI7cDiaITJnNn1GAR69jLi0SJ2+A2DF+AOnVZUbeUHBGmbX8sDvhGDRVl/Azgp1t4v+4GTxRFj8C6Y5k6TZSMB0jLZ8DVXhIBC8/T3Dudf3iQIL02k6p4WvL+Biyc2mUfdZDtxh6VVEvSjDzPMACh4tE4BCw+sAQpfaikBRjWG1ZvkaKhpkBQoBmRJQVzGA282HN/C0Lgciibd+hpmymF8thxPBj9HlZRjh4yocOfxrSDQpTIdla4JTB6ZCZzr8G7l1sCLOfQsGb3yhHBX6MfyJI89gFKUWxKqGFTqwnA8jLLRp6nM2DBcUKlHbaF5kBkSQQt6NA1TT5YJCO1GDHddltiovXwMpxXw+tiy/BxjQKZIHUn5Il9etNVNm2F1a0rx9OnQIOPJLo55PzAsJ89SSN9h4ADBuLDBu0Hk9iwW1iyieTUXgaaNoYIubNwuK2CstwNjOiELztKtcWB1uWM5G8OlYfwU/325ZbvZtkt3rEQEFNWvwjY8TeKuF1mBW2ldR6zNrXIB84HxO05SePyzoQ491rg5YBBnrnEClqrUvCLAKGeKNcEQ5YS0GTeA2kH8Seu40AmRe0E5VFDDvB9dDpJM1Nwd4PETwiWoaXjujp7h+0XvAT/r9e3UxC9xGP+JFYraGXvvhuAZjZU2+NEIGY3X6iosUfyC/yWpHVpAHajfYhthubMOrKr6XAGB77yh+ZztXrm3/6PKBrDpRFZEn+PmX/46yLIIjNbSfPW2iAeLapN0hsHTSkX+AzTAE97KiFgZVssT2rV2iOhksML75luedQUyEolcDULWihf8C6GCiYimMG9Apdjqlc8DGOY0GXIJlbD22MfotomtjWj2/uGh3rUgadXehBbb7lgyamUagbuB8EcKkchPe2OEyAlTPx32QkKANhMAufXruq6/Gd7w7QJC68MKYRADBiMTGiBeeR3JXNauAvkw264qO8XRcA9NTbdKkpsNmJN3zP5F2H+CmIkDTYQ84mTSSjxESELOpHiZdS7/bJtsy9cNrbKShtA4N9ilh+uiaNgV82/Ic9E8HMLKUJ3uAiGf0Iu33jzPctyWARsEEhNLUdqVWAASszT4j09SJvAfDPeEeAw/al9NV7WPTxhXc2yqPWYUoSxDMpVArpAQeQ/q4fJkDgNh+am+lPmgWg8DL9IlaLebW84O2c/8S9arMnSIRtVWIzi0F7B+CEB6ZFo/8eWBkg7DHsxFeOprnlMQxkksd7Tk9d8QnzQpkcNA7K/xHisM+Kk5V3vdyuEH38TSOMTh+AsxtiR3h3ADFOO+wbrL1NOr5zLJKh2PYUzznojVgYuyDmIFGSyihtxdM5bEEIrG/7MfWqTGyiB8uARXGHUDvAIz6o84gmsYOdcUU3NM3X766etMFAC790niTf8FljGvw7xArhxwmRhnLdmStcAgnPcSTPiSHjSV0MVILIx545pIiA2NxBaEMUVXQub3J0bu2ywCfjag5qiNKgWu2sXaR5260o/JmDZpgd0Uj74shxHxTu8+976gC/qxFb9k4oLeDEl5jcOi9ngL6UYKPyoQ3m00IMPwU/Kf2PC4D3ym4PU5Pk4H2qGNV+oTpOj/+jNJ3fIG7ovXdzhj+n8dw7Mi9SL9RG947M04TdagTDuEcwA2pvf8oNioW0Pt+I8VGnfEowfaB6QQ9DEtX+o7y73RwQaD3Eo5j2Yrpgvf7s9RlOqhzCO5mWgk0OSfHyIdfAcleFL+VHsrxf1PqMK0OjemTC6Yn35tGN0t/gy5iqIyx6D1ORylIQDJOhtYKtgL8j7ZOhfaeyRsQm86x7R+QCqy2dxv4C3Loa/6akhxpdXT2tMbhkfy/QlNzQBatu79F49dRsMfvQJUpAxcOvv9ges+VBIeuOAQDd59LCV7Cr61rBabIL5/e3NgR3CRXosKvPdDIwtUSBOhk77yEPs/+AoRljuE=', 'tools/arc_next_materials.py': 'eNq1Wllz2zgSftevQDIPlDKUfCSZcZTlgzf2pFybcVKT7L5oVCyIhCSsKZJLQHa8Lv337cZF8JDkVM26aiYiATS6G3183cTLly+vv5cZT7gkxUPOqnGRZ4+k3C7gHdlQySpOM1JWrKQVlbzI3xO+KYtKCkIrRjgskZPB4LNaxfOcpaRYLoEertJUZEV5zvMVkVTcwbI8JZsiZRkQkHxJE0Mqhf2zgqYsnQy+rVm9TBTbKoGtBBHAqSSLR0WJ8JTlkstHsmDLAgjQ/JFkdMEyy5ooWSKR3G1B/kFXq4yF5OOHLyFZ8CwDyiFJKqaI0CwkRUXEdrHhQoCUpCiZlhf3BYkztoGJSOzly5eDZVVsSBwvt3JbsTg2KgEO8kKqVWIwMO/WVKwzvrCP/xZFrpeXVOKAXfsFHu2kig0Gl398iD98/v33m28kIsHy9fnF6+Wv57++ubig6buLJXv761tGf6FvafLu4oy9oWcXF+zNu2Dw9cunm2/x1+vrK1zHKy6+j2mVjHP2XY6tTserqtiWYnx/FgwGg5QtSULzIucJzYb3NNuy0XRA4K9iIGCumJ6k200p9GgIZ1LJ+I49iuhbpZ61fRSViIZBGIQkmAajkNAsKx7inObRbzQTbDRheQJHPxyZXVO+YkL2bWnUNhFrev72l2GbvdFkzb6b1ZbYQ8Uli5HXIeo2JD5ZfAMKQS2r0ZF7OwHO4Wgnm7uUV0P9YMVi37mQcXGnHvUSvtSr1JAYGvKKc8oFI//CTa+rqqiGwSVZVkysaz9KgV+eO7uq2H+2HExwSgLyMxGy0qx5vGmZFo+SiY4OYMki+DMPrPz6TIc8vwcJiuoxJK9C50VxUmxzGb0+D4GJe5YVJdqzefvGSAGyudWTFZPDICk2Gy6DEXkREc8iwVdaExnypASLkXER8zzJtuDMsBYkBb8gygQOqeuLjh+tsKEjktvOKS3QalryjAk42RY/6nXgjgz35wJCgqR5woZqNCQZHOEIhclYrt8pQc9OT08P8fmhwHggmQ14OH+sQpKLWXvZXTzGPAV2n3aae9SkZBtYoCWp9zUWi6OzAB+CuRszElVsstxmGZhXsh5WQUolPbEsnMxOx+/oeDl/utj9OUGvAK9U5lVv0S+eiuUuiDuR1LGqyMoyvuKLjBmR8A+l14LVHjYRwLrPsp0Esmo1gPAdMQLH95vTHbCs5V9xGS+yYhFDPDgL5kdluNpiUgO3q7OEtlmawb5ICZJAnoJcnhCKqZnhcm50r4MSze8YSodxj6VDNTMkEACjjG4WKdWxZtoOW0MvGv9s4pGLgY0Y5ixVPpZs2PRa50E8l1Zn5+RvUcu58c3FYcttLdhsBaRTRs4nE1gYNHnohIk+Ns5w085MfHl+mJPuGsvM2WRy7pgx2eApAEcB+wimJqWd1CkNjstLZ5B5BANvm5Ja9WHDWAKNJ2IT2qZeXIPFznFjfYQwbo7IjYxa9JxSeSpguraV2bSp63lrkS9/Yx2eIPwH5tJRUZvGNt8Klsb9+ze3R5J7lh/ipI+LaZuOQmWxBmGxAmBxxSgegUqivcrX4XlKnsCFpsbx4OdcBUT4gTFijyKBp2eoardzWZGnDYAB9j1UBm6yqDFqzAUNq8acYKbA0+tTGGyIUtOYnc6fQUbN6qUEwFVzVBUPHUK4Xr1/0abVotKglLAs6/PXU+QAB/Hfd0rb6gnVjZvgC/iBzwZoHXLiT2wFGQL8R2m5neqM786A3mw675CemwMSNOeS/5fF2jSGiyJ9ROSlwXt9aCg8jilN2OFZoLBRMH8GrjB1hIC9wKeXS1ZZgKDyQeQH77PhIlBvLTBzu9fxW+OvU5yhRhqQ1DCtaDT4bWWyZ/ANkU2CQ23IRyx+kGCTe4U8Ig3SsYASmtFmQoE5zh5SniiDEACU9ABw+KTjGAZQUKgMdoc4++zQAe7NcuV9ba3+RK4AKSywjmKAKNLCGKMqy3B/MBDYUdVtRQWQIkeCsDcgIZalYqKtqHhAeIcbzQyL84ZoOGGf18DA38j5QYgOaxiFRfKhgDCygdJNmsKvpLyqMfp7oE/oVhaQhgCdQkGc0UQVhYo3IzNSQHZncwfvjM0jN9MGGmp7vH8s1uOfICGVW4mnUmwl/todhT4NIUSyZhvaOhnH6YSWJcvTod1mqsMleqx5Mx/VW/uj5tV8tGslan1EU01/F8KbpsVPlQljmrb5tYWZOr4UdsJc489EgGkdIJBjY08x2lOs7SnW9qbAgapFbIqw5RkGIZts/DLKq9pUTcVyCug31TXtAe6eV3uFZMkA+Ea3Re6lKLOHtQ3MpJ4ZW2VDOSO3KHuQcqEWoKXkDIy5uouhVMxw8HRnbLMuPg1O917pYxQsA+fUM7r1ZEug5mOfcF0MaeXzdu4rpbWA93Asy8dYd588pLP3cP4qBn8it1qHBEM9HkLqhzIuigx+pUQ1ckDLkFsyYFYXX6yC05NQ+pUFZF4TxkBkdcxIC0+6llXR2FYZugAGGwyApg8EbyGs5p7PGipDGGnVQA8cykWzAIdBGXzDwE2jX05HhGIgEyUEBjbtGKzRtp0wQfRmUpgSKCTJGoRk+QqLZhB1q1pcIaxIGC+lUPVs6P3ngp8ppkJX5DoDmzWh4HyCMxpGAEIA4WWwlrIU05OTij5MIJastwvArVVS5BLbNgDjsRgoK/DeE4Ai48uPN+Pzk6ca1e9OMEX31tEYLnAPp9NmeKwl9KvCfsCC9L16GPXmL1L09hTDnk7aYHzePK2fyKVvxDpbgkXJNYOkifnz2x+XN7c3tx9buaxYEtqipFOCzV/a2ydgm3Uu1o2Ycd3aweyktpTVVq4nDYK1ifhit/KBy+Cz6fhsfiSwm+oKwciUzOoU1aIEhFyy2s13DZrOVn2eZn0EbD6bt0NlT19QH7PXbvQj6wmwXJektvHijvjoWmRq3Dg8R0SNHSPgmce4PhNLon5zjI459aIK2lSdUp1w9sUxmtphxtap7Hr7bBp6GA9lUWSATqokxiI/rvtqOjDqLVx434BHLmErb/vO2A8o+qBZes7aKLyxLj3kyIep9jGHvU/Lfa1qFTMrds/xM0Xk9S4O0lcEY+MEItorhMqRvfM6ArVwXw1FbLpWpYSlrr79GJC2d3dEbv4+/iLPcn+kn2OxXaed44zuUGNGybMG8DiG+GDjYeN7VjPOivcKrbk0ZHvZdQgN+plXMJXm4gFA+l6c+gw45KOhV2EXgO7v/Xsro35saGvYqAMMD8OuH4GFXhHrM3DSjakjBVPMp5HR6EVk2TtU7V0j0sSjw1a5+ZxoSqP3BFAFofZLFut8uoEiMEFpg+eHKRTEj0SalCvaRWjdO+qf2VLB/ojVndcNHk7HWF/aoZlSbFzip1hlnnOXF0GlOLPuXLR81Zaqh0jBQfcQ6oaSZ52Z9yXCbNEqa+sQEam+ogMfStmqv9gCIAZd1OCib2YLYPgtygMyaSDiUqPmaLaXvsMfP0T+kMccAgId56kn4LHupelAwckBQNAh7saf55pg+vdQW/k4V3/k06ftuos6fkd7me1FGy3mfJ9wGaFtr81CRRk+NlpdbaCPs9XYexHdd97ppldjlWlfqtnmt7WA8L5pA/310vN8x/Q+y6qAmI2fQFuu00njgUo2nKVjZmgEYbuxcBp28jtm6r0RI+xL7s0V3dDQ16RR10eG6v94CeFIh6ZuqqiWmxlzbbcFxPBpV3V/h9eM5vpWjH8BhggotaGQb3W7/+qeDWIXJWB9L6bGA/sSkFohbPr5/fPV9af4w+Xt1c3V5bfrryH5cnN7e30V/3bzCZ80UnYIyQZydXGCQ4ZkzZSlNlxvV1AcrpYULBEqcbuVyGkp1oV03KoV+ppP1GFk5o5u/uzOFFflfvQUuLXBtLaAAM6nEFzlZ/165r+a4wQNmb1h86L9Pctao/1I5SvN43znsaXuPESN6KNv3lh1jPU8HYOe1wPD71Q9FBUDY3swNqhxobgdtppCnYP0KYa1+p67z6CnbXQkZvQfVzeSdI3ds3X/Fket8FrqVtLy57RykR461kN3cVNbsMN+JKPJnW63sO80Abunm/rmmuHNIkkNINu4kTUuwRxpNLz3amlPqFD/tr3KS7KkPGvcnMkFBHeImXj/RGyBR32JRgrDt7m50uJ8YgjaCy4psSfqwi+ENgjRFVkxSSCXVMggLMUDAAwBACJV9/PWEAOhbipZpWl24sMQvRNCfNTnrMTVt0d8tf7LCjClGBQY4dc6P3qEECTvWH64W68vqoFuQUoAa/gZabjH7/d+j+CrHL+AOyIA6krJN5CwqleNWphWK6HevJos9Be3V5NS2n/XtrS2tX5RSF2Rxcra4xgxjGo4D0fm8pqYnc1PQEuCQSpY15cjMigjA58YqOmugZi8TU60D44NbsKp+zFTQvNUhZYINxom+lOy6mbWO80CNw1xDXhwMvMCwjyK3MOocx3KLdUXusS2NJcw4ZzdzdEgnM2bMdPv876Igk83H65vv14HPRn+n/ldDtZIfMruTqp/LahIH6OmosDLWC5AqJMahfwcjM1mE/kd1je15rPoPlRBOFLc4idpjQ6PfQtzK3Q3Y34skJlPyIttnmKAMHz3fA6UYJRMNlOY1V2zbY3zetJVc9wX/UWEYkyPo9P93KmgaUj7VyLrD917sEwj35F+xIgNt2M57/+SSI/0zrzM6XMeWNngdSvWHZGkhXNqRES6YbYXfWJO2g3+B8ymYqk=', 'tools/arc_next_models.py': 'eNq1fXuT2za25//9Kbi6tRVpRlLz/XCmp9aO7cS1juMbO3N3t9fFAkmwm2O9Ikpte3z93fd3DkASpKjuztzZ1IxbBPE4ODhvHICTyeRttdnIYm5ty3JVbeTcKo+r1eJum4vsuBL7L1a9FihYbwu5stbiIPeVWFliU1gr8Y8v1k8vrdVWFHK/vLh4td5t94dqc2MdbqvaQpPjSlo7uS+3+3VtbbaW6qXievXc2sjDp+3+oyXyXNZ43u6tH9/+ZuUYsF5e/MyVsy8HWVtiL1H7Tu6tYvtpwyMW1q3cy6X1cwPTWmyqUtaH2ipkne+rTFLREZ1hEgdxI4sL7ux767irD3sp1ta7n54u3CDEdPLb7b62MrnafrKqTSF3Ev9sDmia4dFCJzzPRSbFnmZYVisJGN/fSuvfP8mNtd8eD9KqaOwVRt4DKLSVnzH7ao2ONM7UHEqaoJWJ/KN12FoCEz7WqPD65+XFZDK5KPfbtZWm5fFw3Ms01fhC+832IA7VdlNfqDqFOIh8JeoaGNKV2qILXXAr6ltA1DyqPyhYruVBUO3TN8dDtWpK/15vN81vrP5t83vbDrBbiQMtsAJphzroowHnrdEEOD/mh/bpmO32W1r3tuRLrfo4fNkRinXx082Xi4sLnpP1w3Zz2Iv88GK/3+6nfxOro+SfsycXFv4D8p4CudVqka+2NSikode5lR2LG3lgEpOfZX4kNFq57s66q7YrxuyS8H9xUcjS2svfj9VeTlGpqOjdEyvbbtHVGjCDmp7QhGbW4q/Wm+1GKgCqElR+sLomXEr/7UVVywH8uqOZHjAXm+2mAvFP72hiT2jm3D9TrepqL0ETG16VZXFc72pVd27VQFX6UX6pr97v+VnuBIgQRH01ncwnc2vyZDKbW3JTE02JOq+qq5diVct5C2L/P5Do9lO6ERtVbbaUmxwMOZ0cD+UinjRAF9UNWG4IMTDTg1cT4bK+FWC36WCms9nyVn7WPTUdE4Oluj4R1ROmpX7v1C3Y6WrY/4zffqoOt0yPyy24eTrZZ5OZJWpL8X63NCBeK1ttwYzVxqpAMNOVWGeFeKJrLvFPMY2tP1mO7fr6D3CZTSazJz3sKXiWxx34Sk65z9kQDXh/OtlPe4yb0qoac51bA6x2dMazwgJDsCzXH4tqP1UPzerLz1V9SLcf+VGBcJDrHTDFLQkzWNq15NGW9Mv6szVZHta7SVd7qaBi6jtZMtTPJv93Y1bfS4iCXPVJE/sfrSyagq3/ITcaGsXKLN3f7WSuZgTSZXxrdO22dQXi7ZXdVTVzYVMi9vktAMxJSnalrGJSiBCjjLVZWlf/QFm1OWgU7jFtrHbdldUYEgyZFtQc5UWVH67RyZxqfOAqYO2yuknLSq6KXg0skqqh1GHKs2QQgPXJ0+NhyzN+ud3/wNL+9c8Trm6qCCVhUJ9ZDij8+ZfnL16/Q8FXJd9u9tBxB7nwl+4iziZPOixOW0IcVgLrV9l6oUsvh2+7dmVcSCfMi7zI8jCPoiQXgROFti/twC5iN88dJ5G+Tz3+qHoxZzNvRzY6dWzbC9y5FUeJEyRuEtpz6+vk2UsnBPBd4beuxdfJ5rhOb6sC6jddiS9YIFT10W6iC2kduSgJaW4brOFaFhVYrnnluLFtG4KNuxSHA5AMCkpvwc/UqQfA+BWIL2Wybl/F32aq/eR3aHdvGSySc+g2KqA7MgYu/70pekZFuWu7rhe6XhCFrnTyOHJEYed5Ftp5LGWQCc+LskSYa8E9pAHht9ElYvWj3JBhgd/U7e+qCn66fuy5QFASBp5j+14Ym+jUyFYv7cCN0fal59L8Y/9hzDOS/gDm3Th+GPNOeBbzPt6QCWq04oFQi94a8PaY6tUajPtefj6839K/wAoLG72ItZTFgtTXHjS/gDyHpIWJcWZFz9UGZM8gDJ/DWpSLd6h0Sf8sflA1ny1+7Wp2nSWJG2VR5PqFzJwyduKkEE6ehyKwPTcuyiyOgyK0C+r9NRSPGPDUispMjgoCxwP6Yjew/dB17Zg4Si1oV/avXlbfw5j/ZYb6dnHxb9bLarWCacaWXiPUFzv2QsgJqXKy5ZWYvTxsP8oNQNgre5tNaHgcr1++sxrzdYkuYYXDuCIfod4e97ls7XqGgRwNGGVrrA40BUnaWo0uNtoWxMjkd+yPm+XF21dv3rx4nr589frFO1O+d7/wz4cPJJO/68vS756g6NcXT5///GK5LvD0ne14SZg5Ig7AfXGYxGEphevmpRBxWNhuEBRO4jheETmltBNpZ2GS525WCt+zo0J8N26cfZffikNKOhdGq1z+vdr8XdBwpZ2hYRklmQd9B+ktAxEFsQ9ZLmSeuKWfYHzpB3lRRiTf46hIyjzw8yIBMPHZ4XgxlmSe0DBxEDpO6UvPDz2v8KXjxxLkXIZ+lruFJ1xPZviV2Q4K4qiEKRWFvpR56YYOhjo3zE0r3tLBiE6WZKK0RZQESeE5BfoM4rIMs8AOczcq7EREXuElQS6KPM8jGRaFU/hZJIGOIinOjbiWe5hhy8PnA42SYXV8F/Py/MQGGsGaPqbj2GGRRJ7tB0HpR76fAYle7jpSgg/xRti2jMB47tlRSMQsbPznLLYl//CXtShhfGxgttc8tpCFX3pQXIVTutKLElcUcRKKPAhdx3F9qMks80uQUeS6peO5MnR8x46jwJFJmD88tnt27DCRkQiSLHbs0s9tN/G8JIsEpEjmFyK3c0xVhpBTIgxl7GG2fgxrICPajW3vEWN7Z8fO3Txwy8LNhAhlHqAqMJ4nsQ8mKCKMYotAxHaZJ8LJfDsEsqNcgNxK245FnDw8tn92bLso89yOfCfxcifH9N3ILotQJjB8cgmy8jNIU0fGEbg4cXJXeH6UhUGU255dlvfP2xxsSQGFzy01F14kgb3ED+MC6ww94Be5b2MZPS+AFCgjPywDvwhckYcwvaSLwTzphYH0HAj4c+PW0GKQnSkLzTpdi107pEx8LyglZlfEXlwS14NFiZYjEUuvLD3XzWRUhtBHYBosCICDxAoiP8sCNzs3ZCufO+EQe0kESwPgx5HvRW4IpOVClG7uYT5xIsoiEEUAKoYcLAFECZukdEUGNekW0YMjDWWDC4qAYClKB9RZOLZwi7AUcexgTePSyyEJwCJ+4PhhUoLKXVBvEpWO7/tJDKScG5B9hnYUUJ8kRrBBcxkstrwQoMGwjLPQK0oozjAGBYF3Q/Crm0iZOC6QYOdJYEdgrO9IIX/XmYqsK16/+uHFm3cvmP8zWeSQVTAF7SB2bSwMUBeUtpPFoZMU4PPIFiEosZCeCFAlh/j1YcCVjgyiMU3RV0V5UIKRIJ+LQpCWKTPCO8jPCwRkd1FkDoSOlwgbdmKSQ2eEXiREYtsgG+iqsQHOqCHhCyljUULeSzuCz+BjpWPIjCTJZBjaDtBWOD7WKCnLRBaoBGVlg+3cuIycYHSo/qIXNujMjtC/A/TIwgbpgDchl+HAeCFoGuvvuphFCV0LuzcPgbMy88rEg2IaHaKvDAQUTRAWUQbCklCcfiI9EnwyCbJYgsygVnzpQh7GssDSxQLqKQuzxIfmASWMjjAUDA8ohQIaCWSa+RCKTkwK3Att4UDDBlDBIMAwyDA1O/RAJbIA5QgfnECyFADZ8vEwnFcOnpNHRSRBN0HhgpR9O0NRHiR27IG//NyPYNIGiQMEwdWLIRtDH5SVwaGxQcyPh+G8koC49EJ4ScKHOyp8P/dyKL4QCtKGehRuGcGR8jNSCXBTsUK2U8AagZqEDVXI/PEwnFcWWQi6AVsAF74PmgUJeKDy0i+l6wYl5GURyCiMiZgh4Uo3gGQQsRAOeAvk9ygYBooCa+mBGbPACcE4jlfCwgm90InyDNTpgZpj2F+24ybQXQGoP3FzoETCcIH1IkepfLeXOuy6PRWlEMqBH9hQwOAX9AoGk07i+tAWEHFQQxgevqYPNyayIXpAnI4TO2BoXwLIsfFONUQAnvcLmFiOSNwsSYDQEHxb+tB1Afg6C3OYrrkgCg9ct4Dxh8XMoYjA05nv3jvKcEqeA7/bLiAR7CSyMUiRBVLEZVzmMVY090RIui/JEy/KnciXYeRDG0IcidCHuhob7K4q5Da9D5ERCAEaDz2WohQw6TFZ6KMih7IKI/BJGSdOgL+gkYAMS1s6TgFjllyEwvFGR+3poxxWSubnGRbJK7AGMQxhUk7wNCPhBVguT2Iwm7xMmRUS1n0Ad9PPk6KEDeUpfXTG0T11ZDIQQiHD0nFtuK+B49hJnGWe7ZC7GcBci/IktH0IS5joYenCYvNtn/wRyKHSO+fInJXvCeSd7whYH9BFOQQxuB6kCO2QJOgOzoUnvMTGaJ60iyyB5S6dEHB5vusB5w+Od97XiDEjLBw0VuS7wGUhpAhhMYBog8CDHejlHgxGH7wvwCiwWiCIsaJ5GdglXK4Hhx5xB6ITkzzKyEEsaDo+2ccSGi0qYzeEBxL4vgffA9aV7cIzgPQt3cQWfgGL1c9CqCXvDwDhngUihtCBOPeCwoeFEBD3ixCEANOhIKTDPYAgCLHcAbwRKEwofeglWhW7cOLyDwDhncdEApsV8gFiD1YjfBDXhukDsw6GKvweD7aXD/3syjLJCtsXtp9ATsvCjSEiw8T+A0D4Z4HwPQfEZoMBvMyFd+LFMN3jAoZ87EUe2RqR54G5bF8mWZnbsEWKMg6zIvQhuov4DwARnAXCg92ZBxFpHjgNBexoLIbnRC7UTOyFZSjg0peYORxhgOPnMB79PAeXZhnkkf8HgAjPL4fMYAFDiuRwfWEtijLLoCFgGmdBjtFhC2WuCy8FMMCTC0I4GHkAlxZCyoHG+gNARGeByCIYizKCjioglAQcdZi0UMWZWwYRvIgoggkIyQpzugAvg3xLCXEMI7PIylJGjwPiHsUMqeB7YR5JmD4SlrsLBRnLAJI7CyCvKNYSOB4MIZAKJBTMSJfYNYDvXAbw/h4E4B5XroBD7sLICeExxdD+8JBj2D1BBmfdj5LCxfRDMChcHaxFnvtQ3zAO/MQTDsR38eDYpxobdmjggpjIqoxzPwkESX+IRVgEkFIMQgKlAxPUg/cjYZpknifzyBEgEPkHhhwKYzBTDF8IHjg5cWEJSWi7PvySLIJgjj1ZxFh3HxRfFMQTMMd9Pw5zH6667cfiu2/fLi7e/fb27esXP7948/7paxXXOxe86xyyr9+prUTWtZFTuBihTMBsYQlyhnkks8glq1yCCuGvQWe48CxgisJOchII6yIvITltkT3IeLzgmL3aacOI4JQgekwjDnamx/2KwLw9HHb1k8vLvfi0vKkOt8fsWMs9bXHTLmG+XV+e2wZaic3NUdxIleRRX8J/DwIZyjKGSQTahWMDMReGgS+ywMVL8Jm0s/iywde3Ee/2QQNjHNkU2IIYyYBomO8hHGsfLl6Ze7ACPV86EOpgYS/OBS1LLJIyy0O/IOMJgkA8BtnnsA7z6w81/oPYP7t1cAnPUcY2rP0ogleZCMjYHCyc8SLA0nFgWoH5bAPf3/TeYPrD0zfPXz1/+l4RNe+ifp10+6eTJxbJkmVXMrcmTdS9e6me7539pNtEbdp1Jei0209tXncl30Y7pn13ADzn2rT3rnY7lyDLdT2dYYo//PLri/Tt0x/+59MfeX7TCWaQ39K2yAH0W1PaCQ1Iw8uSNn0mlEq04vQbrtUIlnoyMrnJ7fHmBuRYCiwKVoxaGEKfHjmo4U5mFz/++vTNq/cv0meA5sWb54RsDcwTa+Iu46V9AhVeBHjhtOChwF66DlUdQ4gJO6o6S8dWjY1ZqC7cpUvlQ/C5keee7d+cG3ekodaTRJG3dJbhBJi/ePr27a+//O3F85TX4D9+eqH3ob/rgGTO7R4XCuDF7ou32Gw3ciE2X5afbpk7oBZgUZchZLmARpZREscwZeCyBBI+ICzLMgc/hx5qcXwBln5pe1CmnuOUjqPclcF8eXyjLEXZQiFgFAjYhRHUCUzUBFZlkhcih9wIyhxOXyYS2iiB35dRXIfcxQDGP/xQx/cD9B7AF2QgFLJ4bPVzwUgbHTEOMEW0h9+ZuAWMJC+HyQLvAm6nH8aOiEP4hzLEOBBuMJtjmHWO7QEi2BRhpEYk2uHx6MdCUdD4BEMfaimgHQ+MG/k27FIYrVGQxVQEzZkkHownuPDwYzLfyUM3hrEAvGC+mR6ub2x9/a4XIyGSWeQ7z7EXIqu8xRrDr6rN8XPqpk6Ufo7DNPSXbSlZDU2hBhJma0lhuDyhiFmeu1EcS6AA9jzFv2FZZeTuwIqPAicDZPCsoqKA0wqLI4p8BWTHEgxj97hQ7EEwJv8siB7twiUwPfLcseFj+47vwQQRdpEE+JXB1oNDHeEvMOckMYUGJVwwGUNZ5s2ysXTQ0OHXgoXEn/Oj48aMQUf/a8LnxgNQYEeDIEGncCMyIRxYWqBJPEnXzeH/e9IL4gTWjiODMrFRkgWRkws3LgMyyPrq8Awkrv73XkhgvvpwbWBnRknmglLDBIYgRhRODp8Q/l/gR5mT+3A9UDdge7fw4BtRVCfKNVIMAalwYxQsWFyOknYAe67MkoCsq9CF9wdbL8Ai0V5gDBOzSFxZRAFe5o4T+HkgnTKAzwavPCwKW7IhSJlWN/KQkr6ZqjyhJu2I86sGKUlNAmBbs9NREJrys8gpPRSAWh83208by6hXN42LSS8BTLW+bmt+0PlfKTFZupbrTO6n++320GSAUWpWByCVdQlgEMdUMKU6sx7EtN3NmV1VnYqs3q6OBzmdqQRUyvWiH5PlcsL74phUk052qPuKY9Lm+yrQrPWxPlgZOrAIYIxHWZN3kjvQM83ReUX5bwCPZmJd8lt+92+0eV9vxK6+3VK+5xrU9rGm5Kqt2ozfraq8OrT5uugsv1Ub+ysJq6oieJYWWVK17rA+VKuVdSsABDrB+/x2kL+rUws4967+XmGuqOp8eyf3vFTZ9oi6xbKHwnYahENKPpjOsOjrqq4pNXUFy2els5kVaqA8rT9b5lLwgrfd9BL9GrSmTcqypjGV+0cpY/+p179Ppfebp01nJ93gD6UOYkHoD1NSP2+tzZ79CUiyDkA5UzcmcVOh0heVDLrCb5D76sv3Oo25mQYE72WTlg3SIGxxHi2n1JFtdzXCdhpNRCGajjsczNrMuJxyM/DeSMowucdcMUqdmzZtaLHaLO+TxO0235so67gRd6JaiWwlNQmrXJMr61oNQHYqLey8hSglYiLGoaxbWbSjNqarkRaqGfWEwecGrdB/2wzuwh1Pdpj52lVqptpVvurDZM6aYQTBMk8MybOd5lLsiFWmXyfMw0/0TCdqfDw3Y1FZ6y6hnGUG2PQwneEPewHfVN/1cbdbSZ30fg8WDQSeuuhLUExHLHP4k7MR7Gqx9Aj8tsKcqwwAN/F4bU7zA4tKc0FU+9mghULWh1P+nLQZTT2sQMzBopL6cMN9i2Q2a9aKmLeluTmvxBW3akRPfVwd2EMhxSrY8aj2Vf35UuzzxUZ+PihPf9Ew8OKO/YwW3WjQoX4wp3/aubQmTHAo578n/ZozTZuqZuFJi90eMnxDLjXNUImsFt/D1LKeEBic+5goB9XIwlYCyRSodHJA6zOFzAEOOWZFKfYK/T0toIpMXa8M6rTaYAKEuGGK+1jeGUnqTlT/KiFsSRqzvNY6TKllyrmTe04fJ93DZykONejkC0tooKiWh7oV0Y/Nk9+LTyCqXk78KX+t0AMqMoPEnE2Yb9dYQtgDhiOhQZx07dHuhiWlOiWyPG52Iv84nfzl30GZ1OG1/eFkMNv6S9PwL1dW6PdT9EkYblTMiYT7yPgWRzM6IHTplTpfQciqp+Z81ViazTQ6tQAZk4fWworxf9Wqp6tAmRvUBeVOm8WidVYmWq/jv15ZNqORF7M3B+IRDbxKWGeW1+m+bpvlS3gwykKfsjHx65X+NeSqV22rV22rVzElgeLHb82PZ7/88pp+Ks5p6ZhAaIpU0u64It2QVQH5r2bfSPaO1uggDVmqoKJJmja5nmk66R+4oPgaHBZ5QhkGgnksjV8DkwqLChKDBCDKdwCQETpvWAVT4HqslCZcRR2macu4/rCMINY9TGb3gagHJQNLkQDsrSn1OL2bkYUKXHLxHZMD4/GOtSe1m41oncEkNcQnIDDU1JGmHxrDAEvDbgJG7K2LmcVdKr0YOTs0BL+FuWl9uhQMxOUpwnLINlJna/ZS9ttC4Ws2Jg6umv4hLswnh596nDUGuFF9YfaEmSoo/qRRdc1/PlBwTgHfCFfuulHoxiRadmiU+OEIqdji0qyoWemaGIDzjDXFPWmIU1Pbk4ZKJ1KpSFKYDKbiwPy4J8hAMS33YWH3ByJS0/hqQTP4r0EpN+Dpc2en84WVad3AoSP1SB7VCr/NtWtgkBrbrX+lX5wuCp+BmbZomJ1XIrpl38VuG2p1q5Vg53Q90teiUztnz8r8yx2vvwG08ot2u9gqnFvdnG/Fns4y7uY6L14d0oWEZOe7O6vbqnRlAWvfa+hczc96Y2ZU4Z8xfi5MbPTVKKtHUqLpAfbndNYPWDRtlODUVisLmIcN1xNG7nfWGbXcXxekub+ZYeVyu4Gh+2BrbQSbbVs7WFuk+jA0hSmqw5eh0CBFUrFOHyCHLeM5dOrs4h6NV8lGbs+NMI7ybLsoDlXQ42VfUqVxra+sw66VQ/jhiVbWZaMsSXLoEb5dDE0/3QtPm551RaU8IOl6Fej5jGc/UGrauG85Ysym72yQQextxOzQUJzaHY93KB+OTigBfiJS+5YE+Y09b5Ixdd5LNRp3bursVAM/7LaadHG/H9qGGIae/VlHdNzbv39z9Z8KBXy4R4yMOJTMNr2oyXkHc+hQjjrw9xJdxy5joz2O6Hq8+EfCGardo2MZuvq/JpChcfBv1m8bkjBY/L3Mtze0Q6PF3snJLtgtK7KeVipom9+KzY1sbpcAezfR2ZW8E2wNft2pc8uE6B0hmfBFdLGn49Az8iB2Xex21Nqb7pb1EdP6TM2/TpaDvd9lVvFBy+XuoP/wzrNKdJ98g048JWgNFPdn5LL0dnFPS9Je1XvZZHImNYiP344mNdKbLnn0oe67LPg+zLxqvMFeFCBkNfrjerwnmYq3xr/wrlSq6gHn5vtvs28nOxstESx6GoXdfU1uYgU60OZRT0konatIDpST8QHIZueCcdY3W6aN1WMidHbekNHVDMtDOYatGdAVnpCk2dY8216r5tfc3nyhOFpfONE1PR62RBaToQ7V+DB74NjP9kg4XW+hVylVZyjJaJppixtzJOPNpDFN++gwaqgmRhJJh5Ou8AQn7EkOOvnILe9U/shce8LUT+9sfqsiBlgwLWdFDXpuzbS/ZzMDaKcLRN787emvP0AayfzjblvRyVHaMhhSDhPsGcK5l/xHKOmTrG5uD7SA6FBVZcx15RNWgT08G40aQ6srWqpTsBiA8PZ1o5SzYQZxtGUJD60mcpj2BeHs2wCBrRGm9KNyTs7qQd3NXDmmHKL6Nm9iRKWKTexNb/Qc7GZwyIg2jUdTNfK571OlSRxDw2jYYE2QB1yDJSA6+55ncaQtSdrN1H6vyPfbulY9j0V1iGAN8GnpOnuXp9qYRnzbiDmWaqbW/FKHIkciCQ0Z6VtGui56F5pocM0xWlwOY2i0MNeqwbWOLfABZ/WG5zB4O7dsugGkKW3jD31zzMRxY+93uDH3U3nOrUvUkt5AtCoC0kKjf1eHIvvjWtfqk/wgG21MIOhz422lyyY01lDaYCGGqvsM849r+Hv0B5HmSaOhWKdNFhbXnRHFgnsolpqjeZQ51ymy00N7fYBIGozWMiTGicaXq1qeAbzX1WR21mttqsxVukMbCWqhQHG1I4HeqjLqupslXTmGVkI57f1lemh9zi6McRHaVdedoeCVNDa024h4NmbZ9WeG/M3STsDqWXaTMIAZiNkemOyWdSXDGo3k+ModX090kiq5+rqkKuAtsBShZ1r3wcxNI1A5Vd8MhKdVQYKebDEDjI4n28QAsmNVXUMh6ESb6STb1moIshKl+bATRfdw3HzUD4Zg47HMFesZEPKLGV3m2aMut9E0q1HCzGCunrpsSy0c0zyXdLsH3aSuaR5/tiYpI/PKxD8NoUfgARoQqpqNOQ4Zcuf062KoXTiwXvX2BTjwXVl/OTGnCJ9Vu34E1tAc6ojr1XMdGSqrQy++SF5iQT5Y3Y+0fj2zYWz9a/KPde9A30kfhNf/QTGKe5sPk2zSNqfhJMPC2JxugmcXD29OPyaIoG3SdmgtQK97DsWHc563oUBOexgIr3s60RKy62NwX1orYvUdb/0L2YaoUHnvrapMWeWeJn3P26p9Vd1sUtSj7knaUlu6336qR1PNO6SsxWdFIPgxPUvjHeuRAqW7BTtMNPfXddKyN4DB0BoU/XSC4r4FejJAYwe1Oygt2ihM2r69v1fUbuzpfkXeQGl2hSZZieeDE066JeB3JAPvsDKK9bpIOIXZz955dGospXqlOpyhN3Xv3ORmd0z3mFe1xnva06gkLc/ITYMTzqHrUuF0+hGUPGtncHW9Xd0RLVI6en8r59OtpM0MffHklPixnyjZ3ZSnL6XMPwpKE18exE17VWb9BSoFz2P16ArMtiJoum4GbcbiRulc/0+9486vRqu3MM5Oc9a6xn1HhMzlBkZygE4YUeVIcnPr6bNXlw1CrGIrlSpR+ZCUIAJX7UhZMWptVIS3g2mAX7m5q/bbDQm0KV1i+OBm2X1bWy9UGHUn+TrKhYa6ydDMv1Ay5ccmpZA1LVQP7TcpAtCLIrt8Fb6qsWfP8YZVC+iYFaeonq0SVNPWmi4cMdi6SvftRRloWlCL0d0oUtmmBaEH5d7JB2qN3ebN9e5DF9TsnYA53RRR+bLaE9e4O0XuaTLycJaNyZlyOnbF7E1vJ2xmaN5u7dKuUnNrsCbCscF4Wcw0T2Od1KWVKuyhWFptQHZtYKPzG8av0nP9wEfbrB/r7an4rpLal2qnrV7waM1WzBDF3RLrbTO6DZgYqrPSyaPNthxe7NCgZ88FdZ9aDZjP+oGq3cMkqPG/6JA0SoRmdyARc5nVq9TI5S26VWdUmo1bnlExOP10LkLELS9H6VI3teiiXgpEniFRcxl7VKErcNhhqUkVEJmwmlXOrW9Dv2ZdRcZqB2IIT29d2AxU5NbsmZI4bfdMrxt0aW+q3Q9SvVzrHj60ntG3e5CqSKsLSLQynhPZLcnZ7AQ54ZmytquMLzk+vwk6hOJkM/Scvu1vhI5lOJ8yMEnZfjs4Pj2kEUhjp8760apmIcZqXvf6+9APvj24w6VJd98Is6IqS7mvBxcV0g/5ybyqUIeOyKQeS+BtAB7LD+zAOrdF1wqthzfotGLtS2ROAtekfHpEQUOm13UJYwMkMFWC5jUdBJqohMSmwlrkt+hb11Dng06FDbpZaiKG6Vpur5+4vLhfp97cchzoPP7hgtyHTTvRvPtyuG3yJ0ymM8tbuFRhqgedjicPaPzwvCwFu/WWG1re0nEu8Y/b2EdDcaStJ4qes515eot6M+Npf/+b5t0wdH8nq+nSYHhKdGpEpbZ6OstFkWNHjUyEpnoytI4ZLlDjqFyEZQ2DAUj882Sms9pUnKQTDuoZQA/Ou57dTan2UIq7asX7cjxqzeqxBfGnl5dvX7x8f/n2y3va5uNb7yn/jPPfBvBO1KE0jjRquPUh2w9zw9xpWe+H354/BSEtY6vt/VitiuHqbXfq/txUfZGg7k4Siy35SFl1qEGJigcpFLAC19D1phv9xLf48p2s7EI5fFnr3Wq1VvGnGrO/HUF9/0b9ZQnTQaVdKRqBjuVQj0krQ1CH+M5XUmwsyvHFSKAH6FJpnRgp2zVhSXdl/X4UIOp/8MbwZXtp6yUd6yCbBWYEfEql1PvhnVbZP9GrMWev7eN4IGVg3M3G9Fd6vrlhpBm25tAgNZ1KNksH6rxBn6ayBo10Ck4qb5uCkg9iGaMDXVkFboErD6F95BPTu/12R98VWJCUWC3EEdJjTxbTQsuNRX3kT1nwEWpyqfTpmfTGT2nz55OgTVGivO5W97NuU0PtMO/odlr1LQvj6wU3UM2N35QrqZF9sX5A1R1oY0X7LF1CYLMhS2Mv82Mh+Gxgc/BJG83G2wL8m0sVpFDy3jmJrzE8WKCKIgUd+5snI0lrUu79j7716/v/ZUFnW6Ft23TdrqY2wmnNkdp2bEpP1OPTW7k/VLCObFVfJ6lxM062WB53qDIdSBJzNJYn3cHHZ68h0/7jxevX/XLV42FLET6YMWRj/fXKSuwUPTT/P3XAVG4TTTUJrR+fLdTl9r25PluBGD/J1UrdQwwjeLX9wsz62rcaqhiIj/46ZaUTcpBRbZEaOyAsCfSbodjLxU4oEib0crrxOJK7ikCybosqbasKJHA361K4dTdaPHB35LBP7KU9mTU6ZgkdA8U/2MMzQIIQdlzeSCTscyd/1WWxKpvU69Rx1fINIKckhpR5eci7LbYt1ZodtYqY46Pcb8gZo65Zc/z1inXHAG3FvrrjDaPuCyVLsPcUduVdVVRiUa8rkvuLxe9H2NyLm93xSrVpDJD58VgVD6TFUHvl2F3l9d18s9XnQDZgOKgjNUBVXNnnTNV2/0Ls+DMx2+Nhdzzo7z+QS9n8hFjCuysnmKs8BvUVBtigBYqbTTV97pESJyixj84CN69UMILSwOmgAE+0WeP5ZJCAR7FM1Ut3OoAjILtK5nSUtqhuqkPTJ5VRp6oFLBKDdsZtOHXu9s3fXj1/9VTDcvnbb6+eaznLKm58PZu16RE1Q/AYYHqTHHRIRBvEdP0+MBwQb0LY9Wwh5SF0roMyQTWdoekyCZZ2QCy0wfvmVNTtFm6CFkRX1rYm85w2DaaTdz+kb3/63+/StxQUgin8p5O3eJG+e/V/XgykCqrlu6Mp1P2Yl8gcC/NxIlufofrTn7xzMu/Gp1jepoAAW6AbJfjw4440EK/7Duzzmc9sQxs4MaR+9YyHstRQQ3vjpoFNnzGiZ/blnhjiXpebcro7k3oiwgfA95eOLRC92rpb5ty22KFiAzftQEbZiOlwPJgytTE6uhK1xUASjOQZBfVP5OmwVwMz/UUk87WnHcYNo4PYk9jUDlza7BT015DPgBo1TiL433pfdel9yAVSsv3eEceG2cDpb8J3RU3S++gHVDr7SCfCt9btSXVlZPHuR3vS/r9wvsP606isfeT5jrkJajoeNx/rXmnhFiha/id0Lw/8yh3trz0w6sXZjzeZn5ZJ6Xau/vdljDsvzNWDyfhaHeSn1JyVkWzf3nkw11anaPIG2Yii7CirlvTxK1VtecEdvqePweF/VBs9kC2rdgNIenJcrLv3QfJRzaIxdtms3ddLzLuLFG/3H0tMT+9WQ8qoM7DVXn/k6/IT7XoU2xsrkyXFc6gX2tRRX2r7KOVOyWXazFIEe1xBTW4L9eGJAwxauf+eleiCT9jyB+n0nHdE87vqM1kSZBBxWAiKA1xfLxsEPurqg8bUo/38Mws2U/dT0FfHJmeqdKc5Ns3xIdSHlzg5zSzirdTeN+lozcb7HQgQRt1GyqLmReaPixlePs1tv+l5oqQCjB0Kc9DdYVEN4RueUiK7lzO5aeq9CzdQCg203Y/ds9Keprm69+DX3ExX6LH3bChzzJ7MfbFRXjf67YddVUbLTy/Tn357lv7y8uXrV29ekLh9/+vTN+9e/vLrzy9+fWeWo+bzp++fvnvxvis2IrPQAXr89pTgxFFUZ7ya/MfTN8+fpc9fvXv67PWL55yGMjlAOaiaeneTxW2vwLiS5ySLyny5pA/wvG+zECg0Rdcis0cIN6XLijns++fP1NYvx51TYrDGVN2DlFMlUlLai9ZfqLOOtUxLUR+Mz5l1jlIzfC/PbK72xGVxmkTV5ovp5MOWDnm7sUiNrLle1lPaG2Dgbw4SKk46G8+sIAusodjrkzSNDyOb/LIwk/566W+9eLUOebLhaaaInUyKSweTUTX5HJL4rJ6MhMq/PJi0PdaqP9NhAseHkz7MvIxB6/PpHB9Gd18xTXEACbajznWwUx3KNM9ejV9SMga4mRjy4Vx0dIQELQ0ySV7dx0K9oPyr/qaT+pKcUiFqBgbzqeOdS/Nrc3pGgz1e7sNILZu3m2ctZD+91KK1q316K47W/Po8Du/S8tn9/29s32U8s63MKSxXylRuMlxgJB0Om7RaN+lXZFNcTepiRzFbEhtGKvtV3y4mrUeGNMz49Ejf4dTgKEc61fPkHYwxuUNam2c3W6bspaRpm2tsHrHgCHFLTZoizAoNenUq9FV74knlmKpi/rSncerT6HLQoHszbMOKHsW9tv0sIONbdq3rvlnyIXB9NmB92MEh7o4QdJ2dmhvNlJpTKsOJcXZKA+dsvJqkL6Wm6/pG1+gxGI9iTKexZnqTnDUrsGL6NCidYKOx+aOwNE7dnucgCqD8swZDLCtVcZcOOZ0tVTr7kg/mNzeUaAI6aa7LH2zfZg0bUFz1e70ayYozUza56aVqYuRvnuZmEDVygFu2oPJTyjnEOlNXpZA1p+77KmuQoPahf2CU+KJuts16xtCgU/3h3On95Mjh3lNqHJwRo7wpc2b9A58KpDE9wV662bDZNOPbJhjpPaDV62t9O8OH3vT6w50Es3izvncISZ/XYGOeb7MzNfgCCLnkYdRuP4TVwN69Vpl5DfQmnGnbhOw/kmSDlqak4zq6YKQa+h7mfX7gSOl6ultujmu5amJ7hARFT11GaJPk0iKnudmD2o/jtjtZch96+2zzCIivBjCcmIttG33vx+D4R5NO2R5pao39ruEoAjoTXh0w4fuOpm2NJZfO+pX0OPrKEcrvN0pZSup7QPhMTteXXo8T3jCbzyhz/GvN1pGpWFVimi7kMs89OQl2qte0g0YDfN85f7xJwbdMaey+fOu5eJ1L3oer+RqrhqBLQyX9t74MGLme5GQ6V+PT+aYcaEt/t5Zv37YyURsrreMJ6hLDug99cycDzJIn47fomMtxijlem7+Mnly65muaRk9cN5d7gfTaI0qLQsLWI9uNPXLGZHvWqz2sJOiAyBr6jiucyArNGArwAXvUkyGJDRp3n/1sNll3W2DqC7ebTt49f/uUOYH1UVv5e2tHaV7NPn23ef1crg7ijTx8DwVgvXz99FJttC/0RvvjL2Q/oZ6hBuEjFc3GGYF5gheqnbYRIHNeSrvm3bd4u0DRfBgooivgDsrWUN+9boJEOur9CCjbgx8dbMcNzNlWtLPJxKK3OzXZl6AzazEqJpVMOmynKuzYcp0ZbTTdBeM76ZwiLctDEyh4i98cP2yrNJ2Y+Z5mx51kozxKZe6a57J67S/psDk/L3Tts6e0ejnVuq5xBrsqJgMfsi3/MHozQL+PcwdJhn2eqzfG3HpmDab5/j66wfZAwVIdxyPx1HQ66R/vPHMfaQ/u5soL476M8Vw6Mw2jl8ZnLodO35ud3jk6nkmHOkyaDU3oKyLMNKZme6EBe+Q4bYOn9OQY9eS++Y4uatuXefTmvl76N0+18zgBtt/v40j6oQOhvVyifhMGk4hDIUU5n1vFYI+7rOgU6MdcW9QgQE/TINJzdxi18oZkQSMsTgIGOrBAWtNE1tmAweNVQlWnPAhFMVRQwUg/7RSaHlQvbBdM6l/F+zjZ1Lvt29hbaWZpBKEaCOYqwNHbPpq1W1t7kTabdypZadptr53cD20NcovIpaHtsuFpjCYiRYqLskXFvm5udt5qbXX58/u3l63ryOfRFc1wjKVNLNru8VodO/06+Z3ydf5O8euP7a+79te2/UWZS+3Dcdf+pEuk1YMyqAuyDbjnSpWnv3+8mzBbqsd/mA+Z+cA5e3CAze4oLaqiexr0pQzNd2/S5pG+jb58jH6e9Fvoa4/pKhl2d1uk/Kc1VXN4uEvS4LNWRes0xeue3zzXbzrXgkrbJLfpyTVQ+hZN02VWOOjugl/u2zSHueXMrhcO5+q2kxkRHcZpFjWyJrvlZrN8zcQ0G7mZu021gwm2tdrtapMUrdfbX59aitrpehaoxMnsMVtpRjyL0dh6Y7xAfDkQxVWnp7eEXPPn6m/51HyqlnPywVCTlBDN2WZMOX9mIcXVyNWaLCcn14E1h6u5IaqsV7sln1RVFw3xjRftnUPTR3DCrJcxf398RIGG0ZyZ9d8tv6/tz0Go+J/TXc9DylRsXH7Wc4Tu672Wq/KBvqePkxuzU0+2oanetWZjeUM7yGmFm47OmrBIGxSDRN7kqyObpSpCwy5W46BQdsnf1dG8vpjXJngDzMX/A6lREfk=', 'tools/arc_next_report.py': 'eNq1GmuP27jx+/4KbT6UUlb2PpJuc1qoQNpLigDpXZDkiqKGIdASbbOr14nSvoL9750ZPvSwNsndoU6wlsjhzHBenBn62bNnn0TNG94KTwmei2xRdW3dtV4mbkRe1YUoW0+lVSPLncfLzFMFz3NP3MhMlKnwxF1dNe3y6OiTgSlhXeOle17uhPJUleNrI2A2U6HX8PIavtqGyxKhb4Xc7VvlVbCkKrdy1wErsiqXRz+X+b23Fzkx5H345W/v3/198fnj63c/vfvpH8BdUZWq1cDK2zZV4bV7AdRvvQI200ieezVPr/lOeLwRRzxNRd2KbOl93kvlwX9eeqLcyVII4luljRBl6JVV61XbrUwBw2ndbXKZLsQNzzsiZUWxPHr27NkRkU2Sbdd2jUgSTxYoDMAMSDRnR2Zoz9U+lxv7+l9VlXp1zVucsEs/wKsFUoDDPj/IeitzcXR0lImtl/KyKmXKcx8ZE0HUCOCgJLTLrCtqpSdCBWuTa3Gv4s8NvmpVV42KfRbCv4gFIaizuk1KXsZvea5EsAS9VpnwAyKl9tzfVNm9o2F2soSJiz9f6rnlXtxlEvTdmlWN4JmPWxuzllc8Uz7uUU8uES7Z3LdC+UGg94YWIrKkbsRW3vmw00xmoFAVPg8Lfpe01bUoVfxTVQp6V2BZZaYHgujIg09Xyl87EX95pLctmpYnS69HpcEIVIF9xOmK0UPC24RnhVQKVMfWDkpuvZ40mg7aCBIkh6ClK1aDNuvWALH1iRneiVKgd2Vu5q89rgh4b2XZiSkps6sDWsApMtgiPmDWgBmc5u0QKRgA7lFraLwvmCIKIB8ttkh/rWBiHX95/jwN2U0FMmPR2WMvth5mZabXJ/E5zRt9o+WJzNeQS7JG0HGIrOS82GTcSyN/kbrlITyCQeeJzPSL5dbaRSHaRqbK1/YR8lLdikYZlbdVfR7Dn4u4aniai/jM6V6Wmbgz4LhNUXYFacQfo8BPJhVILm3jVS8kZz+acA+Mnx3KVQdM2MOKaK1HECDjnZWwRR/ZhyWva1Fm/i5wa3AnJ/GmqnLfQpHm7cvqbB3HmvPRoouTuN+jA44uem60ZObAhnr7wpADFuHfEJ8v6PkiZDXwlGgkLNLfAAA6AtvIRenE+aj1hYFSJINTxNdySjLZhOCGPGmqqjXCh2D6EUKBZ3B4FQZ/voU47sHxIbf3GKEhXj6I0p4l+jAC1moB310tmhuJfrvEwEz7JXIxRZuedHCFtPVozwUtsEiqJqb4pRctIWQC96f2teSFOGGLIUWMbSzQSEDfPZ7lTrQ+o4OtKcBlDbsssI6NUTmCs1AJ71/oIm+apmp89vNtiZvSZycEllQo5RWdar2NcHsGnOAfgVFeKmTdDvk+ZWbQsPckd3uZZ4mBZcFxbB5n2PrklnpZJfQONmDzcPJKZdVD+DyHz5K1zBBN8Ck4pJEYk41Ud6e8SReluGsX8LCXrUjxQF2YNYsbsMdDbt7ccfSNwQJLBL5/7WSDgnZhANUWQrICkCA7WVrYFcNzFZx3CVgKCFHRMDySmeDSgPR+HONfTFbwWLSC1vOjo+w4tqRmGH9r7Vh1eQtxrdi4hMlyDG85+BREQK1StNRTNnCmRQ9h9Hu1a6quHsG3XF0v9PDUCJD970A63ZUTWg+Z6ESArXvBHW75x0EyKUtML9Fu0Kkzud2Cxw9YAwPpsQNNHNCbWA2ZhYMCtBaQMsZL/jRcYTNNA/41Jj+juCAm/reSwKZJ/VymOsyHCfmBlVEOCkFSLcEoEzTnxK02aRwmQG4wsYnq3HJIXHe5SFS3MfmIRbFrZBa63SZp1ZWt0ocg8K/CAfZSbiE8x/NErY3g+GKcTmvlhwMAWLiw+KamBMI+oLki20sgdLaypWSKdAcK0PocaCv6urXQpoyVaBmZOONE2xMxvkMSiaciGtJ0UYG4BHniN+WIvQMchANVdU0qYmJoZRaujYGNkyqcPI6/mKlIL1xFi/O1PjBZtPrCyA3cJMytzND6cf34DaE4Nq37aMm4akkjHRdJRjrmhB2GCVPdVM3pMBbA6dMNrGGkb3vUf48y33ZQLg6dB+rGqpAlEpwL1FYlJH/UiSE2G5/b+1r4NOGO1BySGgwKmJLomeNYW4BT2gyXr4nIKWpnxOA4PFke+xSKKETok/40KQP2aPY4Xj1hM1rr+hRh6/U3lG6IjjQ+8YXFVxRPFUdSY2yD6u/l2Q+X4avzHy7C88sXr16GLy7+cvkqfHH56vKlAZeFcNAvzsLLs/D84sxmG3hygYGbczyanuJDO4I41ogdBB08xkMjHaYZTSBJrCGbZM5yD6PuqM7HvLBHQgIl3eoMdGCI2tHspOoKHwEwLumiQpvaxOVtlRI4CrYRkFA2yyIqO5lmMum7ApNp0ExFko+oonbY0opYKiBiSfSpiP2Tt+ke8pFqoyCzggdXpHq6ABbqytOV5YmrJD1TiEJe6nGcvbvXbQtwJp57b9///EEtvc+gQNhfmncZpGoFVPQ5pcygUdEAcsytb6vmetnLkzcF5PJfHvvCGUZQRj5DC2e0tV3DC3hqRM1lwwbu2FYt7Da2BcSZrR7OJqXDmasatB2thqqCCIkCygVVzKhfk+7julFhpSUKulOaxKj8xpGDytuA4QYgExe1HrnFotMW0tHZco5MV16X1W2Z6C7BgCOmvQrcEEpu0aDwYAxdZzj0OAoe2qnCnG9AIyhcf+iaByiD0B/44hR7EIwLUq2EFSGH+h08x78Joj+ok0fi+waZ1VyMN/RbDlH8YOsnNrmzbxZDRcVOwNpO2PC4GYRStG9cuJQqwYx9itSAEQgl6TMp/lxJQz1Pm4vDEULZVrmbMKDziklb5YCkKfO/g+wvpW62kuwW6GWGg5R6h1TmYUjJJnxo04v79tqUGQ1g0i9spRzH5tGrGjcLBHEGvmZ4+2hkAccg5FX3KJECw9RUJ6Da2CLU3LP1lbG/p5zYdYlG9hqPezu294cUsClk23YQtm3bx52lY57QGrGjRSGLTN4Y/Njcg8iwiQ2sk5he6HnOlUyMATibUbihwG13FEQAVIOJnNcKy347MYt/PrgAEjInQqSnCGyCZLTjcQyciYDT+DcRxIAYJCcHAnHBC6kNY9ahJ958t0qHTV3C9i0d/x49T2PqeoWBUZNbm83ffGvDfRD+w/u1Tev/44ZHp8R37NfGfcoB1iv4WmvHUEfDaRPbXNEfDxshk47TpLs/aDJqbOYyQ98ggeHj17BTp8WM/bmDJt6Sb6g+wVuKwQGBsHhAAIhPnQFe3vs1jqj7Ipflta8Tv5oEidDhc1qje3wq+Gp/4KPYdTnHiyoT206p9ISUq+mwmQq7h5LhflrUAG9E6pShUeR432UFNHOWQVqGtUeuY7xed4vhN6t2Cy236QkJmudlV8d2qenr6VEWfnkc1SRmXANhAiiahOfyRiTUb03sOldQUQ5LF3TDldQCgTAiIfDp7JVvcovjGkYnCMY+M9N8cTeCxNKpbrFgr9OicW3UKy+raIR6fzfCQ+7djaQRjMh7uVNdvTAdp98ke9f8pZdvtH5tTTpWw29u/r4edjS1NObl4HaqxGx71PQ6331892nxb7JRd2tbcEKLt6Cgu9L2aY1A4ye3bhEsH2Tdm7dZthR3UHtjuvfEpkBLPEfZ3nsa1GmyuhHNbQO7nqqRUih7HUOuC5EAvdfcL2mPB5GTy4+bAzZddL4/JyUrEQOkkMhGZjDIgjlk2maokqIh1UGJeKcDs+mb6e8cH/JqxyaJKsoLohJaCSXUNnsEWDgETICysz6zbRjAlkJMwZyMI2o4QiCYMlMuyIeZWmDe0X4pXRe8b/z0qE91pYjb1LksNR1RzJP8jzRjL7B8t42Q0tK+nTOUz6HnhGxhrjdIVPA62zowbtZvj0qIJ6xUUzxUnrPO2QR+dEbgfcFhvfH0iVDp+5p+d72nTQ6DeemxwdJTdjIrTNfOfbrtYokujENSx4WZn1gkpheQ2a6EM62ZKV25uNL1VoLTmZ8gLP8j67coEEMkZHcsxCMRDifkP3Zw7z4kP755+/715zc/6h8ZwMrLl/RLBHAg5T1EozLSbRiNhRgYa2ojAErE2uta3vrB1T6e/BohuKKS1Vz/2g+xT+sqkLfPmg0jBuBcCB/0GBFntywEGqlIHsacQuYRzeZk6R6yd9MHbHx9sR0BWkp9/POzi5fP8U8Qbhib8U787JddjbmhT7jMBnSxYUaA+pKCoxkYoaEDdyiUqXX7Wm4wmygIE2H/WlCSWKrBkCyr4Dj2Calb4d7cAjeC8Pq3O8D0cTwm9T3R6P3w6LaXYF7W6Z8baQu78myeZQ98iHyQLRw4VX/7YCrvFap1DR5j0tVoP/ylSsgoN2URcd/3NB60tDFlZom70kgSE7L6H9/YKeOhh9bYO489IscGuB17wGZqSdt5O3ImswnGV/eGCouQd/MC2/z67i1r2nyc1keS/T1BZIyAZ7zG1DCvUt2VJh51fmbmMDNyOZsb6/N5THTopzCPR/8DLAj1Ug==', 'tools/arc_next_training.py': 'eNrVff2T2zaW4O/6K3iculvSltQfdhxHiabGsTsZ73TsrO3MVp1WxWJLUIvTEqmQlNsdT//v9z4AEABBSu1k7m6ndp0WCQIPD+8bDw9hGL4Sq3S/qUdZLsp6GFwV+3wplsE2zffpJvjxabAU2yKv6jKtsyIfFfnmLrgs3r0IFukmu+Kn48HgwzqrAvi/1HweXKWLG5Evh0Fe1PBqtd9sgvc/fDh5dxls0n2+WIsyKMogzYMX714G1aIoRbDYpNl2DB0KeLXM8rS8C7Ltrijrk2Vapye7Uqw22fW6PqlFVQe7tF7jwFW9hHEZvjRfBsUKWuViHFx82m2yRVYP5JREnl5txFbkdVDu8yp4++YiWKyLSuTBpgDgg22xFJsA0QCQBZXYpTAbEezKYiGqKrhN68V6WVyPB28KALbYL/HVx6yCCWf59TBYlGIJvWfQVbrAT4bBfrcp0iX8sSxuc/knzLsUNUxuUxS7KhCfsqoeD8IwHKzKYhskyWpf70uRJHL2MCvAIuG1GnAbxAegq6pEpRtVy2wB66hfDYNVJjbLgXy/Tqs14En9/EdV5OrvLWBS/w0kkckZw6TU46LigRHp0Isa9GfjyxKQX2z1L1EV+3Ih1O8qu87Tjf613teZ/lVnW8Hd13c7GFT1/iK/GwYv080G103OvC6KTTVOy0WSi091QkumcRC9LHKg10V9UZZFCQuS5rA0sLSAluwaiAZxshFJtU7Pv3o2DK5FnVQ7sRgOgsP/w9VLoMdlBigWQ/hdpkmdltgJgLHfCFzbq0qUH0Vy/TRZp+XyNi3FUZ2X4td9Bm2Dj8BE2H8CiyLKDEG/LTP4jQsWDwbALsm7i7+/fv/67ZtgGoSrJ+fPn6y+Pv/66fPn6fKb5yvx1ddfifRZ+lW6+Ob5mXianj1/Lp5+Ew5+fvf2h9eXF+/hq89htS1uxHk4Cc6HQbjLNkV99gx+nT3TP8+f4+/z5/eDweAvmqgiWITfRD79UO5FPKBHwbt9/v1+CWiY0Ey36aek2MGaZr+JMqlqsasmQQZMNw3OdYMs3+3rpAYgcv327NmT5091i2q/A0xmlVg6zZ6ffdP0UwHeRL4QTpunp988021ugYKg4aLIl9BgBetIoz0/PR2f6kYCEO+F6sn518+e283K4rYZ6SB6XjZiEchzlV1LNCHlJshPExBhJa5lSC+AgDOUJ0Bb1U3V2wIEW7YCsvY1KvY1TmaZlc4LkX/MyiJHOZiA3LvxfczA3Yg7/fwa2BvIcPR0fD56fmW28gDxz+BNkQv4DP8jB0UeXk5AzRQbePFDuqn4Dcga5Ek9kKRM46uECNLzKaxrcZuIT0AoGU4HlubXW5F7Wl4xfTakCm9JOkZL1oLJCqRGUd5NdYuYh9ihiAcaxMVfFDDLfbVUROSZZgV0D4y7TNYg/TZ3R3xSCUQLExOT40akJS0vap+GYIFaT0+Z8kn0wILcNKzTPE83u3XaUK8mXVjBZZIX5dbgAckARFBJur8mLJKimQQ16C4xg1UZBuPxeA7NozAj/VbfhUMQRQPAXZDsiiqrMxB5+X57JcoIGGQPUIPsjoPRn2kpmOBB5+3LHEW84EYxwBhEAOeQAYpJfaMyGmfVKkNyUw3xBf0Z/BmQxCNrScmLG7XWmMZHPPP4K9C7eboVNGrokVIhyD5XNqlnLWmkX9gCSD1uiRPrBQqQMJ4MHOkfEW4A8rSuSzmfIcEcx2jp4JIiJrwtEDNtVUMTfhyEoNfBZLoSaIsRZgO1btiruBZlGA9MUFrryoONXYEaw7zcHvG9ZDnV39Lp3uithdrgu2lgvDexiNO35mj2Y69ETy8AcpYD9BUYXmgN0lPZGNZF0pc2N6MFi+22JB8Gj0Bjg/RZ3Wl9bUsfokE0zJiRgCvmvOxg7r1fF7dBDdauMr1B1AAO4Y8CMMJ29G1WgyRBEPeLNZpGH4pysT55+curF2RH7vOcDKZ6HErRrTAMaxBZvFYxYMgGzErIBzy1sRTPYDGZv1nwHmW/OP+T3XQI6Di2+wz18Na4ROsdXWh6Fp9AcNPMRJpXksrQpoMVUOadmqXWatzKFSCy1ZUh/hUy5SupqxBz2ppCIK0vxx7REkyn+ouZ3dncwYUagya4RZcDVreS8wRrDU00WEKwzAI9yAgHCRYiA8fnOuwFfdqoWOzm4HqHZ89OYCwegd6rnqvGQxLS07LWj4wgBxj0BnFFxuaCGnB0rLYD0n8gAazvrspsGZTAHaIBqYHEGmCHbq4DCzGHHBeVsCViT1F8GC+D74LzR4+enJPoIMIB9xgYfguiD3T+gtT4IREq+7P0e2wSkPUGIQCdf+ZMXgsLJW3VRwF91CFvzclq68Ga8ZkxY90Cnz172pK6rd7I5ujvjprgQ6Rct8MuVFl2S2wgHwMRJ/gmQwHOrfcsM11NA6JBdeexcxj/VcP+3jbfTS1wPxuGUBCW6JyLb06bv8G3aH6cf30a3sOvfQ6aDt1UWDyzewkvWoqmXZitNHN2GZWIb+SoxsY5hvw6u0P8av7R5MWNGDz1qUtlBvwHYX7kkZamQRGcBE+eadfMwIL0IFrzREVnS3h0RLQq87hT7XeWD9Oh8OQ3jV9laEuPRxVb6OQAExkhJ9zDCbakeErlQ+ehVfQ6JkqYwEoYpNHiNr1cDYt6uzMnUO1XK/gDua24zUHlqC+ssB/1ukh3vhnxtIFEMGgUtbAZjzFktPkoojYScCbcckzRsipC5ObiNsi2232NhKG6h54EeXIahG+DXICNhhGp/RZDiwGAXVJQBazhXZmxHRX2DptVSSk2KS1EXUTmFBqaM6aA8EmIFhS+Q1slA4tzCeZeWolAmYtaWlXghVJoBgw0eBlOQNiVWfXpJC0XI4x1jQxEj7RxOvp4hqJGcgd85Vh0zsKHDG2GLUkRl4IIDBEGbsJfQvhXPufAJvRdiQ2gFGmDvl2AAkLwzlp9SxujAUI+gD6Y2eENhykdW6vVk5ccm3791Op20imDDCx1NWnjLf2UbffbxFgEaLyFBRaJ6kX3XbWM5vAq26BxBt9/FLy6KpzM5n8FngDwrIo1YzT7WwyPE72jdv/7T0hJwY0AO0x2FsBCZBsyD4kng6rGgHJWh63htaQrrv4BiwlkjCCA6fBRjEBEjJxY/x4ZXrllAc5zI4jBUe96e/8Hc11SreqkIUbygVATbloP3T6W4iM5x3tWigBdhb7rWmyWBA+K79E1mHs7e2MiQGcauAvYRpykK+Cqb4M3l5eEQdx+oA2GdLEA62BxF3auK3x+m5ZL22efeHSV5Y4+9jRouf6tMXOWIx9BJ1ztNylgbVeAlL3DSZfg24N7gR40N7OUBk22EWh7ECS4OCA9cJ3Ono3YlX3zd5h0e67Xu31SIs1sYXD0WjNrkfRmRpUU6LekZV0pGu9ctlzUt0V5k/CGh9Fbuq8LYIpskcg9EPXuXil213E25C8Kw1moZSQGn1qB8R6drx0954ml5uPWeIY5YAxMcf/Wq6jHvPgCh7llZvhNk3h2NjeDaQy5CsVdg0Pkxt82ID5m9A9Y5vP5xLKNYbWBj9J8IaMEQ2oeNwb8RuQqfgC/npwOMRAM5vdaoO7RTvjZePzkVGqy22xZrwPj09npPMbldgeD52o8AWQhg58KNBqe+6KRVTDAhBmYwYQYR4RHMfq4/KVr+OiYyEJsNh5/D57if7+hIAn9yki+xPQA/sDfjA/HK0OjI82vkZtJ4CCWKoreqPgaLCz4QRW4dONv2l4arSb1rBZT0ztKO6auiP5t1rax+z0LSm2HFHti9JAjRE/JeIJvt7v6zrfLq2gPjJSWJYeIwE7AhBjSH4gS6nUMBtUWLLO2he5ARZ/CiM2qyae0cs+95iqu3AKHCk/Pzp88/erZ18+/Sa8WgKmQF0uBgd2g7UXWKpjcz0dr8amZEUH8+lWvye2Aa+IQ/UR8RpB+ZoEBzl0L3NBAIcsL+AOdqJyku4VzVvaTADe1T1B5nIANuec3aSlwelfZEvzNw6BKERbOTbY4V4xstSCGPjv1hIzDy7ev3jYRlbTGCAPwOch5R/Pu0qysDKhwIfCZWgs92GTg2es0gcevXDzjM4lnUoGhNqnJneaAWBsg38pSiIGkI7aYyf7m3Q3kOHMVCybKIuMjqcAhqhXFAjOSbAUI5hQMtnZzOhQBWjor3KUGkM3tnXMnWKx7bkLGLyrcRw9u1wV4PETLBJSyfwB3DkOfmJEGDIqlOWr4VrTY4MKKkI4PFLXTwxa3Qrs/T4NzRxBa5LLPs1/3NqBqdViQIHmcAD5AFBcbX/SGpLWJMN40IqQNEc+xEt8WWuHn2ZBsykA/UsoKvz8dnsVqpDS/AXCmQUWhGT27IZjad9NNur1apkqyTIKI8wiiWajdMjJMiSrYF0MSGKov5rH+M+bxEExyoWDIbZYT5hkGWH6EGmzICP5ToldgvX1kTTGOLd2hhNFEzYO/munhJggKYqTVZKLbzON7SfCcc9AyQqLOzel/kls/5Hg9buV4GKTXLvJuI3OfyFeUTJIor1TtFpvZEP3dc1jPA5W9JdvfCSLqcBe2EOjbAfo+Q6MGrf1SpMu7oMI99uw3pB/CI44gKKOJlZnCvu2uUU7LeECdUjYW+EzguwXLAsT3m7cfdLdA67fBYg1di/xaaDVD39PWGn+SU4w/ZSWMEoUjc3INxsF7+i/uj1H4X/PWYl0UYMe1oCV9uwJVUY/VxHnnPyVDkWIpHrqyBYFDACiirMUH3UCmyxoAy/LcxBraYxoW7ocS1phSPTsUMOcInJoligqEZIy7qGNsXUZsCqq3UaiQSEGYxn2FX6jPE43tsLXpBZIYOjxpPlKwcUzQp/oJOYAyzAUak1MVEXy4WkkNwiiSQqHbeoxbYt/lWEv8t17adhA39jZ0ZtvYQUgqpLxhuQAMTMxTQcY26Wg2kBgAP8LmZFxJFeS0+FO9aGwPktGAvZYyd2FnDp7SphCndYB/4tmtbwFiRlxbwKiXGA0rgPJJKwIH1Vlj63lsFwV1S8Dbw7eEuzl8fG/zEnapjTNeUH6EPfAD2qmDdbXbxsH/Mp9S87hlsPu++6fnO0k6kpDcXogEnV4em0+bXpRVwljIqn8UKHzZQiRbiZS0RGXL9zocedVyV1GxjLpqoaRiqPgtfLUrQdqegNwZvfjx9ei8HYlxpBmtmvWk6dxIWIRmxi9i/XbolNZct/9MOVzSaCEUzeDJnKUY/IVCzDB8gFL8HWo6pbgxoBEBxP8mtPE5YaunNc1FsaNAo2LskV+DVeR3aqHJ1jdoAVgbQbFmI3DUZENyFN6OjwzlesadUZKOcM4hw2bwJaYL6X7Oo7ItgKFjEUiT4EEa8f9rTaVQYisrz4K1dZean2o3xtwMxZvE7kexZ0ue2P21WZe79nDvsT1JXo5d2+TQ98QkPPpxfHKgQy8PUbTLE8INEW5xiwkFbDb1x4PkqnZkGbSkE07Kiy1TqqHQ1kNQkgrmqqOhCtS4/FeZPpZklDFOKRSHTYKUFdm6b0/HlLatrCbSP6Yhb5vtOEvRikQpfe8ZpxHCw+DzvWMfK+1qhaakUXBv6vg+tW9N4L+vCaCyRMjIA7w32t8OuMoYqUK1Cr3CWHmFAifCkNDEjaEPA91gwnHM7ji7tC1mZXE7m8zNMDJ2zbF8BKUiA0RnlUyCUzOxZELRjCa3hNP2m/SSSfDkfqaBmutAbULxa2SjiAaJ3T0WBA2Bjtwg92/ZLnok90Umk9HZPO7ed7jaZxtDpeIWlYpUu1pPYbr95lG3hv2S7GRfKM1RuY0bTgFPYEgayBecHXIkn4M8HGobWYE1NbKOqx0rhTgo35eW9P8qH8mVQzNXBs3bmX3/N4XFkC1T04NqckLXKfyZfRQnyhloG/vKm5rNtec0m98Pmji27JaNJ0ucGmxkbIcYpjSDrbuYx3YAnOBmP7SaqUCh1UIGR5mfCQIPGaAz3ICJ5gNDydtpNjtMBu1Y1rZAqTNjRWgIPQqBo5dgCDrLYYhaAXnpYHr/17krMB/4muKGP1ovjehC0U7Q+qZBibxgeGEKEEyGBsJusqEeVOT7LeZTC9kLbUYG/2Oqh5p7e/11L8o76JM+mum2evfA+xGfB/N9pbYUvJ/9KXgP/tO/VWrPH+yN3QYT7diurszjiNUd2BLbYCuqKr0WY29/+0qUJBJf5yv4C6NitG+r11MGDulEHby0Zd44eMdiXofUZAaICMLe+KhKeaKxUoT639+j/VuW6V1QrPRWKKqIoc5tx/BPmvN50v/KD4zwyoJ0Au0xCUOd84skKcTjpVgUSwFO0OP+/v4r/w9aZ1rUVndEA01nsbcvVJkkUZiTQZDoXc7Q9Jv1373OZKhoJrEWBb5XLyi5wpDXk2ZZ+3uWJIOZGLPPoA0oVStEWkFWxs1J9rbxyX1/uD5scoIwS0YjjOm/wVh/J3K17JkmWQ6amqHMmJe9sqDNxvddyzODyd7i0qCdIE1+tHj87dnqaaQ3ppuJfNl84TOCKPEG/EY2f/iEoM/qGOqWJW3k+xDUdbCRdkIOHs0zgg5+APqNoTckgUZaAu1ARuzqEyMHbAvUDYaeSkOqy32+SCmZkviejUhtCWFi0K6mwx2fFpv9kvbaUOnOGjtVGp2ENY2Pm1ugJbaLFcqM9CJ50qBeZzkCY8eI5LlCgJtcQ9S1Cum4mJjnt07xHc8xIhLR3AGwpctlci1yeQon4b7oQAOswSOGzEx9VrgZGn/LpC4YnLo3GAYtjtxGcyP18Kj8cVAPWlTeTAGkmMHiaYVHndK8tvm8Acnldv/8Je7bCND+v4J9zNljKOEjYyFiPXH4BI/Z28mFevcJmgo656YgkKtphMVlr6gbTTwBy6DcMcbkxcSc1izdSG7imTSddfSip3NMHzqrYVnNJiimGgjjOVpnBsRqJ5030QOntf8kX6g5kvJFsTgCJ/uBbXMtTjiZp0HmtyrvuEpXArQ4Mm2g6YDjOpv0CkxFT1qJzpPKrCSpDDf8TxuRLLcdlLet8Rcoq7BB8Mqcr0++WfNVkkJJ3s9Kfk9saY4+jUgrThflQD+8GsHnYE1VIzXGSGYfH0jMC53cTwXxva0nkH+yfC/0Q0YjSrXR2enpPHjUWtDHiKqZ83TSmIIYX0JBR1IdZ4jWt21xyylLr0f+aQaEuuwG11owTHUcdrzfoZsa8dwBriltgPGcpvyfYWBiZqrw4kdn6zinbh+MXMQMtZRuWpqvO/QzSw4tCGX8bSorXIxlALIRRpKd43i8Fp+k+u/o3BDfPA5GNFiQT1uivaELpeQUwSJmLVuhrQXBbuD6AHxY+X1d7KJ3nJxL1StitSVQVU7TSwH/llphvxNUcgIj/TcqD0h8Eot9zSdDQRJQuglDAhKXV7w5vrdpwhYU/krwOG+SRJXYrIbto/NkhSzw9MpEF+eYoUKnpBy0sHAOY6DBokar0INo0g4JNvOeizfcPP8hb2N3FGAcq3PQ9IMgC9TZXwmp3b4ZHg8s4fuIrEnjudpVJle6eW73UzLm7aRxzCSyXrbPN7da8BFRmQGru08X9R7UjVoufK8XSX4r1wiEgxtRcxJEpRBuMBQhO7rY+PPURGjraJaTxpfCnEzqDbHtiHpXEjduDe5M2jOiWzvkwKD2IdgjBnYS+EmhWJJ/DirKBcqqDHIAImrLmfjH4KFFHhKmdskBH2DtMikHoGs+6AJRWQE+zHzXXi9Hkw+lJiajXmriQLUJWoP5iOLxNDg7htMeT32r19V1G8/TTkQ3jFbl6a5aFzVxWmcul5WIp4d060tMfLNF/cyM3jSXLK8+sAWBxzT0oqc1XN+ZFLOTNj4mh3Dp6U5sAHHYlmUH9IGJjafj0+EBKYQm5QYQUVfNcTGD5nSKIus4I0mR0BnJymPsXAd9jrhXs/WZiHZy7+/UgUdrw95vwRYBJzyvTTBaE8VenCTF3qxExuAoF3vA7UaXp8ByZcPg5c+/jHCHGR7QpsgGbB1stUpvhDItWunFdEZLneAgpjP2EczQtRMNcLdU7e3oJvhN/ePxOECFoLJympmkrNmQ0QSzN22oyDISpvTv0FiPqUGVPB8s86bUNRc8G7+j/0SULaajGkW5pNF4c02HrmhOse4LTNX9arURETWX+y4gpRP2ajC441dvOnJCZRl0dKxbi8a2GYBf0QYT/vE/ySxnEHAtTh0V4oeziXzipGb0Ztbqzwjy1+XdxNndxTVQUsUOyKHi2NWG4sJwskC72O7DxhbQckSNbI/tCho0JiDxC4WV6C86aYioVdKNErvgAUo9nM5jufnqdUDbvGmEgamp+jkfthxMj/LydeiTx12KyzxqK/GrNZjp+a0ayTHwyZOI/jUOb1NdO47FTFV1xzHix163zmPrRgcyEUTqhTimI9mdr4kiPfolL/JcXHMwSRYgksX3JFkj/WNVjO6DPhTiaI28KSqC6nBxKuPTGX8297iVoQsdtuwDqwURd5CochcVJ/NgNNJ7Vso/rbZ1YReVOuYDXIsz75itr9NluiXRE8r0DKJFx6V5fGxvMhcogV7Biy7Brc+rosSiWqccLlNGlCIF+UEgP5CKyYf1DshME1Ttfi1p92AhmWPIkmTK0kO3CYcm+FPjb2VxTFuMaZjDKKxVFEH3GT+AcZ2PtFVqSEqSbPpXr8iQsrCSwtAx9EI9WNKc/fcjdGpV43K0E5p0ZNNxqQHjHPjh7RR/2TdfW23g/JB9kkeo8IAQ2Q4FOCeqL5Qa37/98Fc84XCyw4oW8nC4PmYlwPCBKT7cysHsgbaNw7G79il3CZubICbLMQxxn05tppy6eynqiA/ttBonmTAyNaF/Gy1mGwec+s9j0BEv92y7vU3TVrDQB0H2ODgPHgUdXmTfgXlHxUtYWttwxmb3Ehm2YzQnTU4hDxRIs60tcQ3So8YyaWDWKqPXSHrdUb1dO4NM9ad8Em4tdwH/eFdkt04rLpX5u92Pltcx5Foo3iKo3qOMf7jvoRaJZolE/DlkhqMgNpZ2kJmMmtG0Te0jKdWqdQT8QaypCEGzKNKKQRUOdyIxDoNbOh8PYhF1vYqmW4oBHYg2vocBOcinh3dHdXm3xo1u4njfBR0RPAzOGMUZZaIUzE4KNntuHttEMvYXMLU8jmlgziy72IBiDKoYuy/Ic1ghGnY+ADCSstwy94ngMEMD//sQY9+JnLSrflIyxdLYQeLM4saEVhLDb0Y3BRwfaptyiYUHWKUWfWMBFZ+txDtdfYEz1dTiAPzCAkhJ6r4ebJejZwuw0+EaHLnR5/PDHuh2dQ5F853Ys6eaSioqJt+oB0YSiyktlPIznj3EKCSr1TJFpe1qcoRpwhInTOlf95yxh1lwHia0HuwlmiDyzQaa2/RxokTk4KDnq2RpuNhv97IiVxffudEjT4CVXIdykagKQckWjOJ92RTGuZdJ4RXmzibANWDcguAGpNKTdiZzjRVamyIdPkUn3Rck63urNrFx8kClkuIg6gCCvTFGcoSFw1LU6WIdxePFbo//4hb19b7YV55iapxyjEC6ZZax/CaXLsuxCA3JB+VAMciGLJBzmCHYc64pv07ptBfFvBguekSH8lCSkRMhK0SP6QnVyOM7GTQvjDFdc9ObOhbq823Oni93gAda5Az3sFDPY+xydweIqYurO1h3Zyv43vGUVOfq2BzPlDKOpd85UdO/13uzVJgXy2PJ8Omka0O1wpsV8Ag7WYWdtYXNJGO6eUGsanXPwWVRpqoC8QcQdx8AlXybAbbiOm3ub0nASI7ObijDo8Lyslv6QTik7UrZpHlLD7iB3R0V3cWhZAHefgg4X7CiMkut6xQiNYycULu6UwOluaNkfCTnMwZpkCyARYRVFl4KTyAiqqft+5IW/VOd8K8wdn0Tz1hG+55x7U9VbCeBpoubHaas45YGZ7lFHW85DWv6GVM4wZkXePeFcVwyPjjHJo+OhacUERRpqvrn2v1d5Gzu8a0qLjG4gDQkHbVy5RMUFVNF6OOXL355/+IyufwJfKqpW7F2aBTBn7bqzw4H7WRtRW1TSYuyhyUW497XUzLIr7K0mlLNpDB2poc5FFssBUziHKUhqEZDoFN4vkEEndBcJs1XnHqwG6uCO4TE+5bUdsdSYYaQgA1xEKr1bla5d77xWYHNrT5KJ2GdoivBmyy4xO5eLVavxPt1mtnyly0dtAFsyUOojAlupwHTgiJqsBPro3b2oChAMfXOUL8wvk8nmyB2yQd4pk0BlYzB6oIej18s0+1/RqTGHBSOScFUpCQ3DfGZZZsfWv2NDaFkKRbpHRMbemKASTfBkLEESMbbRmTCuUm/lHcuKTjU/NDUMGxYpEex6mVvKFQX/Kz222intDMf6/BQmYGio8ZhqtCDyOiTTbeU4+yRgNCecnJ7BrrCKptJxcnDZ2ZPaPGxBQlmsgpynvX1BQIUT9RAc+m90cY4izF12GbWbqWqO8774axMnDcGqkp4ViygQuAG1av9eQ+X3BuZDLhZ058v1JXa0NgBilM8Ip5LFkZdjDaG/y9IuOFZVbDTExSn8qYcHUjkMEQ3a3uFhrS52/m8yNJMX9HMcPIy9LPwDL74mC3EFDyJZTrBE3Jkk075OxBf13E7x9PTqUyjfWCPmkx0p4COKtlkN5RDHNvIH2OP4+ouX6zLApNsTTxj4IciwXbUL/LUXm6wdiDjs6FizBye6p8UdWaTxhVQ5MlNVcVk/HWU74ENva6H3ufnqIc1zhh5C71WY5LbjK4sw4i4pQqHfjklNQ1rX6RLlfQ39xaClr3jNS5yHFKbugY91zebBHiAKByGYzx2GMmms8nzeexPs/ZhhMFRODlG0ppos8Ay0IYl9DWt5fkYb2CrQE1nu6bEfnJQ6xlOgl2d/2F6j3bpk2yVaLgdSdBUgTCjWhTuiugyAHvSOoyl1wMbhZ3CiIThA3iMos+/Wy6pXckp6VIWdjPtVM7Zl54pB3SOh5vQqL9rHg26z1gyqVN7JBaC2OhcxxAaoeDeURJ8xvkwjlv4grnNdggc7unOjyHLthFLeSFOYgrtKbs38sjNXDTEeDOfzAPRqjbJjqaCwQFXA+IvP9nFfTxVWgqLBZu7o8SnXZJ+vA65Pj7+mVS/hl0Gj0GlNJ9tweGOuJ05KIOFvAIk5pRRHVsxQ1fOq+h/uwSPDvQa2QPSaDJMoWNssHbH+nOSHEzNCcoTPQV1a4cv8OaxtIykgUmAlV0wj75sUQz22JUQMFE85u+d2kpIdZhH8aHisWHTlJioaSl5Sjdsj4EiBLzc9CbBuh8LKi1PISdctUbIoOTcii0WTNftfBEvoz+9ydDfnWqG7KdNP73J8K80/3CQ6L+XDfZHmFaUBCmVqrRvHYEj7aIvNb18xuEhZevaaLY1dnDaX2SuWRtIjsn2B0o4k6w/YTQ0ScHFUudENIXjhZG6BFcXYeMlbOlHQWfkVKT7RB9jG6rLxVTEA3gLkCH4YrJlVt3AEyxFFJBTqtNBvCHb6mAolD5BSKQsk9FX+T2VLcMMm2O4kixnItvfZ69YlQ7F0rZMulxOnyCVR4Xtkk6cQVrU4+3NMisjUDaonuQpW7r3Iilu/FEQEjgVLB3Wt5LQRbi3gP2hlQpoTCqqBZb9xolYtnlpRr7HzcHPzi4NS1CqBraBE1myDRsFJ43ikBYyVqsKOz61i1l5um3XtuI1URXLsNadqoZqT6gJRujrXubjkotdhn8BkXjWNZ8ZByL4M7R48CoIBIaOzjdjd36uy6Bxe/7RCMumeJ9nwkOntzYhRi6eFSVorgljvKsHGSWKvbmFUd9SOR83jRsScdv5lLZKKzLg0uKFh2ukjJRi3tKfXTLDYNqhHZW1LrVAYfXHCYGmw27eH3Igi6SlYn35XXO6my81lLKVjBlL4dLOkRZ3mgUPrLqh93d7o0MS09PO7dvO2fJ3UyVKUTmlH42E0mW2WmEcnqR2lo/Y/NKSVzZ7+GLiYxMaRMQfuoiqQ3NqNNlmvUinOculJkZDmV4YlZBR9UcQI+ry9n1dfFBEbigtRfZTu8UYO+qQ5nR5GlFDleCGhVQRdbmvcO9pWwBe8ISvqyt2ZXFFVYH6ip2EstbNbHY6PJsPZ+fDJ/P5OLyfHyhEIX0ob6WE5vXBMhVaxWpE4ILS1UBRrFmtranMVt4k8N7SEYQZo5IDDDRon3byDnxMd+wYc2kx/eWJLmDQNgN0dWhZ/LRDmzEhqMZSLzkRAaovOVSFmNBfj8J3Fy9e/XQx3lKy1/cv3l8kP719dXGZvHzx7hU+BYCj8PL1y4s37y+wifrzoJ0Nn5nqQ1ftHaEUHFmaxXEKslUQ6VmfSLANtdKudFWtMWI3XhS7O2rS+nqorBBZhMendZUklWCStBjp6nVyEl7sG2imcvHADrsxbwBaZT+NwBAZd+jCo3knA616gg0LXGHBH4rb9pRgltJpJBtTAeaBt8iRccHdIZuoowdFX7pSrQ4AdPXY9UXHCPICMLG9EsulqpnY3b2/eUffSkeyutCQt0x4urebtw3pv/d95OJgXxGK/Onx8xQUxKKc9iOFeSg/8te07h0w7rpoRyyd7bBGtSlzJJFmSEJ2R+fGIWt5iTyrKaZ5su7UQacDXWnJl9BdHkldZrtD36wAzHXCm4F8FWDCw7YvbGt/vBS7TXG35ZBcrQpvkzEmi/vS1ZO7zb6iXYJvzbJyT0eXT4Fpwaihg9qsO7MKli7bbBrrVJ2+RXxyKV8j8S86IvdNk8gar1rEyFUJk7zJrjC9UvaJ2g2Ue3pNVic/evfL+xc/XiTvLy5/iMflHr+ED93zKH9c+O13ht4YSViTAwkFg/LyxrgEt+8PVGsw8nUKSqYUdQXyylAKRTXmhyBXgz/xtR88GF4FWdGdkOouSdqZ+pavxiAfO1jQZelgMVcLkS9T8LphSsV40Nw/R1DO1MEg9OX00YTo0SNvm7j9uadoJfVF1cmiw03NLuGzVo6cDYrMUy3w6ngKFnXfNTv0x/M8ko+ROqL7Nat1tlOi73O4Ix0DCwGz3+FC8BWKxLOE8ubtdbmL4vsvu+mV26sLBwA1e6TAcJVmeJvmQ+947E11bXWmypEnVFKcy8BXCUtBo9P+6z/vjTA05iLLCwoPliuhwKpeB/q2IuxvwP0udiKPwjSM8Vww9CLSrXswGJ+NaUUjoz4kJ7o21SGxAqRzU5j8dAVici0Jwzq7bG8G/fWH5K+/fJ+8/eGHy9dvyHr88O7Fm/c/vH3308W79+ZzaPnqxQewPj80jx07D+hFXiat82rDs3DgeR3+54s3r75PXr1+/+L7y4tXxFawEHvhb/3h7d8u3rz+3wjRzy/evbi8vLh8/f4n/mqFqxQOzIw7oyj2F92AeeAWS/MAeUW1ejoqSDdgDH1loKc9d7b3mTMoLUa6wLrkaPW790vrpnn1pXWncXyU3e2M/XngbNmxYfaZxFfC1fKT2GNu2w06/RXiexQtHH2oxru7sK+jMTIfxf8i36dxfG9ml7LDb4eg12m5vE0pl0fuzSfXTxP1NHLDE1xKAWsoKLJqbt9ptiwAbXim9FAzqZh124RujzzQHrV3zfpe6nMUtnY2ayWDrcQQIFSWvOdw1EW0FgdMu6+lPSKPwnvf/LTvMvojOqV7CsA83QFzExttkl9vRa667XjdS+zsXbq03u1YerowZtPZkdGmty9FfOp79TtuRafsYqXt0q7GPZutAMnQcyV0q0SSpoupSyhxZ2QIvRk8uycvf1TJUSRCuc4SmXhyN6674hJdDCXhoEu3+2dKtdf/lfN0DnvzXobvFLgJuH1//KCpo/1KnZrm67QWWIYE/J47uoSOsp0pcxpe8pYUHerFKBXfv96kkDT4AmUBtv1CZDujcjyCZCVMWHRBZ0XUuiX+D/T7nihTqBYnoVVXcQL1dGg0QOxYjcy17RkBm/FhOqq4XBq1kunwqTp6KimhJ4k1VIehnZXr7dpZ5L7u7QvPqKBU19UEDYP2T/zIzpgHeiZON2HJvBKnNrcMHqijbPJue9SvuGHe8LF9N4ysoNqHDU1eiLekdWhJU9f9QUtIEreSiybBG6m5nMwMHOCedIqc40G9Qhiz0FuSXB2GVdnttrDSScHe0/VDl4Qc4aDOMuqj5MfkKboloIb6KOXUKYzjw6n4KC8J0RjVs4jtPXoOe/ZXMhuaasmZGt1TaFg2D0nCPH6OsiK6vqSJfXj5w58sgEmvnuogHHUkBSXPfWMhkl1zAB01mBIjgeEGB7KHphaLr55PR9JI8J1HV1nH4tuxtN5j8nLLViad2Cflca1VtujvI1iueHDMkraXsl1aYNoQ4az/7Oz8mCEfyA80F5Md6EGbxEhpNQUI+GCp5NyJwUe6IMSk6e044g9zWHcwQkDGcXbfNsv3VWKNofL8es8wz4PRUQOaeD/Q430LH0xphAddMcCT9eSEzUMXsarKM8ePpg1nmqEmWBlf9Gjad8RG7aVMe7dZtKk9VX909afCvdW0J8DcaTBxlWW8UT6hW/WmunoHqqsTpcJMuYIm4bdBXtAtwfARcDeYJqCI1sWSqlMXBn+rqnTAnhf0J/bQLkynFw8fSxqW55+poAOXqBsnnGCTDPXNHHKnhN/PJuenp6cGTbTKWagYYCBHmrgjSycdJC5IszsPgE6Nzrkn0VEJ0T4Ol0Qm2VvdwDnQJ+eLXYIB1KWMh0cyTKoL4xxM1gvD8CXGrMEuKuVd8WJpB7jpAi+AnS5IWSANbDay1oE8K4k5ZXVTm3KXLflUKsXZZV8Yw6Xa6lSohtpx7+bxXQoGG0lf/hixiuTjOO7dwNTUt9nb+BlGbhi39maDKeCUCUdVQnAGlDtgP7Vi0/QeGjbHw/CkGr2LKjyQnuW1v4Y0tXEutHJLO8rw40222eyuox1ehVOZhqHBTD8zWJdFcbPfXbQrPOoi6ByCMYEEqh6/f/3jh4t3P+nboeUsEbfA5R8t5Kp3ICFB6KM8jFvv6NiOpKPpV3H3qH97fXn5oFERGUcOSH1QgSC32/YNsPgcU+yRv9QWEO/ZcS9siyCOThDkb8niIjuLI2Q6rJVtt2KZUT340KD8RAHT0H+bCFoE4Cz+qb30dseoXwYPpApNEXo7UW61UadSs9M2F+3DWFhK3F0a4lA5J6743q67G8oLwxGzSiNhZg+PhHOFjox5eb9Wu7JpgB/QhaBqB47OXZdiizcQNt0HV2UmVpu74HaNxm96m2ZUnu7te2ib7tAWV3uNoLYTQ7FFnYUlji8E/AbAvYIVQCOB6nuBjBUjLAzIx4yGTBOjq7sRQJDithZt2qSk77V4lsQ95sTtNxR2qRZrQaeWT1BbgPoDsVhgUT2cxzj4G6AH51kb+5qUkPbm7Qcm4DT48eXPwd9/+pb3Z4CcWSrTHYDNzQaa2rOSU4q5bjDd1leRpQCoFTlFzSoAaazmbu88yj/AMmjvQTJp77BcANfj25WwZGjWRaY/DASG/KqjtLRH5qke3mzxZTAfurGnueNEsqz8miwz1C16d562IOXoSNgSKI9/I62zLWdpGZt5aALK6yPoXHRqvr/XW6zde6uDB+d1eywKPQtlVKipaGwTaU1hvTd1Jn9iogjmxVFy3ycsIrtLbzFZV0oTeTvZ1NqWB/MAi2hE9ibOMFbVmHkf2btdrM5cNXu6EonIWO19QyXN6TPvIRNLEnoVRL8fa1X86NNJf+BG9C2mhy6L65EEsdNh9VSE77I1O7ugyfLkjW4e6twft1WufKQEI4ErztrpyLxRE/yI3s+CRDzFD5FVjX1rKd8r8jeq/QKXZ7XfWP7IYpNm29Aw+jdScHiN7c6EQWNxjaViO5XdhpG8flGuKy6qYgswk3F7nAUI0ZB69Ifj0pxoJTrmYJjCHVjw2samp+NObW5YV+qh95So24jKUdMddwwAF6tmFMd4LKbLt/afDU2v8Iwk0IlUcvLSLqICRSJf4nw6Ky/9IVgTujWxUsqPXc1h8Ddxd1WAzn6Nh2zL/Q5Mf7KDdMCiUGTjnYj2P7/Mzz0jP9fb8e+nK19OjxJZfr/VujvF70Cry4IEXXfY60Oavp9N39I85dz+DheZvePYTR2Wi9zZtUn6ygqe02VH9PfgAV19Gf7sKzL/D3KLwp4=', 'tools/arc_next_workflow.py': 'eNrFW3lv20iW/z+fojqDBSmHohXHnemWh411EqdjjHMgdvdi12sQtFiSOOY1PHx0Jt99f+9VFVmUKCczi8UG6LZEvnr16t1H6enTp6+KNo9lLLIob6NUHH9+PT3+9XR6IKJqsU4auWjaSooySYtGRHksiuUyTXIplkVbTc8ORZlGuf/kycU6qUVWxG0qRVzIWuQEv1jIuhbyNkrbqCmq2hO5bO6K6gafyqq4TWJZ1aKoxKKSscybJEpr/8lxmhIqmYokK4uqqUGLFGn0x8ORaNbAn9TRdQqasaJ6EGWR5I3A9ss2TR8EiKsa/8nTp0+fLKsiE2G4bOkMYajR4RggLmqSIq+fKJg4aqJFGtU1CDdAdZwsGq9/9US/WEf1Ok2uzde/1UVuPmdRszafK6lQl3gGcIP2kwXSJJl88uRJLJdiEeVFniyi1CVeycn8icC/SoLwnLfw4zYra/XWEzWWhzfyoQ4uKv4uy6hiDgeu4zmecObOxBNRmhZ3YR7lwVswVk58mS/AV3eid10mqQzrdeQSkXrPtQjMEX28OvjxJcDpxV3SrJl8Be0Xpcxdp7p2JuCVWKrV9G9J8ly3+Q1EIaBBlZtG2XUczcXSr2QUu89nB4diT9AfEHntOJO5WPttCVZLl1dO7OOv/bW8j5OVrJuO8rsKiENiDFPjCZtt9ASn6GntnvpgE5TGz27ipHLVF8NDeZ/UTVjc8Fe1JFmqVfyqdkFmFSW1FL/TZidVVVSuc5plbUPqCDWHOucL6CcoTXJWMIiAjvygsNeORYo6wvVDI2t3U/ziGdjy37lDx/33TgNdKNQfMtcE8iNxbFnp6yJfJivFA7afkDaai7qp1DPwt4KJhbD1ZAkiN94v1lAYmYPRGy+KtinbJgTPLGS8AXSQH4HdzqoC2kZOD/2D6U/XDkNFcVRi0x6f+If4UMB9BPyHYWR+m1RFnkEWYVosbizgQMCKGQ9rsrwvcQACxCH+fifzubguihRgrN8aG/uGkTdRldVz0bRlKi+B3BO+718BwgXlSUxGA5eEQ2T0sYJFJZWWVxPVN2GaZEkzF+RrAnHIz2tJG6knz2eznzSX70NIMgUTcXBsOlxDr1cSPgrCiMOmuJH5NuTs55cdMIjKymYH5IuDP7/st83lnQ1HVPTkHfT7L4q8kfcGpwH56fnPBx3IHdaGtQRkbG26TIuIsR3M/JmBTbI2G8BbcD/NDGBUksPHkRUBUL62jg3kiFLAhDJm0RqBJn34xhLlFWA8CTkROiAswVV/tFeAMTcPpdQPfa0nEwocFKtYXeC5bJgdWjdY0/u9bd9wcl+mySJpxCsAyiiHF5CLlv3CMo1WNTzc39sEoc/p/I3ZGboqfviGbj629dukqhsdthdQILi6moNnHWXkoeA9sK843n+1/5otQ5NAzhtGTa7bdXq9p223FNs83KXOjtcRaP45OzXaIBvR4W7zgdra5zeyXUnE9abS8vPoJJNOWqTlON0YjPhLIGbzAbXbLP1U1EmT3EpCJCGLVFAmA2WMxXUbA+0j8uw5KX6B0c6/ITFQzMCQUFMAnj2QWFVFW9ZbuHdwDRtZADtdziM69ElWU8bES/QhocMLuL1aIDkgDQIO/WZMg3a5Ei3SLddhC5VDIQx8TGBbktdxU8lZuBCRpxzFhGROjyk585N6mVCQMvB4p7b5HgV4y0uRb2o9IMK/LXoTkAb5UcegUbdI3NnpAG0WfTeb/m9ZNc6uj3dQOZG10Gr4TnyMetZpXaKCYtFWlIZRiJ+CHnXazv87A1H3Cr2lOWJfvHg5myGvNHzfxcDeMMZj0je0QGVZLel+1eaUw3fEdrZR8NEHZqGTWbWzjld5UWUIWX/IMEOdFCHxcs0HLWRUMR8R5doa6Hg9F08qF2fLXFdSCso1wdRGZqjGkE96EOwibeMkX4lzEIT6jJBdwP0XVbJCbpqKChEJNdW1TEl3swisp0QTK2uksUAMH9RtewRSRf1QYwdeqFS8RiKLkol9VNTtjqUIz1Dy6wScSBpfnFPIMWdVaTHZANFPkcc3Jx2E6Y4Rxn2nSKA7BdUvx/zoe/1u0yLLRKIaDS6vOi+V0HFw7g7bllchiI4AqgeJgFo2+gUC9BeH+EEGy9Epb5yvQqO+VK+uOktzFAdRoBFj8Qfcw6EiLJp0qYdaaZBddbsjY/2WZn7MYTgUIVlGqmqu25JqzY4LPSdI+2UeuzapKMLgoF2qP5w56g/83aTHVuZLffq5OZABm6N4wXL/byjNXbXd5OsVtF4VLe/e/qrCUaHPRMYQhuRuwtCtZbqkkhbMKXJPFxpJjFIclRT1HiwHSLB+B8rfenj+ahbBQz6CcogQWhla6wZATCo0W5O5Ya8Wc2zSfI6gMPSKuE6+CDYSGotxH/MD2zkUVyQan65dozg2ER676QRLv1PkYLO8pv6AU1Iyp4JSCCvPb+AldH+gO+CigGcb5mnfPC/iCMP4zJ0OjpoIpbycPr+aWKKmtKITNRkqXKidz+kGSYHqto+b1E5pUGTWMN6MGkca7LwpyhKneE0VNQpcb+vJGcysw2NJNoN6DrTE21aAjj4fztx9ah6HWQGzpKr96YamTXpLU+r+Rr9wN8naiOQjrElyKrzBUOyxKCpy7Xt7N3dRtaonIzFYSYJikt+RB2v+JdhWdHJxj8AZz3oBGJT/JiPk7JTzno6LlYwW64F/Ib8fiF4XNP8uHaMUcCOwCHdgIWyT4OFtspA9qhyISLGAZlORlBlAQXzVugypGu/3om/Y50jDLNo4sgFDnGELuMPL8VUtTPKlrKitE2bcOhtyXXVGuuMOD2NybreTYsCSRJpGvU4Yaoa0PlD7oIqtUSHcSDrriNUPs/yg5+nguXMFn1aEKPLK1DgHcjNEB1KWYOb/GQ+KMizx8ecft7eBI4cGQqCms6hVNlxonQ3GbMu97HR8cjVCfBnpugNMCNwdntEGIvX8LjAVHLkVgExG7loki7pbZNlnVxVBgFqUlzMvn1/1GkbhNNiFFnU0KUWHBvy6ScqwLuWC+mxaVn1D0TLSLw6hdub0f88ZuFpnnnvOZsHmzMkOuqd+3mYydSeTsRpbhzdnPgyIDiX09TqExdZFjiBNhuyMeYJfgs68pWmf2f9ch/qEzZoXb9EkgkDs1E8lJof0ypl81YlwiEw6tAcOXRWz5wmqsGUVkIy19e3o85j8Meeex6D46rmOfKtpwU3HTBGQtigOcW/Dmc9YGjya6B59tdqgAfeVNf6+M9rtrh492jN+C0e+RvQynWNV9AMJDl9UD5uZqwp8RZEiZ6sW4U20WlHfvr3OElZIEwe7/m3I8bseW5xTAwU+mCYmatUZNRoovhTpLTICeKSNZnDA8weSQu3aZ99oF0+4ua8b2trGFB3BJmFuv7RjW5Q/uCS4Svo0x0E5tVi7lXM5m/4cTZdXX376irSF2yiwYE7f9RdKrS18j/fjIqTwMlmtm+la3tOwS3VWEh4+NYlVNRxxl35KHlmcvql1QQO4vsDHjvZJftnq9YzOCygWdIWiNB1CpQJMDPNIVVxJSoKCpOuujqR92SDI95FNzDcSpKGoWbW7TIlWQqnzmM3HEwUQV5D66jBcR1WMvEI+noONLHAZovdv2lyHW6l2eRNRJRJsepujLpkINMizRwp9yxRH5jh6VtWPhxTkvqMx0jPHU8M94zd2ruG4Pb1Ocqqk9dIvjqX3akDmzLtB2g7T6J206qmyiJ35hgJ5CAl1Y17WbeYqc/G570JGBYiqbdbIzGXNdRbn7V8nehrB3iBQAjCk9IMgb/DkRj54gzlQ0PF8bEo0WoaMTm6CzvWOvBxFs6PNHjzehR9FpfLHwKF0bz6DoK35U4fPejbgnE7baFytZ57aLzwEBsI8sEZgyGgcpQdz59nG5LSf65mFk8lgmmknIYFVFrumVu2Cd1dd7FJWhpyafbS2dtvqsr1u06brgfxJfKY5uOQeblGRT7l+UD6oKQActwtJgZSaFOSHYO3ZtC33m7WketUsSSLdwvmTOLmVCF+EjnuiiCvUChNwqQCHplcrpFqLqDyijAM21bWeMqQkqqUWEXaNrllHOdVQDAtfyd6bcb09+/ipFtGiKlBfpVG+ahHTuJ1I85KuPK40ZdztyWN577k6anj0d0KxQ1LWwml6zY0SyyJ9an6Q3VlRhQ8dUDPKtWY1l4z9317Mr57ZT+f6sVVacOer4r4To9po6jA7qCI38p5uesvNnm635C/BbC6uEYRvBiAciepAhXm3G+IE1nBgMNUZLz0209HgO2YL45gGqa6NZnQs9K1CqBt3BI9PQ8bxDCdKA5YM3oyvtqNSQK0Ea/3Y0MPrRDUZStEwcLPxtFWgPxP/3DYbukXWH/R5HtuChxUBrWL/ERhStk880hPqoP2Rl55WPPVnGx/V3Ibj9PkZ28re89nMI1+/IH3qPKz5sHmiBWwo+OJoo0Y1pa3boRnTnI7lqGM7c/X36wDBths1/uGZM3WeYf0zR3nSiac2mxxpJ2rap/qxTnep7IYnp8Cu4SZBsBHm915oZ7yQSdmAempnRdQ0Taqkvt9H7jal3G1qF0RTDT69fW6NVftqxmztqDGJ+qJrLeRHdLPKIV5wVUaW+kga0qPv4Ukz1VHrrQV7L7xu/xFIwwgL72DkN99U86nOAj2rku3iYI/kf5eHgUAqt3LE2ZBba7HOpjwnLaiFYjgLsqg8BfeK5TLhup7hnTll3xbGMkri8DbD/m0TI+yRPXKnssNs7468CvGUrBR70GRGD/i549JfoeNvv77+hDCbppSFcm+TJ/m/v9eX9tIH39YJYgII/1L6eZTJnielKptKCj061Ok0epUW166zpzV98vXrrhxDK6FOLvS3wUhAPzMXzIq2CtND9qYVorfb555U2W+4X7r20s+9jvO+OoplmRYPlPwJg8pTNwxFw/qpE7/pkkopeCFUXb6ZKHWl/y4746uM6eHUoGYbczpSjfIR1T2X1YbGhA4982AZZUmKNc6H30/fnB6Ls0PgIs+s32dRdcN2srp25geHdmVQlEVarGgtUcQJC7kYLgEh+lROf/30m74ZWUliTaRHgZyz1dSGbUlJbG2IVqtKrqhZksmsqB7CpA5JpUu6kgLN5DafE9PsCRtf8yD4+Uszy+lvW8x3RUQHxchSwpjI9JtkCQYCkUrWyiTPIZzriDpz0VLqIQTiGBid0Kuz4vOxuSWGx1T69zPKLo2zD4T8DymtmiOyeeZ0I0Ms+foEJYcJrJTfHlHr+FAUNBaLlrSBvicp6WZqC1E/cMp4HEfiRla5pBxVLm5qezsmlBqghCZk2wr7XsD80ilKymX/IOyUTGNxt4Ynqg6rhkyLkvU3yus75KV4+iqNFjd3Egcm55nQhVZFBXVxhz6YGhLhirHTjvoKrlVhCaqwgFOxvYb6LOQ+a8p+x0S+1inrjVs5StfihK4sYunZoTYl8dtvp2+ITPB3qrWtT9N//3z8XkS0LTlXuunbUFquEoDNHYx4K8mXZeg8KLGopw70LHHS332yZGyew3pzqoHw8vnBlIb33V1hc2+AlR7eQV205ZbE1qYN8CyUQPmG8L5yYpw0UJni6GJ6uqQRet9NY+fGxpRFA0GsyjbUE/8QCKBmVswgEUpqIRW51ZkLI2wAdf5jLAZ04YR6yKEWMwmTos66acp6vr9/d3fnq4Yfcqxs39qmZgdWEvLpwezgJaUL02iVTA/26Xy3ibzbj2W9qOCOyVRMr3VHq9W6cECXF6a6c0qs50vcdVsSUpivfbtAOSOO1y0HFr6frTNYfe3gWJyf/vrX07MzlHJFra+h0Q2JWC83W4DtRDkI9MVpQ3eiOQIinAqKp7SOEVIoJB18f8QVZh5VVXEHG0dYLi2K76iFGBcrMySIaP82NpGU6tDh1YP/p7aybrQVtf2tTlbwgIMn7bU+2eDpw+Br2ybxd/Sq/egaZUDbSHdX25ruI1BLtvQRL+qHDBy7ca0EQqcE3p5ep3twk8cvJXLfGx57qjHqtqe2zJ0N8MEeY5e3R+5uL5YrbvoM1u6bjIcyI6T4ijPI9IlvPv3v0OUGTZf2b2ZCGu1oBxHB+9tbahn6ALY7uqtvXjRXzDPKjRWbXEI4CJioojbNN+/d2/Ddb6/Cj2/fnp1+OAkcJDYXn48/nL/9+Pn9yefzwQvAvjm+OD4/uRg+H1Zu/3H84c2r8M3p+fGrs5M3AflQ+N6Lj389+XD6X4Ty0/Hn47Ozk7PT8/eBsySf53if/vPi3ccP747P352f0KKZJnl3R5iexOCcupnAkZw8Q0nQ3SVhcmeUIIfsTVzNm+G9UMNv5FY0mrIGhp31GhD1lbxxP/6rHuawTP8GPqNcuR0yFCLKQn34touTz+/7wpRa+2UjPinQs6K4aUsW4ZyvSw1Rb1BwFyWN26iRe/DjFs7eBfhmLn9fkgYMm0jfRTQ55GE1/b2EWzn1KN3PZ9YPRoxmq1+L3Otfi+BpT7JGElin+8Tgl3BuvroyTS7Vq5tKDaBCzsPC0PZjE8+hu3GUKxKcttPJ1VB964aVCvsDCq64snc9v3jz8bcLj5WSe0imo658DMwL/00GAhxhCulPMMqYR4Ya/7KojZkQibuIGTGRze1eIUM/4Y/8o5VaSBb8sNdomSC//lf2W1Jin24wbtRMbRN9FO1jq9mLflZJm3UtdTPz4CqbLjzmLCOq2Y+Qc6i0gXKlW9ld4/ynujYmawJXVVXZn4c6Uubzho91WKwh5KqaIyRhz+lIC/lSBqWUu53bJsbv7riMXcqeP6K5mxvRWFBJKdTUdR3NUOZg2qKvOr+vWaLwwmS32zzau0w0c0YgjB/YRGcpM9+sMF8e42EHRpm9KpjVlc1++cQPQwr09GPDpdiBV7XlxnYYb+dZNhDMuGIlTEYx+MHGTv0uiMBUWYYFzbtUG/Crnu8naWy0ebS/091jtyApGyTecoqiHl06DBDqr86VfV9gsHb7fsBWi2lH0mSZkdWO7ZtPu3ltfID1erxdldDdOyW6IHDCkFroYejMx5Jtiksc1ZIctWfjzsZCE9I16rQjn9Sp6uXzq8ng7gChiarV7eSH4AVfLNbfAfhDYKKZOcA53x0+uUc8cU7pMLn+AQoX1pwI1npEttA31NVVayJUOy+eI29d5+i2PbjauMNxpFZcOvwDoauAfzvnDp7Zg8Vg+xrP9s8T3b099YMCfSu7omtN1i9cFSpv+AtXAP8P96BMjg=='}
HASHES={'research/arc-next-line/licenses/granite-4.2-8b-LICENSE.txt': 'c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4', 'research/arc-next-line/licenses/seed-coder-8b-reasoning-LICENSE.txt': '32e6feeb26ff1769664186fc3b9f34e1d36ffe38cac71d8a9fbc64dd506a7fa4', 'research/arc-next-line/model-source-lock.json': '8cf82737a0c6c92d0d26f3a7a268e59c55f4a4de213733777f71324930ece239', 'research/arc-next-line/training-source-inventory.json': '53fc35f54fa1b8a29403c5ede3e3880e8588c40c75a4a5783692a9a13a3e95cb', 'tools/__init__.py': '043476c65f87abc0ff3ca92bc13154d994e4aad5a3b5ae0121692e1cdc478c44', 'tools/arc_kaggle_submission.py': 'cdd570c99614d5eb326f9f07d86b36f80e4f5e53ec197370b5a0e03f262366bf', 'tools/arc_next_agent.py': '98c8970f4001cc5e9f57912f8caf74b3a892174cdee574dba2c54b229beb6159', 'tools/arc_next_environment.py': 'a1e33001b0fbc90360fa72307ef6d49ef8f5813adbf6d90d1c7435c698f832b6', 'tools/arc_next_materials.py': '582c82603f271757c56eda846cb4e724d3b90485906e7a48d9bb3ee34af4320c', 'tools/arc_next_models.py': '9b050596953a29ba6b3896b2c9f5b3aa867dec5eb1b3ff23923fe47b6ef0f058', 'tools/arc_next_report.py': 'e37f85557da05a70dc921ed0835a68f10e2f8b6219c69c3e8f234fd669f910b9', 'tools/arc_next_training.py': '25d21850a15ff693546fc8846bbb0496c49ae582b6fe5cf0fb8576bbaa7a7e2d', 'tools/arc_next_workflow.py': 'b8572d724eda88ecc5be2baf71fae5c559d621ff5badb99dd4c10703fd449e63'}
SOURCE_DIGEST='223c43d71f659d2d0da303835660ddd92e5636415d6a59bd6bf6fe2d84a64480'
import json, hashlib, sys, base64, zlib, re
for _flag in FLAGS:
    if type(globals()[_flag]) is not bool: raise ValueError('所有手動開關必須是 bool，不能是字串')
_phase_flags = [PREPARE_ENVIRONMENT, PREPARE_PUBLIC_MATERIALS, RUN_CALIBRATION, RUN_ARCHITECTURE_PILOT, SCORE_DEVELOPMENT]
if sum(_phase_flags) > 1: raise ValueError('每次只開啟一個準備／訓練／架構 phase')
if not re.fullmatch(r'[a-z0-9][a-z0-9-]{2,79}', RUN_TAG): raise ValueError('RUN_TAG 必須是安全且新的名稱')
ACTIVE = any(_phase_flags) or EXPORT_REPORT
if ACTIVE:
    CODE = BASE / ('code-' + SOURCE_DIGEST[:16])
    for name, encoded in PACKED.items():
        body = zlib.decompress(base64.b64decode(encoded, validate=True))
        if hashlib.sha256(body).hexdigest() != HASHES[name]: raise ValueError('內嵌來源 hash 不符')
        target = CODE / name
        if target.is_symlink() or any(p.is_symlink() for p in target.parents): raise ValueError('來源路徑不可是 symlink')
        if not target.resolve().is_relative_to(CODE.resolve()): raise ValueError('非法來源路徑')
        if target.exists():
            if target.read_bytes() != body: raise ValueError('已存在來源不同，請勿覆寫')
        else:
            target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(body)
    if 'tools' in sys.modules and any(Path(p).resolve() != (CODE / 'tools').resolve() for p in getattr(sys.modules['tools'], '__path__', [])):
        raise RuntimeError('已載入其他 tools package；請重啟 kernel，不可混用來源')
    sys.path.insert(0, str(CODE))
else:
    print('僅保存來源。所有操作關閉：沒有下載、安裝、模型載入、GPU 執行或輸出寫入。')


## 1. 環境（手動、需要網路，無 GPU 訓練）

In [ ]:
if PREPARE_ENVIRONMENT:
    from tools.arc_next_environment import prepare_environment
    print(prepare_environment(CODE / 'research/arc-next-line/model-source-lock.json', ENVIRONMENT, enabled=True, install=True))
    print('先重啟 kernel，再將 PREPARE_ENVIRONMENT 改回 False。既有不相容的可選套件會讓 pip check 明確失敗。')


## 2. 固定 public materials（與運算分離）

In [ ]:
if PREPARE_PUBLIC_MATERIALS:
    from tools.arc_next_materials import materialize_model, materialize_training
    inventory = json.loads((CODE / 'research/arc-next-line/training-source-inventory.json').read_bytes())
    print(materialize_training(inventory, DATA_PATH, enabled=True, training_count=32, development_count=4))
    print(materialize_model(MODEL_KEY, MODEL_PATH, enabled=True))


## 3. 真實 bounded LoRA 訓練／dev NLL／匯出重載

In [ ]:
if RUN_CALIBRATION:
    from tools.arc_next_training import CalibrationConfig, RunBudget, PROFILES, preflight, run_calibration
    if TRAINING_PROFILE not in PROFILES: raise ValueError('未知訓練 profile')
    steps = PROFILES[TRAINING_PROFILE]
    budget = RunBudget(max_optimizer_steps=steps, max_input_tokens=steps * 4096,
        max_supervised_tokens=steps * 4096, max_sequence_tokens=4096,
        max_eval_input_tokens=32768, max_eval_rows=4, max_wall_seconds=MAX_WALL_SECONDS)
    config = CalibrationConfig(model_path=str(MODEL_PATH), model_key=MODEL_KEY,
        training_tasks_path=str(DATA_PATH / 'train-demonstrations.json'),
        training_manifest_path=str(DATA_PATH / 'training-manifest.json'),
        environment_lock_path=str(ENVIRONMENT / 'environment-lock.json'),
        model_manifest_path=str(MODEL_PATH / 'irisx-model-material.json'),
        output_dir=str(OUTPUT_PATH), enabled=True, profile=TRAINING_PROFILE,
        enable_pilot=ENABLE_TRAINING_PILOT, allow_experimental_qwen=ALLOW_QWEN_HYBRID,
        budget=budget, approved_max_cost_usd=APPROVED_MAX_COST_USD,
        estimated_hourly_cost_usd=ESTIMATED_HOURLY_COST_USD, seed=1008)
    print(json.dumps(preflight(config), ensure_ascii=False, indent=2))
    print(json.dumps(run_calibration(config), ensure_ascii=False, indent=2))
    print('程序已返回不等於 VM 停止計費。請檢查 result/watchdog receipt，下載後停止 VM。')


## 4. 固定權重 A/B/C 架構比較

In [ ]:
if RUN_ARCHITECTURE_PILOT:
    from tools.arc_next_workflow import ArchitectureConfig, run_architecture
    config = ArchitectureConfig(model_path=str(MODEL_PATH), model_key=MODEL_KEY,
        material_manifest_path=str(MODEL_PATH / 'irisx-model-material.json'),
        environment_lock_path=str(ENVIRONMENT / 'environment-lock.json'),
        challenges_path=str(DATA_PATH / 'development-challenges.json'),
        output_dir=str(OUTPUT_PATH), adapter_path=ADAPTER_PATH, enabled=True,
        allow_experimental_qwen=ALLOW_QWEN_HYBRID, maximum_wall_seconds=MAX_WALL_SECONDS,
        approved_max_cost_usd=APPROVED_MAX_COST_USD,
        estimated_hourly_cost_usd=ESTIMATED_HOURLY_COST_USD)
    print(json.dumps(run_architecture(config), ensure_ascii=False, indent=2))
    print('A/B/C 只使用 challenge。尚未開啟 development answers，也未取得官方分數。')


## 5. 獨立 development 評分與 matched-cost prefixes

In [ ]:
if SCORE_DEVELOPMENT:
    from tools.arc_next_report import score_development
    target = OUTPUT_PATH.parent / (OUTPUT_PATH.name + '-development-aggregate.json')
    if target.exists(): raise ValueError('已存在評估彙總，禁止覆寫')
    aggregate = score_development(OUTPUT_PATH, DATA_PATH)
    target.write_text(json.dumps(aggregate, ensure_ascii=False, indent=2) + '\n')
    print(json.dumps(aggregate, ensure_ascii=False, indent=2))


## 6. 結果下載

In [ ]:
if EXPORT_REPORT:
    from tools.arc_next_report import export_report
    from IPython.display import display, FileLink
    exported = export_report(OUTPUT_PATH)
    display(FileLink(str(exported['archive'])))
    print(json.dumps(exported, ensure_ascii=False, indent=2))
    print('下載後請停止 VM／session。adapter 權重留在獨立資料夾，需與 manifest 一起保存，不會自動上傳。')


## 訓練 artifact → 離線 L4 ×4

保留 base 的官方 revision／shard hashes、完整 tokenizer/template、LoRA safetensors／adapter config、環境 wheel lock、程式與資料 split manifest。先在一張 L4 上實測最長 prompt 的 BF16 峰值與 adapter 重載一致性，再以四個獨立 replica 分派 task。4 ×24 GB 不會自動變成單一卡的 96 GB。

不要把 Blackwell 編譯快取／NVFP4-only kernel 帶到 Ada L4。首次資格確認不使用量化；之後若改 INT4，另外量測精度與 kernel 支援。正式 offline notebook 還要通過四個 distinct L4 UUID、完整 test denominator、12 小時內輸出保留、失敗復原與 submission.json schema。此處不會自動上傳模型、建立 Kaggle Dataset 或 Submit to Competition。

四個 held-out public-training task 僅是工程 screen，不是120個 public-evaluation benchmark，更不是官方隱藏分數。模型選擇須看實際驗證結果；第一順位 Granite 不是先驗勝出結論。完整 NeMo、RL、learned verifier 與 fresh-fold LODO 尚未執行。
